In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType

sale_path='development_042_silver_sandbox.demand_forecast.sales_silver_comp'
eilers_ts_path='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'
ai_atr_path='development_042_silver_sandbox.demand_forecast.game_attr_ai_bronz'
dest_ai_path='development_042_silver_sandbox.demand_forecast.'

In [0]:
df_sales = spark.table(sale_path)
df_eilers_ts = spark.table(eilers_ts_path)
df = spark.table(ai_atr_path)

print("=== Sales Silver ===")
df_sales.display()

print("=== Eilers Group Historical Time Series ===")
df_eilers_ts.display()

print("=== Game Attributes AI Bronze ===")
df.display()

In [0]:
# Filter sales for 'sales_normalized_title' and eilers for 'game_name' where supplier is IGT or Everi
df_sales_filtered = df_sales.select("sales_normalized_title").distinct()
df_eilers_filtered = (
    df_eilers_ts
    .filter(F.upper(F.col("supplier")).isin("IGT", "EVERI",'igt','everi'))
    .select("game_name")
    .distinct()
)

print("=== Sales: sales_normalized_title ===")
df_sales_filtered.display()

print("=== Eilers: game_name (supplier = IGT or Everi) ===")
df_eilers_filtered.display()

In [0]:

# 1. Clean the name.
name_source = (
    "game_name_key"
    if "game_name_key" in df.columns
    else "game_name"
)

df = df.withColumn(
    "game_name",
    F.trim(
        F.regexp_replace(
            F.col(name_source),
            r"(?i)(?:^|[\s_-]+)overview(?:[\s_-]*\d+)?\s*$",
            ""
        )
    )
)


# 2. Extract values from structs or JSON strings.
# Fall back to existing clean columns when rerunning this cell.
def extract_field(source, field="value", fallback=None):
    if source not in df.columns:
        if fallback and fallback in df.columns:
            return F.col(fallback)
        return F.lit(None)

    if isinstance(df.schema[source].dataType, StructType):
        return F.col(source).getField(field)

    return F.get_json_object(
        F.col(source).cast("string"),
        f"$.{field}"
    )


for source, field, output, dtype in [
    ("number_of_reels", "value", "reels", "int"),
    ("paylines_or_ways_count", "value", "lines_or_ways", "int"),
    ("persistence", "score", "persistence_score", "double"),
    (
        "perceived_persistence",
        "score",
        "perceived_persistence_score",
        "double"
    ),
]:
    df = df.withColumn(
        output,
        extract_field(source, field, fallback=output).cast("string")
    )

    df = df.withColumn(
        output,
        F.expr(f"try_cast(`{output}` AS {dtype})")
    )


# Persistence: 0 or 1. Perceived persistence: 0–5.
# Invalid persistence scores become 0; null also becomes 0.
df = (
    df
    .withColumn(
        "persistence_score",
        F.when(
            F.col("persistence_score").isin(0, 1),
            F.col("persistence_score").cast("int")
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "perceived_persistence_score",
        F.when(
            F.col("perceived_persistence_score").between(0, 5),
            F.col("perceived_persistence_score")
        )
    )
)


# 3. Identify individual theme columns.
theme_cols = sorted([
    c for c in df.columns
    if c.startswith("theme_")
    and c.endswith("_score")
    and c != "theme_presence_score"
])

# Keep one column per mechanic: prefer _present over _score.
mechanic_cols = sorted([
    c for c in df.columns
    if c.startswith("mechanic_")
    and (
        c.endswith("_present")
        or (
            c.endswith("_score")
            and c[:-6] + "_present" not in df.columns
        )
    )
])


# 4. Make theme/mechanic values numeric and replace nulls with 0.
feature_cols = theme_cols + mechanic_cols

for c in feature_cols:
    df = df.withColumn(
        c,
        F.expr(f"try_cast(`{c}` AS DOUBLE)")
    )

if feature_cols:
    df = df.fillna(0, subset=feature_cols)


# 5. If all themes are zero, assign Other = 5.
if theme_cols:
    if "theme_other_theme_score" not in theme_cols:
        df = df.withColumn("theme_other_theme_score", F.lit(0.0))
        theme_cols = sorted(theme_cols + ["theme_other_theme_score"])

    all_themes_zero = F.lit(True)

    for c in theme_cols:
        all_themes_zero = all_themes_zero & (F.col(c) == 0)

    df = df.withColumn(
        "theme_other_theme_score",
        F.when(all_themes_zero, F.lit(5.0))
         .otherwise(F.col("theme_other_theme_score"))
    )


# 6. Keep only the requested columns.
df = df.select(
    "game_name",
    "reels",
    "lines_or_ways",
    "persistence_score",
    "perceived_persistence_score",
    *theme_cols,
    *mechanic_cols
)

display(df)

In [0]:
"""Game-name mappings for the supplied feature, Eilers and sales exports.

Created 2026-10-19. Match basis: supplied names only; not game IDs or PDFs.

FEATURE_NAME_MAP: feature game_name -> canonical_game_name.
EILERS_NAME_MAP: Eilers game_name -> canonical_game_name, for relevant titles
and the candidate targets referenced by REVIEW_MATCHES.
SALES_NAME_MAP: sales_normalized_title -> canonical_game_name or None.
SALES_CROSSWALK: readable per-sales-title view with all corresponding names.
REVIEW_MATCHES: proposed Eilers matches; these are NOT applied.
FEATURE_MATCH_REVIEW: proposed feature matches; these are NOT applied.

Policy:
* Ignore case, spacing, separators, apostrophes and &/and spelling.
* Normalize Roman II/III to 2/3, WOF to Wheel of Fortune and Ca$h-style $ to s.
* Remove only terminal Ascent / ASC / Ascent CC / ASC CC cabinet labels.
  Do not remove a genuine title word, e.g. Ascent of the King or Cat Peak.
* Additional accepted aliases are listed in REVIEWED_ALIASES.
* Retain Grand, named variants, sequel numbers, multipliers, Extra, Boost,
  Evolution, Deluxe, Ultra, Multiplay, and other product qualifiers.
* Do not remove Link, Free Games, Multiway, Prog, Buy a Bonus, HHRAG or CTC
  globally. Suspected label-only differences are in the review dictionaries.
* All original source names are preserved as dictionary keys.
* Literal missing name 'null' is excluded, not made into a game identity.
* Unrelated Eilers titles outside the accepted matches and review targets
  are omitted. Including a candidate in this map does not approve its sales link.

Regal Riches is present in Eilers and sales, but absent from the feature file.
Mistress of Egypt Grand is not matched to Mistress of Egypt. Scarab Grand is
not matched to Scarab. Lobstermania 4 is not matched to Lobstermania 5.

The feature export has 802 rows and 744 distinct names (742 canonical games).
This mapping does not choose among duplicate feature rows or aggregate facts.
Resolve feature-row grain before joining attributes to monthly sales/performance.

Databricks usage after importing/pasting this file:
    df = add_canonical_name(df, "features", "game_name")
    eilers_df = add_canonical_name(eilers_df, "eilers", "game_name")
    sales_df = add_canonical_name(sales_df, "sales", "sales_normalized_title")

Join on canonical_game_name and the appropriate month/other grain keys.
The lookup itself is many-to-one and cannot multiply rows. This does not
prevent a later many-to-many join between monthly fact tables.

To accept a reviewed sales alias after checking it:
    SALES_NAME_MAP["sales spelling"] = EILERS_NAME_MAP["chosen Eilers spelling"]
All candidate Eilers names listed in REVIEW_MATCHES are in EILERS_NAME_MAP.
SALES_CROSSWALK and MATCH_SUMMARY are snapshots and do not refresh after edits.
"""

SOURCE_FILES = {'eilers': {'file_name': 'benchmark (1)(1).csv',
            'name_column': 'game_name',
            'row_count': 3826,
            'sha256': 'e8e257a46bf61896e886ba751012e5cb88600f3a5541b07ce6a3a6dd2b73e6ec',
            'valid_distinct_names': 3825},
 'features': {'file_name': 'benchmark (1).csv',
              'name_column': 'game_name',
              'row_count': 802,
              'sha256': 'fa0dc316e4d0ec375b7b0e2686eaa6ce1e59d424b2e9108b4b2434aed48106a9',
              'valid_distinct_names': 744},
 'sales': {'file_name': 'benchmark(1).csv',
           'name_column': 'sales_normalized_title',
           'row_count': 287,
           'sha256': '679072ae9235d2a77e84299ac5d12679b4c4207bcaf763cb67fb035f3663bb3c',
           'valid_distinct_names': 286}}

MATCH_SUMMARY = {'canonical_games_in_accepted_mapping': 854,
 'eilers_names_in_mapping': 900,
 'feature_canonical_games': 742,
 'feature_names_mapped_to_eilers': 744,
 'sales_names_matched_across_all_three': 138,
 'sales_names_matched_to_eilers': 250,
 'sales_names_requiring_eilers_review': 36,
 'sales_names_without_an_accepted_feature_match': 148,
 'valid_sales_names': 286}

REVIEWED_ALIASES = {'5x pay lucky 7': {'canonical_game_name': '5x pay lucky 7s',
                    'equivalent_title': '5x pay lucky 7s',
                    'reason': '7 / 7s spelling'},
 'aztec temple treasures': {'canonical_game_name': 'treasures aztec temple',
                            'equivalent_title': 'treasures - aztec temple',
                            'reason': 'Same complete title, reordered'},
 'chinese palace treasures': {'canonical_game_name': 'treasures chinese palace',
                              'equivalent_title': 'treasures - chinese palace',
                              'reason': 'Same complete title, reordered'},
 'classic hits money storm': {'canonical_game_name': 'money storm classic hits',
                              'equivalent_title': 'money storm classic hits',
                              'reason': 'Same complete title, reordered'},
 'connect collect fire stampede': {'canonical_game_name': 'connect and collect fire stampede',
                                   'equivalent_title': 'connect & collect - fire stampede',
                                   'reason': 'Missing and connector'},
 "egyptian link - nefturi's treasures": {'canonical_game_name': 'egyptian link nefturis treasure',
                                         'equivalent_title': 'egyptian link nefturis treasure',
                                         'reason': 'Treasure / treasures spelling'},
 "egyptian link - ramosis' treasures": {'canonical_game_name': 'egyptian link ramosis treasure',
                                        'equivalent_title': 'egyptian link ramosis treasure',
                                        'reason': 'Treasure / treasures spelling'},
 'fu gui ji li animal treasures': {'canonical_game_name': 'fu gui ji li animal treasure',
                                   'equivalent_title': 'fu gui ji li - animal treasure',
                                   'reason': 'Treasure / treasures spelling'},
 'legend of the 3x 2x peacock': {'canonical_game_name': 'legend of the 3x2x peacock',
                                 'equivalent_title': 'legend of the 3x2x peacock',
                                 'reason': 'Multiplier spacing; same multiplier sequence'},
 'lucky larrys lobstermania ultimate treasure': {'canonical_game_name': 'lucky larrys lobstermania ultimate '
                                                                        'treasures',
                                                 'equivalent_title': 'lucky larrys lobstermania - ultimate '
                                                                     'treasures',
                                                 'reason': 'Treasure / treasures spelling; Ultimate retained'},
 'lucky triple goldsmokeys den': {'canonical_game_name': 'lucky triple gold smokeys den',
                                  'equivalent_title': 'lucky triple gold - smokeys den',
                                  'reason': 'Missing space before Smokeys'},
 'magic treasures lock respin lion': {'canonical_game_name': 'magic treasures lock and respin lion',
                                      'equivalent_title': 'magic treasures lock & respin - lion',
                                      'reason': 'Missing and connector; Lion retained'},
 'midnight treasures': {'canonical_game_name': 'midnight treasure',
                        'equivalent_title': 'midnight treasure',
                        'reason': 'Treasure / treasures spelling'},
 'samurai 888 rise kimiko': {'canonical_game_name': 'samurai rise 888 kimiko',
                             'equivalent_title': 'samurai rise 888 - kimiko',
                             'reason': 'Same complete title, reordered; 888 and Kimiko retained'},
 'split n win linkfortune': {'canonical_game_name': 'split n win link fortunes',
                             'equivalent_title': 'split n win link fortunes',
                             'reason': 'Missing space and singular/plural ending'},
 'stinkin richskunks gone wild': {'canonical_game_name': 'stinkin rich skunks gone wild',
                                  'equivalent_title': 'stinkin rich skunks gone wild',
                                  'reason': 'Missing space before Skunks'},
 'surefire jackpots': {'canonical_game_name': 'sure fire jackpots',
                       'equivalent_title': 'sure fire jackpots',
                       'reason': 'Word spacing'},
 'temujin treasures': {'canonical_game_name': 'temujin treasure',
                       'equivalent_title': 'temujin treasure',
                       'reason': 'Treasure / treasures spelling'},
 'triple big wheel 6x 4x 3x 2x vip': {'canonical_game_name': 'triple big wheel 6x4x3x2x vip',
                                      'equivalent_title': 'triple big wheel - 6x4x3x2x vip',
                                      'reason': 'Multiplier spacing; same multiplier sequence'},
 'white hot 7': {'canonical_game_name': 'white hot 7s',
                 'equivalent_title': 'white hot 7s',
                 'reason': '7 / 7s spelling'},
 'wild hits clovers': {'canonical_game_name': 'wild hits clover',
                       'equivalent_title': 'wild hits clover',
                       'reason': 'Clover / clovers spelling; Boost remains separate'},
 'wild tiles gems': {'canonical_game_name': 'gems wild tiles',
                     'equivalent_title': 'gems wild tiles',
                     'reason': 'Same complete title, reordered'}}

FEATURE_NAME_MAP = {'100 ladies': '100 ladies',
 '100 play super times pay poker': '100 play super times pay poker',
 '10x pay strike': '10x pay strike',
 '25 50 100 play poker': '25 50 100 play poker',
 '27 ways to win dazzling dollar': '27 ways to win dazzling dollar',
 '27 ways to win double diamond': '27 ways to win double diamond',
 '3 5 10 mega hot poker': '3 5 10 mega hot poker',
 '3 5 10 play hot roll poker': '3 5 10 play hot roll poker',
 '3 sacred treasures': '3 sacred treasures',
 '3 way action poker': '3 way action poker',
 '3x wild diamonds': '3x wild diamonds',
 '3x2x wild diamonds': '3x2x wild diamonds',
 '4x3x2x jackpot': '4x3x2x jackpot',
 '4x3x2x jackpot plus': '4x3x2x jackpot plus',
 '50 play super times pay': '50 play super times pay',
 '50 play super times pay poker': '50 play super times pay poker',
 '5x3x2x gold bar 7s': '5x3x2x gold bar 7s',
 '5x3x2x red hot tamales': '5x3x2x red hot tamales',
 '5x3x2x sizzling 7s': '5x3x2x sizzling 7s',
 '5x3x2x strike': '5x3x2x strike',
 '5x3x2x strike deluxe': '5x3x2x strike deluxe',
 '5x5x5x jackpot gem': '5x5x5x jackpot gem',
 '7 leagues under the sea': '7 leagues under the sea',
 '888 empress': '888 empress',
 '888 panda': '888 panda',
 '99 riches': '99 riches',
 'a night of mystery': 'a night of mystery',
 'abracadabra': 'abracadabra',
 'aladdin prince of adventure': 'aladdin prince of adventure',
 'all in': 'all in',
 'all star poker 3': 'all star poker 3',
 'ancient arcadia': 'ancient arcadia',
 'anything but six': 'anything but six',
 'arctic fire': 'arctic fire',
 'as good as gold': 'as good as gold',
 'asian dreams - dragon': 'asian dreams dragon',
 'asian dreams - lion': 'asian dreams lion',
 'aztec adventures': 'aztec adventures',
 'babooshka': 'babooshka',
 'baccarat': 'baccarat',
 'bao zang bao zha': 'bao zang bao zha',
 'barnyard poker': 'barnyard poker',
 'bear mountain': 'bear mountain',
 'bears cottage': 'bears cottage',
 'beauty of egypt jackpots': 'beauty of egypt jackpots',
 'berry cherry': 'berry cherry',
 'big 5 cats': 'big 5 cats',
 'big 7s classic edition': 'big 7s classic edition',
 'big city 4s poker': 'big city 4s poker',
 "big city 5's jackpot": 'big city 5s jackpot',
 'big dragon lounge': 'big dragon lounge',
 'big horn bucks': 'big horn bucks',
 'big juan': 'big juan',
 'big ocean jackpots': 'big ocean jackpots',
 'big tiger': 'big tiger',
 'bighorn buck$': 'bighorn bucks',
 'birds of beauty': 'birds of beauty',
 'black cherry': 'black cherry',
 'black magic': 'black magic',
 'black stallion': 'black stallion',
 'black tie': 'black tie',
 'black widow - superstar': 'black widow superstar',
 'black widow grand': 'black widow grand',
 'boingy beans bean to be wild': 'boingy beans bean to be wild',
 'boingy beans cool beans': 'boingy beans cool beans',
 'bomb blitz - apollo': 'bomb blitz apollo',
 'bomb blitz - medusa': 'bomb blitz medusa',
 'bonsai blessings': 'bonsai blessings',
 'bonus bells': 'bonus bells',
 'bonus pick poker': 'bonus pick poker',
 'bonus sizzling 7s': 'bonus sizzling 7s',
 'bonus wheel triple diamond': 'bonus wheel triple diamond',
 'bonus wheel triple fire': 'bonus wheel triple fire',
 'book of wonder': 'book of wonder',
 'bora bora breeze': 'bora bora breeze',
 'bring the boom - egypt': 'bring the boom egypt',
 'bring the boom - pirate bay': 'bring the boom pirate bay',
 'bubble craze': 'bubble craze',
 'buffalo burst': 'buffalo burst',
 'buffalo storm': 'buffalo storm',
 'build a wheel poker': 'build a wheel poker',
 'bullseye bonanza': 'bullseye bonanza',
 'burning bars': 'burning bars',
 'cable car cash': 'cable car cash',
 'cai fu long': 'cai fu long',
 'cai qing grand': 'cai qing grand',
 'captain payback': 'captain payback',
 'cash on reels': 'cash on reels',
 'cash up jackpots': 'cash up jackpots',
 'cat paws': 'cat paws',
 'cat peak': 'cat peak',
 'cats n dogs': 'cats n dogs',
 'cats wild serengeti': 'cats wild serengeti',
 'cave king': 'cave king',
 'celtic magic': 'celtic magic',
 'chamber of treasures - meow de bao zang': 'chamber of treasures meow de bao zang',
 'chameleon 7s': 'chameleon 7s',
 'cheeky lil devil': 'cheeky lil devil',
 'cherries on top': 'cherries on top',
 'cherry cash deluxe': 'cherry cash deluxe',
 'chinese fortune': 'chinese fortune',
 'city of lights': 'city of lights',
 'cleopatra - triple fortune link': 'cleopatra triple fortune link',
 'cleopatra gold free games': 'cleopatra gold free games',
 'clovers & gold': 'clovers and gold',
 'clovers and gold': 'clovers and gold',
 'colossal titans': 'colossal titans',
 'connect & collect - fire stampede': 'connect and collect fire stampede',
 'connect & collect - firebird spirit': 'connect and collect firebird spirit',
 'cosmic cash': 'cosmic cash',
 'country cash': 'country cash',
 'country club cash': 'country club cash',
 'coyote moon - desert nights': 'coyote moon desert nights',
 'coyote moon classic hits': 'coyote moon classic hits',
 'crazy chameleon': 'crazy chameleon',
 'crown of egypt': 'crown of egypt',
 'crystal countdown': 'crystal countdown',
 'crystal kingdom': 'crystal kingdom',
 'da vinci diamonds dual play': 'da vinci diamonds dual play',
 'da vinci diamonds masterpiece': 'da vinci diamonds masterpiece',
 'dancing phoenix soaring dragon': 'dancing phoenix soaring dragon',
 'dancing reels salsa': 'dancing reels salsa',
 'dao zhang panda': 'dao zhang panda',
 'dark and stormy': 'dark and stormy',
 'deep pockets': 'deep pockets',
 'desert spirit': 'desert spirit',
 'diamond arrow': 'diamond arrow',
 'diamond crown king': 'diamond crown king',
 'diamond crown queen': 'diamond crown queen',
 'diamond galaxy': 'diamond galaxy',
 'diamond jackpots': 'diamond jackpots',
 'diamond mania': 'diamond mania',
 'diamond nights': 'diamond nights',
 'diamond skies': 'diamond skies',
 'diamond tide jungle': 'diamond tide jungle',
 'diamond unlimited': 'diamond unlimited',
 'dianmu surging skies': 'dianmu surging skies',
 'dirty sallys free game': 'dirty sallys free game',
 'disco nights': 'disco nights',
 'divers dream pearl coast': 'divers dream pearl coast',
 'dogs': 'dogs',
 'dolphin bay': 'dolphin bay',
 'double 4x wild cherry': 'double 4x wild cherry',
 'double big wheel poker': 'double big wheel poker',
 'double black pearl': 'double black pearl',
 'double bungah': 'double bungah',
 'double chili mania - extremo': 'double chili mania extremo',
 'double chili mania respin': 'double chili mania respin',
 'double da vinci diamonds': 'double da vinci diamonds',
 'double diamond 3x4x5x': 'double diamond 3x4x5x',
 'double diamond pinball': 'double diamond pinball',
 'double dinosaur': 'double dinosaur',
 'double gold - rising respins': 'double gold rising respins',
 'double gold pink diamond': 'double gold pink diamond',
 'double hot peppers - spicy jackpots': 'double hot peppers spicy jackpots',
 'double jackpot jewels': 'double jackpot jewels',
 'double jeweled 7s': 'double jeweled 7s',
 'double red white & blue': 'double red white and blue',
 'double rollin in the cash': 'double rollin in the cash',
 'double silver fire 777': 'double silver fire 777',
 'double strike - rising respins': 'double strike rising respins',
 'double wild cherry jackpot': 'double wild cherry jackpot',
 'dragon boat sunrise': 'dragon boat sunrise',
 'dragon boat sunset': 'dragon boat sunset',
 'dragon bowl': 'dragon bowl',
 'dragon dance': 'dragon dance',
 'dragon fairy': 'dragon fairy',
 'dragon lights gold - eternal peaks': 'dragon lights gold eternal peaks',
 'dragon lights gold - imperial tides': 'dragon lights gold imperial tides',
 'dragon of fortune': 'dragon of fortune',
 'dragon of fortune evolution': 'dragon of fortune evolution',
 'dragon spell': 'dragon spell',
 'dragon sphere': 'dragon sphere',
 'dragon storm': 'dragon storm',
 'dragonfly': 'dragonfly',
 'dragons vs pandas': 'dragons vs pandas',
 'dream card 3 5 10 play poker': 'dream card 3 5 10 play poker',
 'dream card poker': 'dream card poker',
 'dream pool': 'dream pool',
 'dreams of big money - fortune': 'dreams of big money fortune',
 'dreams of big money - wealth': 'dreams of big money wealth',
 'dreams of egypt': 'dreams of egypt',
 'dreamy genie': 'dreamy genie',
 'duck stamps': 'duck stamps',
 'ducks in a row': 'ducks in a row',
 'dynasty roulette': 'dynasty roulette',
 'egyptian dreams': 'egyptian dreams',
 'egyptian link nefturis treasure': 'egyptian link nefturis treasure',
 'egyptian link ramosis treasure': 'egyptian link ramosis treasure',
 'egyptian link wealthy wilds - amasis treasure': 'egyptian link wealthy wilds amasis treasure',
 'egyptian stars': 'egyptian stars',
 'el gran festival': 'el gran festival',
 'electric butterfly': 'electric butterfly',
 'elephant king': 'elephant king',
 'elephants vs tigers': 'elephants vs tigers',
 'emerald rose': 'emerald rose',
 'empress of egypt': 'empress of egypt',
 'empresses vs emperors': 'empresses vs emperors',
 'enchanted glass': 'enchanted glass',
 'enchanted unicorn returns': 'enchanted unicorn returns',
 'eskimo money': 'eskimo money',
 'eternal link - princess empire': 'eternal link princess empire',
 'eternal link - warriors empire': 'eternal link warriors empire',
 'eternal waters': 'eternal waters',
 'everlasting turtle': 'everlasting turtle',
 'exotic island': 'exotic island',
 'extra hot poker': 'extra hot poker',
 'face card frenzy poker': 'face card frenzy poker',
 'festival fantastico': 'festival fantastico',
 'festival link - joy and luck': 'festival link joy and luck',
 'fever spins - rodeo rails': 'fever spins rodeo rails',
 'fever spins - samurai steam': 'fever spins samurai steam',
 'fire chariot': 'fire chariot',
 'fire diamonds': 'fire diamonds',
 'fire it up free games': 'fire it up free games',
 'fire opals': 'fire opals',
 'fire pearl': 'fire pearl',
 'fire poker': 'fire poker',
 'fire rubies': 'fire rubies',
 'fish on ultimate catch': 'fish on ultimate catch',
 'fishing bob 2': 'fishing bob 2',
 'five card plus 2 poker': 'five card plus 2 poker',
 'flaming pearl': 'flaming pearl',
 'flip & pay poker': 'flip and pay poker',
 'fort knox heist gemstone dreams': 'fort knox heist gemstone dreams',
 'fortune charm': 'fortune charm',
 'fortune coin extra': 'fortune coin extra',
 'fortune cookie': 'fortune cookie',
 'fortune disc': 'fortune disc',
 'fortune fans': 'fortune fans',
 'fortune hunter': 'fortune hunter',
 'fortune ingot': 'fortune ingot',
 'fortune lanterns': 'fortune lanterns',
 'fortune link 4': 'fortune link 4',
 'fortune owl': 'fortune owl',
 'fortune rockets': 'fortune rockets',
 'fortune rooster': 'fortune rooster',
 'fortune storm': 'fortune storm',
 'fortune x poker': 'fortune x poker',
 'fountain of wishes': 'fountain of wishes',
 'four great chinese beauties': 'four great chinese beauties',
 "fox 'n hound": 'fox n hound',
 'freddy the lawn flamingo': 'freddy the lawn flamingo',
 'freedom bells': 'freedom bells',
 'frequent flyer': 'frequent flyer',
 'frog prince': 'frog prince',
 'frog princess': 'frog princess',
 'fruitastic': 'fruitastic',
 'fu gui hao men 2 - everlasting fortunes': 'fu gui hao men 2 everlasting fortunes',
 'fu gui hao men 3 - glorious tiger': 'fu gui hao men 3 glorious tiger',
 'fu gui hao men 3 - splendid dragon': 'fu gui hao men 3 splendid dragon',
 'fu gui hao men glorious fortunes': 'fu gui hao men glorious fortunes',
 'fu gui hao men splendid fortunes': 'fu gui hao men splendid fortunes',
 'fu ren wu': 'fu ren wu',
 'fu ru dong hai': 'fu ru dong hai',
 'fu tao peach of riches': 'fu tao peach of riches',
 'fuzzys fortune': 'fuzzys fortune',
 'garden party': 'garden party',
 'get eggcited': 'get eggcited',
 'ghostbusters 3d': 'ghostbusters 3d',
 'glacier hammer': 'glacier hammer',
 'goddess of desire': 'goddess of desire',
 'gold & dragon': 'gold and dragon',
 'gold & dragon 2': 'gold and dragon 2',
 'gold and dragon ii': 'gold and dragon 2',
 'gold bar 7s red hot tamales': 'gold bar 7s red hot tamales',
 'gold party': 'gold party',
 'golden 12': 'golden 12',
 'golden dragon': 'golden dragon',
 'golden egypt grand': 'golden egypt grand',
 'golden egypt jackpots': 'golden egypt jackpots',
 'golden empress': 'golden empress',
 'golden gate': 'golden gate',
 'golden gecko': 'golden gecko',
 'golden jungle grand': 'golden jungle grand',
 'golden knight': 'golden knight',
 'golden link golden eagle': 'golden link golden eagle',
 'golden link golden phoenix': 'golden link golden phoenix',
 'golden odyssey': 'golden odyssey',
 'golden pepper': 'golden pepper',
 'golden respins triple double - butterfly 7s': 'golden respins triple double butterfly 7s',
 'golden respins triple double - lucky 7s': 'golden respins triple double lucky 7s',
 'golden rooster': 'golden rooster',
 'golden stag win zone': 'golden stag win zone',
 'golden tiger eternal dragon': 'golden tiger eternal dragon',
 'golden treasures': 'golden treasures',
 'good times pay poker': 'good times pay poker',
 'grand buddha link': 'grand buddha link',
 'grand cat link': 'grand cat link',
 'great plains': 'great plains',
 'great white': 'great white',
 'greek lightning': 'greek lightning',
 'greenback attack wild peppers': 'greenback attack wild peppers',
 'griffins throne': 'griffins throne',
 'griffins throne grand': 'griffins throne grand',
 'guardian of the rose': 'guardian of the rose',
 'guardians of egypt': 'guardians of egypt',
 'guardians treasure': 'guardians treasure',
 'gypsy treasure': 'gypsy treasure',
 'hao yun fa': 'hao yun fa',
 'hao yun lun dragon': 'hao yun lun dragon',
 'hao yun lun lion': 'hao yun lun lion',
 'happy buddha': 'happy buddha',
 'happy camper': 'happy camper',
 'happy firecracker': 'happy firecracker',
 'happy lucky': 'happy lucky',
 'happy snake': 'happy snake',
 'harmony riches': 'harmony riches',
 'hawaiian diamonds': 'hawaiian diamonds',
 'haywire - supercharged': 'haywire supercharged',
 'haywire poker': 'haywire poker',
 'heart of the amazon': 'heart of the amazon',
 'hephaestus cash eruption': 'hephaestus cash eruption',
 'hexbreaker 3': 'hexbreaker 3',
 'high on a hog': 'high on a hog',
 'high on the hog': 'high on the hog',
 'highliner': 'highliner',
 'holiday in venice': 'holiday in venice',
 'holy guacamole': 'holy guacamole',
 'honey bucks': 'honey bucks',
 'hoodoo link': 'hoodoo link',
 'hot bar 7s': 'hot bar 7s',
 'hot bingo my super sweet tooth': 'hot bingo my super sweet tooth',
 'hot flashes': 'hot flashes',
 'hot hit double diamond': 'hot hit double diamond',
 'hot hit ignite': 'hot hit ignite',
 'hot hit sizzling 7s': 'hot hit sizzling 7s',
 'hot hit sizzling 7s free games': 'hot hit sizzling 7s free games',
 'hot poker dice stinkin rich': 'hot poker dice stinkin rich',
 'hot roll cash cash cash': 'hot roll cash cash cash',
 'hot roll fairy ring': 'hot roll fairy ring',
 'hot roll golden dice - pink diamond': 'hot roll golden dice pink diamond',
 'hot roll golden dice - triple red hot 7s': 'hot roll golden dice triple red hot 7s',
 'hot roll golden dice - wild red 7s': 'hot roll golden dice wild red 7s',
 'hot roll sizzling 7': 'hot roll sizzling 7',
 'hot roll triple double diamond': 'hot roll triple double diamond',
 'hot roulette enchanted unicorn': 'hot roulette enchanted unicorn',
 'hot roulette lobstermania': 'hot roulette lobstermania',
 'hot roulette lucky larry': 'hot roulette lucky larry',
 'hot roulette triple double diamonds': 'hot roulette triple double diamonds',
 'hot roulette triple red hot 7s': 'hot roulette triple red hot 7s',
 'house of jewels': 'house of jewels',
 'hyper bonus poker': 'hyper bonus poker',
 'hyper hits flower empress': 'hyper hits flower empress',
 'hyper hits longship legend': 'hyper hits longship legend',
 'hyper hits the golden bow': 'hyper hits the golden bow',
 'i dream of jeannie': 'i dream of jeannie',
 'i heart jackpots double shot': 'i heart jackpots double shot',
 'ice heist': 'ice heist',
 'ice hot': 'ice hot',
 'icy wilds deluxe': 'icy wilds deluxe',
 'in the chips red hot respin deluxe': 'in the chips red hot respin deluxe',
 'island dreams': 'island dreams',
 'its a blast': 'its a blast',
 'its magic amber': 'its magic amber',
 'its magic lilly': 'its magic lilly',
 'its magic ruby': 'its magic ruby',
 'jade fortune': 'jade fortune',
 'jade rising': 'jade rising',
 'jalapeno pays': 'jalapeno pays',
 'jin tu fa cai': 'jin tu fa cai',
 'jin zhu - dragon': 'jin zhu dragon',
 'jin zhu - rooster': 'jin zhu rooster',
 'jolly roger': 'jolly roger',
 'jumpin jungle': 'jumpin jungle',
 'jungle tower': 'jungle tower',
 'jungle vip': 'jungle vip',
 'kangaroo crossing': 'kangaroo crossing',
 'keno': 'keno',
 'king cheetah': 'king cheetah',
 'king khufu': 'king khufu',
 'king khufu - dawn': 'king khufu dawn',
 'king of huaguo': 'king of huaguo',
 'kingdom of gold': 'kingdom of gold',
 'kitty glitter grand': 'kitty glitter grand',
 'kodiak king': 'kodiak king',
 'kossack kash': 'kossack kash',
 'lady aces': 'lady aces',
 'lady and the tiger': 'lady and the tiger',
 'lady falconer deluxe': 'lady falconer deluxe',
 'lady lake': 'lady lake',
 'lady of the lake': 'lady of the lake',
 'lady swan': 'lady swan',
 'latino machino': 'latino machino',
 'legacy link - treasures of troy': 'legacy link treasures of troy',
 'legacy link - white orchid': 'legacy link white orchid',
 'legend of the 3x2x peacock': 'legend of the 3x2x peacock',
 'legend of the 3x2x phoenix': 'legend of the 3x2x phoenix',
 'legend of the 3x2x phoenix wheel': 'legend of the 3x2x phoenix wheel',
 'legends of asgard': 'legends of asgard',
 'legends of fire and water': 'legends of fire and water',
 'leopard lagoon': 'leopard lagoon',
 'liger loot': 'liger loot',
 'lil lady 2 wild wedding': 'lil lady 2 wild wedding',
 'lily of the valley': 'lily of the valley',
 'little green men': 'little green men',
 'little green men return': 'little green men return',
 'lock and key': 'lock and key',
 'locks of luck': 'locks of luck',
 'loot booster': 'loot booster',
 'lotus flower blossoms': 'lotus flower blossoms',
 'luck rising - shrine': 'luck rising shrine',
 'luck rising - temple': 'luck rising temple',
 'lucky bar free games': 'lucky bar free games',
 'lucky beans': 'lucky beans',
 'lucky buddha': 'lucky buddha',
 'lucky golden toad': 'lucky golden toad',
 'lucky hearts': 'lucky hearts',
 'lucky hot 7s free games deluxe': 'lucky hot 7s free games deluxe',
 'lucky koi': 'lucky koi',
 'lucky larry lobstermania classic hits': 'lucky larry lobstermania classic hits',
 'lucky larrys lobstermania 5': 'lucky larrys lobstermania 5',
 'lucky llamas': 'lucky llamas',
 'lucky ox': 'lucky ox',
 'lucky seven bars': 'lucky seven bars',
 'lucky suit poker': 'lucky suit poker',
 'lucky triple gold - shamrock sheamus': 'lucky triple gold shamrock sheamus',
 'lucky triple gold - smokeys den': 'lucky triple gold smokeys den',
 'lucky wealth cat': 'lucky wealth cat',
 'lulu and luigi': 'lulu and luigi',
 'lunar king': 'lunar king',
 'lunar queen': 'lunar queen',
 'maggie and the martians': 'maggie and the martians',
 'magic butterfly': 'magic butterfly',
 'magic diamond': 'magic diamond',
 'magic fortune': 'magic fortune',
 'magic of the nile': 'magic of the nile',
 'magic orchid': 'magic orchid',
 'magic rockets - dragon': 'magic rockets dragon',
 'magic rockets - tiger': 'magic rockets tiger',
 'magic treasures dragon': 'magic treasures dragon',
 'magic treasures lock & respin - lion': 'magic treasures lock and respin lion',
 'magic treasures lock & respin - panther': 'magic treasures lock and respin panther',
 'magic treasures tiger': 'magic treasures tiger',
 'magically wild': 'magically wild',
 'magnificent phoenix': 'magnificent phoenix',
 'maid of money': 'maid of money',
 'majestic beauty': 'majestic beauty',
 'majestic stallion': 'majestic stallion',
 'major multipliers poker': 'major multipliers poker',
 'malibu': 'malibu',
 'maltese fortune': 'maltese fortune',
 'master of olympus': 'master of olympus',
 'matterhorn': 'matterhorn',
 'max action poker': 'max action poker',
 'mayan riches ixchel': 'mayan riches ixchel',
 'mega wheel': 'mega wheel',
 'mighty mammoth': 'mighty mammoth',
 'mighty miner - diamond': 'mighty miner diamond',
 'mighty miner - gold': 'mighty miner gold',
 'mighty miner extreme': 'mighty miner extreme',
 'miners dream - blue mountain': 'miners dream blue mountain',
 'miners dream - red mountain': 'miners dream red mountain',
 'mistress of egypt': 'mistress of egypt',
 'mistress of olympus': 'mistress of olympus',
 'mistress of the west': 'mistress of the west',
 'money comb': 'money comb',
 'money island buccaneer ruby': 'money island buccaneer ruby',
 'money mover panther pride': 'money mover panther pride',
 'money mover tiger pride': 'money mover tiger pride',
 'money storm classic hits': 'money storm classic hits',
 'moon warriors': 'moon warriors',
 'moving multiplier poker': 'moving multiplier poker',
 'multi hand blackjack': 'multi hand blackjack',
 'multi strike platinum poker': 'multi strike platinum poker',
 'multi win poker': 'multi win poker',
 'multipliers rising poker': 'multipliers rising poker',
 'mummy link - dawn': 'mummy link dawn',
 'mummy link - night': 'mummy link night',
 'mystery of the clockwork': 'mystery of the clockwork',
 'mystery of the congo': 'mystery of the congo',
 'mystery of the talisman pays': 'mystery of the talisman pays',
 'mystical mermaid returns': 'mystical mermaid returns',
 'mystical princess': 'mystical princess',
 'mystical treasures lion': 'mystical treasures lion',
 'mystical treasures panther': 'mystical treasures panther',
 'nefertiti': 'nefertiti',
 'night sky': 'night sky',
 'nordic journey': 'nordic journey',
 'nuttin to it': 'nuttin to it',
 'ocean magic bubble boost': 'ocean magic bubble boost',
 'ocean magic grand': 'ocean magic grand',
 'ocean magic ultra': 'ocean magic ultra',
 'ocean spirit': 'ocean spirit',
 'old money': 'old money',
 'oodles of poodles': 'oodles of poodles',
 'otter king': 'otter king',
 'outback cashback': 'outback cashback',
 'outgrow - spooky moon': 'outgrow spooky moon',
 'outgrow - tropical moon': 'outgrow tropical moon',
 'panda park': 'panda park',
 'panda queen': 'panda queen',
 'pearl bay': 'pearl bay',
 'peggle': 'peggle',
 'pelican pays': 'pelican pays',
 'pets': 'pets',
 'phantom of the opera': 'phantom of the opera',
 "pharaoh's gold": 'pharaohs gold',
 'pharaohs of the nile': 'pharaohs of the nile',
 'phoenix bowl': 'phoenix bowl',
 'phoenix gold grand': 'phoenix gold grand',
 'pick a multiplier poker': 'pick a multiplier poker',
 'pick a pair poker': 'pick a pair poker',
 'pig and pug': 'pig and pug',
 'piggy blessings': 'piggy blessings',
 'pinball double gold': 'pinball double gold',
 'pinball gold': 'pinball gold',
 'pinball grand triple diamond': 'pinball grand triple diamond',
 'pink diamond - rising respins': 'pink diamond rising respins',
 'pirate link drakes treasure': 'pirate link drakes treasure',
 'pirate link marys treasure': 'pirate link marys treasure',
 'polar plunge': 'polar plunge',
 'poppin multipliers poker': 'poppin multipliers poker',
 'port royale': 'port royale',
 'poseidon': 'poseidon',
 'power charms': 'power charms',
 'power coins panda': 'power coins panda',
 'power coins pirate': 'power coins pirate',
 'power of cash': 'power of cash',
 'power quads': 'power quads',
 'power stacks poker': 'power stacks poker',
 'powerhouse': 'powerhouse',
 'powerhouse plus poker': 'powerhouse plus poker',
 'price check': 'price check',
 'prince of lightning': 'prince of lightning',
 'prism paradise': 'prism paradise',
 'pyramid progressives': 'pyramid progressives',
 'pyramid treasures': 'pyramid treasures',
 'queen isabella': 'queen isabella',
 'queen of olympus': 'queen of olympus',
 'quick quads poker': 'quick quads poker',
 'rainbow goddess': 'rainbow goddess',
 'raise the sails': 'raise the sails',
 'realm of beauty': 'realm of beauty',
 'red hot respin double diamond': 'red hot respin double diamond',
 'red hot tamales pinata bash': 'red hot tamales pinata bash',
 'red hot tamales respin': 'red hot tamales respin',
 'red rhino': 'red rhino',
 'red rooster': 'red rooster',
 'reel jazz': 'reel jazz',
 'regal link lion': 'regal link lion',
 'regal link raven': 'regal link raven',
 'rembrandt riches': 'rembrandt riches',
 'rembrandt riches masterpiece': 'rembrandt riches masterpiece',
 'rich girl': 'rich girl',
 'riches of angkor': 'riches of angkor',
 'rising phoenix': 'rising phoenix',
 'rising rockets emperor': 'rising rockets emperor',
 'rising rockets empress': 'rising rockets empress',
 'rocket man': 'rocket man',
 'romeo and juliet': 'romeo and juliet',
 'roulette': 'roulette',
 'royal lion': 'royal lion',
 'royals egypt': 'royals egypt',
 'ruby reels': 'ruby reels',
 'sakura tree': 'sakura tree',
 'salsa 7s red hot respin deluxe': 'salsa 7s red hot respin deluxe',
 'samurai 888 katsumi': 'samurai 888 katsumi',
 'samurai 888 spin - kenji': 'samurai 888 spin kenji',
 'samurai 888 spin - sakura': 'samurai 888 spin sakura',
 'samurai 888 takeo': 'samurai 888 takeo',
 'samurai dynasty': 'samurai dynasty',
 'samurai rise 888 - haruki': 'samurai rise 888 haruki',
 'samurai rise 888 - kimiko': 'samurai rise 888 kimiko',
 'sands of gold': 'sands of gold',
 'scarab': 'scarab',
 'scarab link - cleora': 'scarab link cleora',
 'scarab link - nefturi': 'scarab link nefturi',
 'sea cash': 'sea cash',
 'secret lagoon': 'secret lagoon',
 'secrets of babylon': 'secrets of babylon',
 'shadow masquerade': 'shadow masquerade',
 'shake your booty': 'shake your booty',
 'shambala': 'shambala',
 'shamrock stars': 'shamrock stars',
 'shanghai fantasy': 'shanghai fantasy',
 'shishenli pick a reel': 'shishenli pick a reel',
 'show dogs': 'show dogs',
 'siberian storm 2': 'siberian storm 2',
 'silver skies': 'silver skies',
 'silverhawk': 'silverhawk',
 'sizzling 7s piggy parade': 'sizzling 7s piggy parade',
 'sizzling rubies': 'sizzling rubies',
 'sizzling wilds - classic edition': 'sizzling wilds classic edition',
 'sky pilot': 'sky pilot',
 'slammin multipliers poker': 'slammin multipliers poker',
 'soaring wings': 'soaring wings',
 'solar disc': 'solar disc',
 'songbirds': 'songbirds',
 'southern belle': 'southern belle',
 'sphinx link cleora': 'sphinx link cleora',
 'sphinx link nefturi': 'sphinx link nefturi',
 'sphinx wild': 'sphinx wild',
 'spitfire triple red hot bells': 'spitfire triple red hot bells',
 'spitfire triple stars free game': 'spitfire triple stars free game',
 'split n win link cats': 'split n win link cats',
 'spooktacular': 'spooktacular',
 'stack em high poker': 'stack em high poker',
 'stack the deck poker': 'stack the deck poker',
 'star goddess': 'star goddess',
 'star rise': 'star rise',
 'star stax eastern lights': 'star stax eastern lights',
 'star stax vegas nights': 'star stax vegas nights',
 'state fair 2': 'state fair 2',
 'stinkin rich skunks gone wild': 'stinkin rich skunks gone wild',
 'stinkin rich stink link - hawaii': 'stinkin rich stink link hawaii',
 'stinkin rich stink link - wild west': 'stinkin rich stink link wild west',
 'sultan of mars': 'sultan of mars',
 'super fusion 7s': 'super fusion 7s',
 'super hand poker': 'super hand poker',
 'super happy fortune cat': 'super happy fortune cat',
 'super hot roll poker': 'super hot roll poker',
 'super money storm': 'super money storm',
 'super reel charms': 'super reel charms',
 'super star poker ii': 'super star poker 2',
 'super times pay 25 50 100 play': 'super times pay 25 50 100 play',
 'super times pay keno': 'super times pay keno',
 'super times pay poker': 'super times pay poker',
 'super times pay super stacks': 'super times pay super stacks',
 'super times pay wheel poker': 'super times pay wheel poker',
 'super wheelmania colossal cats': 'super wheelmania colossal cats',
 'super wheelmania legendary beauty': 'super wheelmania legendary beauty',
 'super wheelmania treasure fish': 'super wheelmania treasure fish',
 'super wheelmania vip riches': 'super wheelmania vip riches',
 'sure fire jackpots': 'sure fire jackpots',
 'swan lake': 'swan lake',
 'sweet 3': 'sweet 3',
 'sweet riches': 'sweet riches',
 'tailgate party': 'tailgate party',
 'tales of riches': 'tales of riches',
 'tap house poker': 'tap house poker',
 'temple of fire multiplay': 'temple of fire multiplay',
 'texas tea strike it rich': 'texas tea strike it rich',
 'texas tina': 'texas tina',
 'the amazing money machine sapphire': 'the amazing money machine sapphire',
 'the count of monte cristo': 'the count of monte cristo',
 'the fates': 'the fates',
 'the great detective': 'the great detective',
 'the legend of robin and marian': 'the legend of robin and marian',
 'the munsters': 'the munsters',
 'the shoe shop': 'the shoe shop',
 'the wild life': 'the wild life',
 'three kings deluxe': 'three kings deluxe',
 'throne of zinon': 'throne of zinon',
 'tiger empire': 'tiger empire',
 'top stop': 'top stop',
 'totally puzzled': 'totally puzzled',
 'toucan tropics': 'toucan tropics',
 'treasure blast - balloon': 'treasure blast balloon',
 'treasure blast - fleet': 'treasure blast fleet',
 'treasure box - dynasty': 'treasure box dynasty',
 'treasure box - kingdom': 'treasure box kingdom',
 'treasures - aztec temple': 'treasures aztec temple',
 'treasures - chinese palace': 'treasures chinese palace',
 'treasures of tao': 'treasures of tao',
 'treasures of the pyramid': 'treasures of the pyramid',
 'triple big wheel - 6x4x3x2x vip': 'triple big wheel 6x4x3x2x vip',
 'triple big wheel - triple double diamond': 'triple big wheel triple double diamond',
 'triple big wheel tales of the legends dairy': 'triple big wheel tales of the legends dairy',
 'triple black opals': 'triple black opals',
 'triple bucks': 'triple bucks',
 'triple diamond deluxe plus': 'triple diamond deluxe plus',
 'triple diamond strike': 'triple diamond strike',
 'triple double diamonds progressive free games': 'triple double diamonds progressive free games',
 'triple double dollars free games': 'triple double dollars free games',
 'triple double gold bars': 'triple double gold bars',
 'triple double gold doubloon': 'triple double gold doubloon',
 'triple double jackpot jewels': 'triple double jackpot jewels',
 'triple double red hot bells free games': 'triple double red hot bells free games',
 'triple double scatter pays': 'triple double scatter pays',
 'triple double sizzling 7s': 'triple double sizzling 7s',
 'triple egypt': 'triple egypt',
 'triple fortune dragon - twin': 'triple fortune dragon twin',
 'triple gold bars triple strike triple gold': 'triple gold bars triple strike triple gold',
 'triple golden panther': 'triple golden panther',
 'triple jackpot jewels': 'triple jackpot jewels',
 'triple red hot 7s free games': 'triple red hot 7s free games',
 'triple top dollar': 'triple top dollar',
 'turn of chance': 'turn of chance',
 'turtle coast': 'turtle coast',
 'twin strike': 'twin strike',
 'twin win diamond': 'twin win diamond',
 'twin win pearl': 'twin win pearl',
 'ultimate 4 of a kind bonus poker': 'ultimate 4 of a kind bonus poker',
 'ultimate x gold': 'ultimate x gold',
 'ultimate x keno': 'ultimate x keno',
 'uncle sam': 'uncle sam',
 'unicorn magic': 'unicorn magic',
 'valley of gold mayan': 'valley of gold mayan',
 'valley of the kings': 'valley of the kings',
 'vamps': 'vamps',
 'vanishing act': 'vanishing act',
 'vegas city of riches': 'vegas city of riches',
 'vesuvius': 'vesuvius',
 'vivaldis seasons': 'vivaldis seasons',
 'volatile': 'volatile',
 'water dragons': 'water dragons',
 'way too cute forest': 'way too cute forest',
 'wedding party': 'wedding party',
 'welcome home': 'welcome home',
 'wheel hottie': 'wheel hottie',
 'wheel poker deluxe': 'wheel poker deluxe',
 'white pearl': 'white pearl',
 'wild bear paws': 'wild bear paws',
 'wild fury free games': 'wild fury free games',
 'wild fury jackpot': 'wild fury jackpot',
 'wild garden': 'wild garden',
 'wild hits - clover boost': 'wild hits clover boost',
 'wild hits - pearl boost': 'wild hits pearl boost',
 'wild hits clover': 'wild hits clover',
 'wild hits pearls': 'wild hits pearls',
 'wild mermaid': 'wild mermaid',
 'wild pirates': 'wild pirates',
 'wild totem': 'wild totem',
 'wild wild wild poker': 'wild wild wild poker',
 'wild wind ups': 'wild wind ups',
 'wildwood': 'wildwood',
 'winners choice 2': 'winners choice 2',
 'winners only wild koi': 'winners only wild koi',
 'winning colors red white and blue': 'winning colors red white and blue',
 'witches riches': 'witches riches',
 'wizard riches': 'wizard riches',
 'wizards gold': 'wizards gold',
 'wolf peak': 'wolf peak',
 'wolf princess': 'wolf princess',
 'wolf ridge': 'wolf ridge',
 'wolf run - wild moon': 'wolf run wild moon',
 'wolf run eclipse': 'wolf run eclipse',
 'wolf run jackpots free games': 'wolf run jackpots free games',
 'wolf run respin': 'wolf run respin',
 'wonder coins - phoenix': 'wonder coins phoenix',
 'wonders of africa': 'wonders of africa',
 'woolly mammoth': 'woolly mammoth',
 'wrecking ball': 'wrecking ball',
 'wu da jiang': 'wu da jiang',
 'wu dragon': 'wu dragon',
 'wu wang zhe': 'wu wang zhe',
 'wu zhao link - cherished treasures': 'wu zhao link cherished treasures',
 'wu zhao link - zodiac kingdom': 'wu zhao link zodiac kingdom',
 'yin yang': 'yin yang',
 'ying cai shen fortunes fulfilled': 'ying cai shen fortunes fulfilled',
 'ying fu wa greeting wealth': 'ying fu wa greeting wealth',
 'ying fu wa incoming treasure': 'ying fu wa incoming treasure',
 'zhao cai zhu - gettin piggy with it': 'zhao cai zhu gettin piggy with it'}

EILERS_NAME_MAP = {'100 ladies': '100 ladies',
 '100 play super times pay poker': '100 play super times pay poker',
 '10x pay strike': '10x pay strike',
 '1421 voyages of zheng he': '1421 voyages of zheng he',
 '25/50/100 play poker': '25 50 100 play poker',
 '27 ways to win dazzling dollar': '27 ways to win dazzling dollar',
 '27 ways to win double diamond': '27 ways to win double diamond',
 '2x3x4x5x dragon': '2x3x4x5x dragon',
 '2x3x4x5x gold': '2x3x4x5x gold',
 '2x3x4x5x red hot 7s': '2x3x4x5x red hot 7s',
 '2x3x4x5x super lucky pay': '2x3x4x5x super lucky pay',
 '3 sacred treasures': '3 sacred treasures',
 '3 way action poker': '3 way action poker',
 '3/5/10 mega hot poker': '3 5 10 mega hot poker',
 '3/5/10 play hot roll poker': '3 5 10 play hot roll poker',
 '3x wild diamonds': '3x wild diamonds',
 '3x2x wild diamonds': '3x2x wild diamonds',
 '4x3x2x jackpot': '4x3x2x jackpot',
 '4x3x2x jackpot plus': '4x3x2x jackpot plus',
 '5 star': '5 star',
 '50 play super times pay': '50 play super times pay',
 '50 play super times pay poker': '50 play super times pay poker',
 '5x gold': '5x gold',
 '5x pay lucky 7s': '5x pay lucky 7s',
 '5x3x2x gold bar 7s': '5x3x2x gold bar 7s',
 '5x3x2x red hot tamales': '5x3x2x red hot tamales',
 '5x3x2x sizzling 7s': '5x3x2x sizzling 7s',
 '5x3x2x strike': '5x3x2x strike',
 '5x3x2x strike deluxe': '5x3x2x strike deluxe',
 '5x5x5x jackpot gem': '5x5x5x jackpot gem',
 '7 leagues under the sea': '7 leagues under the sea',
 '888 empress': '888 empress',
 '888 panda': '888 panda',
 '99 riches': '99 riches',
 'a night of mystery': 'a night of mystery',
 'abracadabra': 'abracadabra',
 'aladdin prince of adventure': 'aladdin prince of adventure',
 'all in': 'all in',
 'all star poker 3': 'all star poker 3',
 'ancient arcadia': 'ancient arcadia',
 'anything but six': 'anything but six',
 'arctic fire': 'arctic fire',
 'arctic fox': 'arctic fox',
 'as good as gold': 'as good as gold',
 'asian dreams - dragon': 'asian dreams dragon',
 'asian dreams - lion': 'asian dreams lion',
 'aztec adventures': 'aztec adventures',
 'babooshka': 'babooshka',
 'baccarat': 'baccarat',
 'bao zang bao zha': 'bao zang bao zha',
 'barnyard poker': 'barnyard poker',
 'bear mountain': 'bear mountain',
 'bears cottage': 'bears cottage',
 'beauty of egypt jackpots': 'beauty of egypt jackpots',
 'benny big game': 'benny big game',
 'berry cherry': 'berry cherry',
 'big 5 cats': 'big 5 cats',
 'big 7s classic edition': 'big 7s classic edition',
 'big bucks': 'big bucks',
 'big city 4s poker': 'big city 4s poker',
 "big city 5's jackpot": 'big city 5s jackpot',
 'big dragon lounge': 'big dragon lounge',
 'big horn bucks': 'big horn bucks',
 'big juan': 'big juan',
 'big ocean jackpots': 'big ocean jackpots',
 'big tiger': 'big tiger',
 'bighorn buck$': 'bighorn bucks',
 'birds of beauty': 'birds of beauty',
 'black cherry': 'black cherry',
 'black magic': 'black magic',
 'black stallion': 'black stallion',
 'black tie': 'black tie',
 'black widow': 'black widow',
 'black widow - superstar': 'black widow superstar',
 'black widow grand': 'black widow grand',
 'blue blazes': 'blue blazes',
 'boingy beans bean to be wild': 'boingy beans bean to be wild',
 'boingy beans cool beans': 'boingy beans cool beans',
 'bomb blitz - apollo': 'bomb blitz apollo',
 'bomb blitz - medusa': 'bomb blitz medusa',
 'bonsai blessings': 'bonsai blessings',
 'bonus bells': 'bonus bells',
 'bonus pick poker': 'bonus pick poker',
 'bonus sizzling 7s': 'bonus sizzling 7s',
 'bonus wheel triple diamond': 'bonus wheel triple diamond',
 'bonus wheel triple fire': 'bonus wheel triple fire',
 'book of wonder': 'book of wonder',
 'bora bora breeze': 'bora bora breeze',
 'bring the boom - egypt': 'bring the boom egypt',
 'bring the boom - pirate bay': 'bring the boom pirate bay',
 'bubble craze': 'bubble craze',
 'buffalo burst': 'buffalo burst',
 'buffalo storm': 'buffalo storm',
 'buffet mania': 'buffet mania',
 'build a wheel poker': 'build a wheel poker',
 'bullseye bonanza': 'bullseye bonanza',
 'burning bars': 'burning bars',
 'cable car cash': 'cable car cash',
 'cai fu long': 'cai fu long',
 'cai qing grand': 'cai qing grand',
 'caifu zhi wu': 'caifu zhi wu',
 'captain payback': 'captain payback',
 'cash on reels': 'cash on reels',
 'cash up jackpots': 'cash up jackpots',
 'cat paws': 'cat paws',
 'cat peak': 'cat peak',
 'cats': 'cats',
 'cats n dogs': 'cats n dogs',
 'cats wild serengeti': 'cats wild serengeti',
 'cave king': 'cave king',
 'celtic magic': 'celtic magic',
 'chamber of treasures - meow de bao zang': 'chamber of treasures meow de bao zang',
 'chameleon 7s': 'chameleon 7s',
 'cheeky lil devil': 'cheeky lil devil',
 'cherries on top': 'cherries on top',
 'cherry cash deluxe': 'cherry cash deluxe',
 'chinese fortune': 'chinese fortune',
 'chinese treasure': 'chinese treasure',
 'city of lights': 'city of lights',
 'cleopatra - triple fortune link': 'cleopatra triple fortune link',
 'cleopatra gold free games': 'cleopatra gold free games',
 'cleopatra multiplay': 'cleopatra multiplay',
 'clovers & gold': 'clovers and gold',
 'clovers and gold': 'clovers and gold',
 'colossal titans': 'colossal titans',
 'connect & collect - fire stampede': 'connect and collect fire stampede',
 'connect & collect - firebird spirit': 'connect and collect firebird spirit',
 'cosmic cash': 'cosmic cash',
 'country cash': 'country cash',
 'country club cash': 'country club cash',
 'coyote moon - desert nights': 'coyote moon desert nights',
 'coyote moon classic hits': 'coyote moon classic hits',
 'crazy chameleon': 'crazy chameleon',
 'crown of egypt': 'crown of egypt',
 'crystal countdown': 'crystal countdown',
 'crystal kingdom': 'crystal kingdom',
 'da vinci diamonds dual play': 'da vinci diamonds dual play',
 'da vinci diamonds masterpiece': 'da vinci diamonds masterpiece',
 'dancing phoenix soaring dragon': 'dancing phoenix soaring dragon',
 'dancing reels salsa': 'dancing reels salsa',
 'dao zhang panda': 'dao zhang panda',
 'dark and stormy': 'dark and stormy',
 'deep pockets': 'deep pockets',
 'desert spirit': 'desert spirit',
 'diamond arrow': 'diamond arrow',
 'diamond crown king': 'diamond crown king',
 'diamond crown queen': 'diamond crown queen',
 'diamond fire': 'diamond fire',
 'diamond galaxy': 'diamond galaxy',
 'diamond jackpots': 'diamond jackpots',
 'diamond jackpots - double spin - triple double stars': 'diamond jackpots double spin triple double stars',
 'diamond mania': 'diamond mania',
 'diamond nights': 'diamond nights',
 'diamond queen': 'diamond queen',
 'diamond skies': 'diamond skies',
 'diamond tide jungle': 'diamond tide jungle',
 'diamond unlimited': 'diamond unlimited',
 'dianmu surging skies': 'dianmu surging skies',
 'dirty sallys free game': 'dirty sallys free game',
 'disco nights': 'disco nights',
 'divers dream pearl coast': 'divers dream pearl coast',
 'dogs': 'dogs',
 'dolphin bay': 'dolphin bay',
 'double 4x wild cherry': 'double 4x wild cherry',
 'double big wheel poker': 'double big wheel poker',
 'double black pearl': 'double black pearl',
 'double bungah': 'double bungah',
 'double chili mania - extremo': 'double chili mania extremo',
 'double chili mania respin': 'double chili mania respin',
 'double da vinci diamonds': 'double da vinci diamonds',
 'double diamond': 'double diamond',
 'double diamond 3x4x5x': 'double diamond 3x4x5x',
 'double diamond pinball': 'double diamond pinball',
 'double dinosaur': 'double dinosaur',
 'double gold': 'double gold',
 'double gold - rising respins': 'double gold rising respins',
 'double gold pink diamond': 'double gold pink diamond',
 'double gold/pink diamond': 'double gold pink diamond',
 'double hot peppers - spicy jackpots': 'double hot peppers spicy jackpots',
 'double jackpot jewels': 'double jackpot jewels',
 'double jeweled 7s': 'double jeweled 7s',
 'double red hot 7s': 'double red hot 7s',
 'double red white & blue': 'double red white and blue',
 'double rollin in the cash': 'double rollin in the cash',
 'double silver fire 777': 'double silver fire 777',
 'double strike - rising respins': 'double strike rising respins',
 'double wild cherry jackpot': 'double wild cherry jackpot',
 'dragon boat sunrise': 'dragon boat sunrise',
 'dragon boat sunset': 'dragon boat sunset',
 'dragon bowl': 'dragon bowl',
 'dragon dance': 'dragon dance',
 'dragon fairy': 'dragon fairy',
 'dragon lights - secret fortress': 'dragon lights secret fortress',
 'dragon lights gold - eternal peaks': 'dragon lights gold eternal peaks',
 'dragon lights gold - imperial tides': 'dragon lights gold imperial tides',
 'dragon lights gold eternal peaks': 'dragon lights gold eternal peaks',
 'dragon lights gold imperial tides': 'dragon lights gold imperial tides',
 'dragon lights mystical falls': 'dragon lights mystical falls',
 'dragon lights secret fortress': 'dragon lights secret fortress',
 'dragon of fortune': 'dragon of fortune',
 'dragon of fortune evolution': 'dragon of fortune evolution',
 'dragon spell': 'dragon spell',
 'dragon sphere': 'dragon sphere',
 'dragon storm': 'dragon storm',
 'dragonfly': 'dragonfly',
 'dragons vs pandas': 'dragons vs pandas',
 'dragons vs. pandas': 'dragons vs pandas',
 'dream card 3/5/10 play poker': 'dream card 3 5 10 play poker',
 'dream card poker': 'dream card poker',
 'dream pool': 'dream pool',
 'dreams of big money - fortune': 'dreams of big money fortune',
 'dreams of big money - wealth': 'dreams of big money wealth',
 'dreams of egypt': 'dreams of egypt',
 'dreamy genie': 'dreamy genie',
 'duck stamps': 'duck stamps',
 'ducks in a row': 'ducks in a row',
 'dynasty roulette': 'dynasty roulette',
 'egyptian dreams': 'egyptian dreams',
 'egyptian link - amasis treasure': 'egyptian link amasis treasure',
 "egyptian link - nefturi's treasures": 'egyptian link nefturis treasure',
 "egyptian link - ramosis' treasures": 'egyptian link ramosis treasure',
 'egyptian link nefturis treasure': 'egyptian link nefturis treasure',
 'egyptian link ramosis treasure': 'egyptian link ramosis treasure',
 'egyptian link ramosis� treasures': 'egyptian link ramosis treasure',
 'egyptian link ramosis��� treasures': 'egyptian link ramosis treasure',
 'egyptian link wealthy wilds - amasis treasure': 'egyptian link wealthy wilds amasis treasure',
 'egyptian stars': 'egyptian stars',
 'el gran festival': 'el gran festival',
 'electric butterfly': 'electric butterfly',
 'elephant king': 'elephant king',
 'elephants vs tigers': 'elephants vs tigers',
 'elephants vs. tigers': 'elephants vs tigers',
 'emerald rose': 'emerald rose',
 'empress of egypt': 'empress of egypt',
 'empresses vs emperors': 'empresses vs emperors',
 'enchanted glass': 'enchanted glass',
 'enchanted unicorn returns': 'enchanted unicorn returns',
 'eskimo money': 'eskimo money',
 'eternal link - princess empire': 'eternal link princess empire',
 'eternal link - warriors empire': 'eternal link warriors empire',
 'eternal waters': 'eternal waters',
 'everlasting turtle': 'everlasting turtle',
 'exotic island': 'exotic island',
 'extra hot poker': 'extra hot poker',
 'fa cai long': 'fa cai long',
 'face card frenzy poker': 'face card frenzy poker',
 'fame and fortune': 'fame and fortune',
 'festival fantastico': 'festival fantastico',
 'festival link - joy and luck': 'festival link joy and luck',
 'fever spins - rodeo rails': 'fever spins rodeo rails',
 'fever spins - samurai steam': 'fever spins samurai steam',
 'fire chariot': 'fire chariot',
 'fire diamonds': 'fire diamonds',
 'fire it up free games': 'fire it up free games',
 'fire opals': 'fire opals',
 'fire pearl': 'fire pearl',
 'fire poker': 'fire poker',
 'fire rubies': 'fire rubies',
 'fish on ultimate catch': 'fish on ultimate catch',
 'fishing bob 2': 'fishing bob 2',
 'five card plus 2 poker': 'five card plus 2 poker',
 'flaming pearl': 'flaming pearl',
 'flip & pay poker': 'flip and pay poker',
 'fort knox heist gemstone dreams': 'fort knox heist gemstone dreams',
 'fortune charm': 'fortune charm',
 'fortune coin boost': 'fortune coin boost',
 'fortune coin extra': 'fortune coin extra',
 'fortune coin free games': 'fortune coin free games',
 'fortune coin respin': 'fortune coin respin',
 'fortune cookie': 'fortune cookie',
 'fortune disc': 'fortune disc',
 'fortune fans': 'fortune fans',
 'fortune hunter': 'fortune hunter',
 'fortune ingot': 'fortune ingot',
 'fortune lanterns': 'fortune lanterns',
 'fortune link 4': 'fortune link 4',
 'fortune owl': 'fortune owl',
 'fortune rockets': 'fortune rockets',
 'fortune rooster': 'fortune rooster',
 'fortune storm': 'fortune storm',
 'fortune x poker': 'fortune x poker',
 'fountain of wishes': 'fountain of wishes',
 'four great chinese beauties': 'four great chinese beauties',
 "fox 'n hound": 'fox n hound',
 'freddy the lawn flamingo': 'freddy the lawn flamingo',
 'freedom bells': 'freedom bells',
 'frequent flyer': 'frequent flyer',
 'frog prince': 'frog prince',
 'frog princess': 'frog princess',
 'fruitastic': 'fruitastic',
 'fu gui hao men 2 - everlasting fortunes': 'fu gui hao men 2 everlasting fortunes',
 'fu gui hao men 2 - magnificent fortunes': 'fu gui hao men 2 magnificent fortunes',
 'fu gui hao men 3 - glorious tiger': 'fu gui hao men 3 glorious tiger',
 'fu gui hao men 3 - splendid dragon': 'fu gui hao men 3 splendid dragon',
 'fu gui hao men glorious fortunes': 'fu gui hao men glorious fortunes',
 'fu gui hao men splendid fortunes': 'fu gui hao men splendid fortunes',
 'fu gui ji li - animal treasure': 'fu gui ji li animal treasure',
 'fu ren wu': 'fu ren wu',
 'fu ru dong hai': 'fu ru dong hai',
 'fu tao peach of riches': 'fu tao peach of riches',
 'fuzzys fortune': 'fuzzys fortune',
 'garden party': 'garden party',
 'gems wild tiles': 'gems wild tiles',
 'genie wishes': 'genie wishes',
 'get eggcited': 'get eggcited',
 'ghostbusters 3d': 'ghostbusters 3d',
 'glacier hammer': 'glacier hammer',
 'goddess of desire': 'goddess of desire',
 'goddess of gold': 'goddess of gold',
 'gold & dragon': 'gold and dragon',
 'gold & dragon 2': 'gold and dragon 2',
 'gold and dragon ii': 'gold and dragon 2',
 'gold bar 7s': 'gold bar 7s',
 'gold bar 7s/red hot tamales': 'gold bar 7s red hot tamales',
 'gold party': 'gold party',
 'golden 12': 'golden 12',
 'golden dragon': 'golden dragon',
 'golden eagle': 'golden eagle',
 'golden egypt': 'golden egypt',
 'golden egypt grand': 'golden egypt grand',
 'golden egypt jackpots': 'golden egypt jackpots',
 'golden empress': 'golden empress',
 'golden gate': 'golden gate',
 'golden gecko': 'golden gecko',
 'golden goddess': 'golden goddess',
 'golden goddess .': 'golden goddess',
 'golden jungle grand': 'golden jungle grand',
 'golden knight': 'golden knight',
 'golden link golden eagle': 'golden link golden eagle',
 'golden link golden phoenix': 'golden link golden phoenix',
 'golden odyssey': 'golden odyssey',
 'golden pepper': 'golden pepper',
 'golden phoenix': 'golden phoenix',
 'golden respins triple double - butterfly 7s': 'golden respins triple double butterfly 7s',
 'golden respins triple double - lucky 7s': 'golden respins triple double lucky 7s',
 'golden rooster': 'golden rooster',
 'golden stag win zone': 'golden stag win zone',
 'golden tiger eternal dragon': 'golden tiger eternal dragon',
 'golden treasures': 'golden treasures',
 'goldify': 'goldify',
 'good times pay poker': 'good times pay poker',
 'grand buddha link': 'grand buddha link',
 'grand cat link': 'grand cat link',
 'great plains': 'great plains',
 'great white': 'great white',
 'greek lightning': 'greek lightning',
 'greenback attack wild peppers': 'greenback attack wild peppers',
 "griffin's throne": 'griffins throne',
 'griffins throne': 'griffins throne',
 'griffins throne grand': 'griffins throne grand',
 'guardian of the rose': 'guardian of the rose',
 'guardians of egypt': 'guardians of egypt',
 'guardians treasure': 'guardians treasure',
 'gypsy treasure': 'gypsy treasure',
 'hao yun fa': 'hao yun fa',
 'hao yun lun dragon': 'hao yun lun dragon',
 'hao yun lun lion': 'hao yun lun lion',
 'happy buddha': 'happy buddha',
 'happy camper': 'happy camper',
 'happy firecracker': 'happy firecracker',
 'happy lucky': 'happy lucky',
 'happy snake': 'happy snake',
 'harmony riches': 'harmony riches',
 'hawaiian diamonds': 'hawaiian diamonds',
 'haywire - supercharged': 'haywire supercharged',
 'haywire deluxe': 'haywire deluxe',
 'haywire poker': 'haywire poker',
 'haywire! extreme': 'haywire extreme',
 'heart of the amazon': 'heart of the amazon',
 'hephaestus cash eruption': 'hephaestus cash eruption',
 'hexbreaker': 'hexbreaker',
 'hexbreaker 3': 'hexbreaker 3',
 'hidden kingdom': 'hidden kingdom',
 'high on a hog': 'high on a hog',
 'high on the hog': 'high on the hog',
 'highliner': 'highliner',
 'holiday in venice': 'holiday in venice',
 'holy guacamole': 'holy guacamole',
 'honey bucks': 'honey bucks',
 'hoodoo link': 'hoodoo link',
 'hot bar 7s': 'hot bar 7s',
 'hot bingo my super sweet tooth': 'hot bingo my super sweet tooth',
 'hot flashes': 'hot flashes',
 'hot hit double diamond': 'hot hit double diamond',
 'hot hit ignite': 'hot hit ignite',
 'hot hit rainbow riches 7s free games': 'hot hit rainbow riches 7s free games',
 "hot hit sizzling 7's": 'hot hit sizzling 7s',
 'hot hit sizzling 7s': 'hot hit sizzling 7s',
 'hot hit sizzling 7s free games': 'hot hit sizzling 7s free games',
 'hot link blue moon ii express': 'hot link blue moon 2 express',
 'hot link money beans express': 'hot link money beans express',
 'hot poker dice stinkin rich': 'hot poker dice stinkin rich',
 'hot roll cash cash cash': 'hot roll cash cash cash',
 'hot roll fairy ring': 'hot roll fairy ring',
 'hot roll golden dice - pink diamond': 'hot roll golden dice pink diamond',
 'hot roll golden dice - triple red hot 7s': 'hot roll golden dice triple red hot 7s',
 'hot roll golden dice - wild red 7s': 'hot roll golden dice wild red 7s',
 'hot roll sizzling 7': 'hot roll sizzling 7',
 'hot roll triple double diamond': 'hot roll triple double diamond',
 'hot roulette enchanted unicorn': 'hot roulette enchanted unicorn',
 'hot roulette lobstermania': 'hot roulette lobstermania',
 'hot roulette lucky larry': 'hot roulette lucky larry',
 'hot roulette triple double diamonds': 'hot roulette triple double diamonds',
 'hot roulette triple red hot 7s': 'hot roulette triple red hot 7s',
 'house of jewels': 'house of jewels',
 'hyper bonus poker': 'hyper bonus poker',
 'hyper hits flower empress': 'hyper hits flower empress',
 'hyper hits longship legend': 'hyper hits longship legend',
 'hyper hits the golden bow': 'hyper hits the golden bow',
 'i dream of jeannie': 'i dream of jeannie',
 'i heart jackpots double shot': 'i heart jackpots double shot',
 'ice heist': 'ice heist',
 'ice hot': 'ice hot',
 'icy wilds deluxe': 'icy wilds deluxe',
 'in the chips red hot respin deluxe': 'in the chips red hot respin deluxe',
 'island dreams': 'island dreams',
 'island eyes': 'island eyes',
 'its a blast': 'its a blast',
 'its da bomb gopher be gone': 'its da bomb gopher be gone',
 'its magic amber': 'its magic amber',
 'its magic lilly': 'its magic lilly',
 'its magic ruby': 'its magic ruby',
 'jackpot 7s': 'jackpot 7s',
 'jade fortune': 'jade fortune',
 'jade rising': 'jade rising',
 'jalapeno pays': 'jalapeno pays',
 'jewel of arabia': 'jewel of arabia',
 'jin hu': 'jin hu',
 'jin tu fa cai': 'jin tu fa cai',
 'jin zhu - dragon': 'jin zhu dragon',
 'jin zhu - rooster': 'jin zhu rooster',
 'jolly roger': 'jolly roger',
 'jumpin jungle': 'jumpin jungle',
 'jungle tower': 'jungle tower',
 'jungle vip': 'jungle vip',
 'kangaroo crossing': 'kangaroo crossing',
 'keno': 'keno',
 'king cheetah': 'king cheetah',
 'king khufu': 'king khufu',
 'king khufu - dawn': 'king khufu dawn',
 'king of huaguo': 'king of huaguo',
 'king of olympus': 'king of olympus',
 'kingdom of gold': 'kingdom of gold',
 'kitty glitter grand': 'kitty glitter grand',
 'kodiak king': 'kodiak king',
 'kossack kash': 'kossack kash',
 'lady aces': 'lady aces',
 'lady and the tiger': 'lady and the tiger',
 'lady falconer deluxe': 'lady falconer deluxe',
 'lady lake': 'lady lake',
 'lady of the lake': 'lady of the lake',
 'lady swan': 'lady swan',
 'latino machino': 'latino machino',
 'legacy link - treasures of troy': 'legacy link treasures of troy',
 'legacy link - white orchid': 'legacy link white orchid',
 'legend of the 3x 2x peacock': 'legend of the 3x2x peacock',
 'legend of the 3x2x peacock': 'legend of the 3x2x peacock',
 'legend of the 3x2x phoenix': 'legend of the 3x2x phoenix',
 'legend of the 3x2x phoenix wheel': 'legend of the 3x2x phoenix wheel',
 'legends of asgard': 'legends of asgard',
 'legends of fire and water': 'legends of fire and water',
 'leopard lagoon': 'leopard lagoon',
 'liger loot': 'liger loot',
 'lil lady 2 wild wedding': 'lil lady 2 wild wedding',
 'lily of the valley': 'lily of the valley',
 'lion dance': 'lion dance',
 'little green men': 'little green men',
 'little green men return': 'little green men return',
 'lock and key': 'lock and key',
 'locks of luck': 'locks of luck',
 'loot booster': 'loot booster',
 'lotus flower': 'lotus flower',
 'lotus flower blossoms': 'lotus flower blossoms',
 'luck rising - brysons bounty': 'luck rising brysons bounty',
 'luck rising - shrine': 'luck rising shrine',
 'luck rising - temple': 'luck rising temple',
 'lucky bar free games': 'lucky bar free games',
 'lucky beans': 'lucky beans',
 'lucky buddha': 'lucky buddha',
 'lucky coin asian dreaming': 'lucky coin asian dreaming',
 'lucky coin atlantica': 'lucky coin atlantica',
 'lucky golden toad': 'lucky golden toad',
 'lucky hearts': 'lucky hearts',
 'lucky hot 7s free games deluxe': 'lucky hot 7s free games deluxe',
 'lucky koi': 'lucky koi',
 'lucky larry lobstermania classic hits': 'lucky larry lobstermania classic hits',
 "lucky larry's lobstermania 4": 'lucky larrys lobstermania 4',
 'lucky larrys lobstermania - ultimate treasures': 'lucky larrys lobstermania ultimate treasures',
 'lucky larrys lobstermania 4': 'lucky larrys lobstermania 4',
 'lucky larrys lobstermania 5': 'lucky larrys lobstermania 5',
 'lucky lion fish': 'lucky lion fish',
 'lucky llamas': 'lucky llamas',
 'lucky lock': 'lucky lock',
 'lucky ox': 'lucky ox',
 'lucky seven bars': 'lucky seven bars',
 'lucky suit poker': 'lucky suit poker',
 'lucky triple gold - shamrock sheamus': 'lucky triple gold shamrock sheamus',
 'lucky triple gold - smokeys den': 'lucky triple gold smokeys den',
 'lucky wealth cat': 'lucky wealth cat',
 'lulu and luigi': 'lulu and luigi',
 'lunar king': 'lunar king',
 'lunar queen': 'lunar queen',
 'maggie and the martians': 'maggie and the martians',
 'magic butterfly': 'magic butterfly',
 'magic diamond': 'magic diamond',
 'magic fortune': 'magic fortune',
 'magic link celestial queen': 'magic link celestial queen',
 'magic link stellar riches': 'magic link stellar riches',
 'magic of the nile': 'magic of the nile',
 'magic orchid': 'magic orchid',
 'magic rockets - dragon': 'magic rockets dragon',
 'magic rockets - tiger': 'magic rockets tiger',
 'magic treasures dragon': 'magic treasures dragon',
 'magic treasures gold - emperor': 'magic treasures gold emperor',
 'magic treasures gold - empress': 'magic treasures gold empress',
 'magic treasures lock & respin - lion': 'magic treasures lock and respin lion',
 'magic treasures lock & respin - panther': 'magic treasures lock and respin panther',
 'magic treasures tiger': 'magic treasures tiger',
 'magic wishes': 'magic wishes',
 'magically wild': 'magically wild',
 'magnificent phoenix': 'magnificent phoenix',
 'maharajas money': 'maharajas money',
 'maid of money': 'maid of money',
 'majestic beauty': 'majestic beauty',
 'majestic stallion': 'majestic stallion',
 'major multipliers poker': 'major multipliers poker',
 'malibu': 'malibu',
 'maltese fortune': 'maltese fortune',
 'master of olympus': 'master of olympus',
 'matterhorn': 'matterhorn',
 'max action poker': 'max action poker',
 'mayan riches': 'mayan riches',
 'mayan riches ixchel': 'mayan riches ixchel',
 'mega wheel': 'mega wheel',
 'midnight treasure': 'midnight treasure',
 'mighty mammoth': 'mighty mammoth',
 'mighty miner': 'mighty miner',
 'mighty miner - diamond': 'mighty miner diamond',
 'mighty miner - gold': 'mighty miner gold',
 'mighty miner extreme': 'mighty miner extreme',
 'miners dream - blue mountain': 'miners dream blue mountain',
 'miners dream - red mountain': 'miners dream red mountain',
 'mistress of egypt': 'mistress of egypt',
 'mistress of egypt grand': 'mistress of egypt grand',
 'mistress of olympus': 'mistress of olympus',
 'mistress of the west': 'mistress of the west',
 'money comb': 'money comb',
 'money island buccaneer ruby': 'money island buccaneer ruby',
 'money mover panther pride': 'money mover panther pride',
 'money mover tiger pride': 'money mover tiger pride',
 'money storm classic hits': 'money storm classic hits',
 'moon spirit': 'moon spirit',
 'moon warriors': 'moon warriors',
 'moving multiplier poker': 'moving multiplier poker',
 'multi hand blackjack': 'multi hand blackjack',
 'multi strike platinum poker': 'multi strike platinum poker',
 'multi win poker': 'multi win poker',
 'multipliers rising poker': 'multipliers rising poker',
 'mummy link - dawn': 'mummy link dawn',
 'mummy link - night': 'mummy link night',
 'mystery of the clockwork': 'mystery of the clockwork',
 'mystery of the congo': 'mystery of the congo',
 'mystery of the lamp enchanted palace': 'mystery of the lamp enchanted palace',
 'mystery of the lamp mystical gardens': 'mystery of the lamp mystical gardens',
 'mystery of the talisman pays': 'mystery of the talisman pays',
 'mystical mermaid returns': 'mystical mermaid returns',
 'mystical princess': 'mystical princess',
 'mystical treasures lion': 'mystical treasures lion',
 'mystical treasures panther': 'mystical treasures panther',
 'nefertiti': 'nefertiti',
 'night sky': 'night sky',
 'nordic journey': 'nordic journey',
 'nuttin to it': 'nuttin to it',
 'ocean magic': 'ocean magic',
 'ocean magic bubble boost': 'ocean magic bubble boost',
 'ocean magic grand': 'ocean magic grand',
 'ocean magic ultra': 'ocean magic ultra',
 'ocean spirit': 'ocean spirit',
 'old money': 'old money',
 'oodles of poodles': 'oodles of poodles',
 'otter king': 'otter king',
 'outback cashback': 'outback cashback',
 'outgrow - spooky moon': 'outgrow spooky moon',
 'outgrow - tropical moon': 'outgrow tropical moon',
 'overloot': 'overloot',
 'panda park': 'panda park',
 'panda queen': 'panda queen',
 'passion flowers': 'passion flowers',
 'pearl bay': 'pearl bay',
 'peggle': 'peggle',
 'pelican pays': 'pelican pays',
 'pets': 'pets',
 'phantom of the opera': 'phantom of the opera',
 "pharaoh's gold": 'pharaohs gold',
 'pharaohs gold': 'pharaohs gold',
 'pharaohs of the nile': 'pharaohs of the nile',
 'phoenix bowl': 'phoenix bowl',
 'phoenix gold grand': 'phoenix gold grand',
 'phone tag': 'phone tag',
 'pick a multiplier poker': 'pick a multiplier poker',
 'pick a pair poker': 'pick a pair poker',
 'pig and pug': 'pig and pug',
 'piggy blessings': 'piggy blessings',
 'pinball double gold': 'pinball double gold',
 'pinball gold': 'pinball gold',
 'pinball grand triple diamond': 'pinball grand triple diamond',
 'pink diamond': 'pink diamond',
 'pink diamond - rising respins': 'pink diamond rising respins',
 "pirate link - drake's treasure": 'pirate link drakes treasure',
 "pirate link - mary's treasure": 'pirate link marys treasure',
 'pirate link drakes treasure': 'pirate link drakes treasure',
 'pirate link marys treasure': 'pirate link marys treasure',
 'polar plunge': 'polar plunge',
 'poppin multipliers poker': 'poppin multipliers poker',
 'port royale': 'port royale',
 'poseidon': 'poseidon',
 'power charms': 'power charms',
 'power coins panda': 'power coins panda',
 'power coins pirate': 'power coins pirate',
 'power of cash': 'power of cash',
 'power quads': 'power quads',
 'power stacks poker': 'power stacks poker',
 'powerhouse': 'powerhouse',
 'powerhouse plus poker': 'powerhouse plus poker',
 'price check': 'price check',
 'prince of lightning': 'prince of lightning',
 'prism paradise': 'prism paradise',
 'prosperity link blessings': 'prosperity link blessings',
 'prosperity link cai yun heng tong': 'prosperity link cai yun heng tong',
 'prosperity link wan shi ru yi': 'prosperity link wan shi ru yi',
 'prosperity pearl': 'prosperity pearl',
 'pyramid progressives': 'pyramid progressives',
 'pyramid treasures': 'pyramid treasures',
 'queen isabella': 'queen isabella',
 'queen of olympus': 'queen of olympus',
 'queenie': 'queenie',
 'quick quads poker': 'quick quads poker',
 'rainbow goddess': 'rainbow goddess',
 'raise the sails': 'raise the sails',
 'realm of beauty': 'realm of beauty',
 'red hot jackpots double diamond': 'red hot jackpots double diamond',
 'red hot respin double diamond': 'red hot respin double diamond',
 'red hot tamales pinata bash': 'red hot tamales pinata bash',
 'red hot tamales respin': 'red hot tamales respin',
 'red hottie': 'red hottie',
 'red rhino': 'red rhino',
 'red rooster': 'red rooster',
 'red white & blue 2x 3x': 'red white and blue 2x 3x',
 'reel jazz': 'reel jazz',
 'regal link lion': 'regal link lion',
 'regal link raven': 'regal link raven',
 'regal riches': 'regal riches',
 'rembrandt riches': 'rembrandt riches',
 'rembrandt riches masterpiece': 'rembrandt riches masterpiece',
 'rich girl': 'rich girl',
 'riches of angkor': 'riches of angkor',
 'rising phoenix': 'rising phoenix',
 'rising reels link - mayan magic': 'rising reels link mayan magic',
 'rising reels link mayan magic': 'rising reels link mayan magic',
 'rising rockets emperor': 'rising rockets emperor',
 'rising rockets empress': 'rising rockets empress',
 'rising spins': 'rising spins',
 'rocket man': 'rocket man',
 'romeo and juliet': 'romeo and juliet',
 'roulette': 'roulette',
 'royal lion': 'royal lion',
 'royals egypt': 'royals egypt',
 'ruby reels': 'ruby reels',
 'sakura tree': 'sakura tree',
 'salsa 7s red hot respin deluxe': 'salsa 7s red hot respin deluxe',
 'samurai 888 katsumi': 'samurai 888 katsumi',
 'samurai 888 spin - kenji': 'samurai 888 spin kenji',
 'samurai 888 spin - sakura': 'samurai 888 spin sakura',
 'samurai 888 takeo': 'samurai 888 takeo',
 'samurai dynasty': 'samurai dynasty',
 'samurai rise 888 - haruki': 'samurai rise 888 haruki',
 'samurai rise 888 - kimiko': 'samurai rise 888 kimiko',
 'san xing riches': 'san xing riches',
 'sands of gold': 'sands of gold',
 'scarab': 'scarab',
 'scarab grand': 'scarab grand',
 'scarab link - cleora': 'scarab link cleora',
 'scarab link - nefturi': 'scarab link nefturi',
 'scarab multiplay': 'scarab multiplay',
 'sea cash': 'sea cash',
 'secret lagoon': 'secret lagoon',
 'secrets of babylon': 'secrets of babylon',
 'secrets of the forest': 'secrets of the forest',
 'sevens': 'sevens',
 'shadow masquerade': 'shadow masquerade',
 'shake your booty': 'shake your booty',
 'shambala': 'shambala',
 'shamrock stars': 'shamrock stars',
 'shanghai fantasy': 'shanghai fantasy',
 'shishenli pick a reel': 'shishenli pick a reel',
 'show dogs': 'show dogs',
 'siberian storm 2': 'siberian storm 2',
 'silver skies': 'silver skies',
 'silverhawk': 'silverhawk',
 'sizzling 7s piggy parade': 'sizzling 7s piggy parade',
 'sizzling rubies': 'sizzling rubies',
 'sizzling wilds - classic edition': 'sizzling wilds classic edition',
 'sky pilot': 'sky pilot',
 'slammin multipliers poker': 'slammin multipliers poker',
 'soaring wings': 'soaring wings',
 'solar disc': 'solar disc',
 'solar disc grand': 'solar disc grand',
 'songbirds': 'songbirds',
 'southern belle': 'southern belle',
 'sphinx': 'sphinx',
 'sphinx link cleora': 'sphinx link cleora',
 'sphinx link nefturi': 'sphinx link nefturi',
 'sphinx wild': 'sphinx wild',
 'spitfire triple red hot bells': 'spitfire triple red hot bells',
 'spitfire triple stars free game': 'spitfire triple stars free game',
 'split n win link cats': 'split n win link cats',
 'split n win link fortunes': 'split n win link fortunes',
 'spooktacular': 'spooktacular',
 'stack em high poker': 'stack em high poker',
 'stack the deck poker': 'stack the deck poker',
 'star goddess': 'star goddess',
 'star rise': 'star rise',
 'star stax eastern lights': 'star stax eastern lights',
 'star stax vegas nights': 'star stax vegas nights',
 'state fair 2': 'state fair 2',
 'stinkin rich': 'stinkin rich',
 'stinkin rich skunks gone wild': 'stinkin rich skunks gone wild',
 'stinkin rich stink link - hawaii': 'stinkin rich stink link hawaii',
 'stinkin rich stink link - wild west': 'stinkin rich stink link wild west',
 "stinkin' rich - skunks gone wild": 'stinkin rich skunks gone wild',
 'sultan of mars': 'sultan of mars',
 'super fusion 7s': 'super fusion 7s',
 'super hand poker': 'super hand poker',
 'super happy fortune cat': 'super happy fortune cat',
 'super hot roll poker': 'super hot roll poker',
 'super money storm': 'super money storm',
 'super reel charms': 'super reel charms',
 'super sallys shrimpmania 5': 'super sallys shrimpmania 5',
 'super star poker 2': 'super star poker 2',
 'super star poker ii': 'super star poker 2',
 'super times pay 25/50/100 play': 'super times pay 25 50 100 play',
 'super times pay keno': 'super times pay keno',
 'super times pay poker': 'super times pay poker',
 'super times pay super stacks': 'super times pay super stacks',
 'super times pay wheel poker': 'super times pay wheel poker',
 'super wheelmania colossal cats': 'super wheelmania colossal cats',
 'super wheelmania legendary beauty': 'super wheelmania legendary beauty',
 'super wheelmania link treasure fish': 'super wheelmania link treasure fish',
 'super wheelmania link vip riches': 'super wheelmania link vip riches',
 'super wheelmania treasure fish': 'super wheelmania treasure fish',
 'super wheelmania vip riches': 'super wheelmania vip riches',
 'sure fire jackpots': 'sure fire jackpots',
 'swan lake': 'swan lake',
 'sweet 3': 'sweet 3',
 'sweet riches': 'sweet riches',
 'tailgate party': 'tailgate party',
 'tales of riches': 'tales of riches',
 'tap house poker': 'tap house poker',
 'temple of fire': 'temple of fire',
 'temple of fire multiplay': 'temple of fire multiplay',
 'temujin treasure': 'temujin treasure',
 'texas tea - strike it rich': 'texas tea strike it rich',
 'texas tea strike it rich': 'texas tea strike it rich',
 'texas tina': 'texas tina',
 'the amazing money machine gold': 'the amazing money machine gold',
 'the amazing money machine sapphire': 'the amazing money machine sapphire',
 'the count of monte cristo': 'the count of monte cristo',
 'the fates': 'the fates',
 'the great detective': 'the great detective',
 'the legend of robin and marian': 'the legend of robin and marian',
 'the munsters': 'the munsters',
 'the shoe shop': 'the shoe shop',
 'the wild life': 'the wild life',
 'three kings deluxe': 'three kings deluxe',
 'throne of zinon': 'throne of zinon',
 'tiger & dragon cash on reels': 'tiger and dragon cash on reels',
 'tiger empire': 'tiger empire',
 'top stop': 'top stop',
 'totally puzzled': 'totally puzzled',
 'toucan tropics': 'toucan tropics',
 'treasure blast - balloon': 'treasure blast balloon',
 'treasure blast - fleet': 'treasure blast fleet',
 'treasure box - dynasty': 'treasure box dynasty',
 'treasure box - kingdom': 'treasure box kingdom',
 'treasure fleet': 'treasure fleet',
 'treasures - aztec temple': 'treasures aztec temple',
 'treasures - chinese palace': 'treasures chinese palace',
 'treasures of tao': 'treasures of tao',
 'treasures of the pyramid': 'treasures of the pyramid',
 'treasures of troy': 'treasures of troy',
 'triple big wheel - 6x4x3x2x vip': 'triple big wheel 6x4x3x2x vip',
 'triple big wheel - triple double diamond': 'triple big wheel triple double diamond',
 'triple big wheel 6x 4x 3x 2x vip': 'triple big wheel 6x4x3x2x vip',
 'triple big wheel tales of the legends dairy': 'triple big wheel tales of the legends dairy',
 'triple big wheel triple double dmnd': 'triple big wheel triple double dmnd',
 'triple black opals': 'triple black opals',
 'triple bucks': 'triple bucks',
 'triple diamond deluxe plus': 'triple diamond deluxe plus',
 'triple diamond strike': 'triple diamond strike',
 'triple double diamonds progressive free games': 'triple double diamonds progressive free games',
 'triple double dollars free games': 'triple double dollars free games',
 'triple double gold bars': 'triple double gold bars',
 'triple double gold doubloon': 'triple double gold doubloon',
 'triple double jackpot jewels': 'triple double jackpot jewels',
 'triple double red hot bells free games': 'triple double red hot bells free games',
 'triple double red white & blue': 'triple double red white and blue',
 'triple double scatter pays': 'triple double scatter pays',
 'triple double sizzling 7s': 'triple double sizzling 7s',
 'triple egypt': 'triple egypt',
 'triple fortune dragon - twin': 'triple fortune dragon twin',
 'triple fortune dragon gold': 'triple fortune dragon gold',
 'triple fortune link hong se cai fu red riches': 'triple fortune link hong se cai fu red riches',
 'triple fortune link huang se cai fu yellow riches': 'triple fortune link huang se cai fu yellow riches',
 'triple gold bars/triple strike/triple gold': 'triple gold bars triple strike triple gold',
 'triple golden panther': 'triple golden panther',
 'triple jackpot jewels': 'triple jackpot jewels',
 'triple red hot 7s free games': 'triple red hot 7s free games',
 'triple top dollar': 'triple top dollar',
 "tully's treasure hunt": 'tullys treasure hunt',
 'turn of chance': 'turn of chance',
 'turtle coast': 'turtle coast',
 'twin strike': 'twin strike',
 'twin win': 'twin win',
 'twin win diamond': 'twin win diamond',
 'twin win pearl': 'twin win pearl',
 'ultimate 4 of a kind bonus poker': 'ultimate 4 of a kind bonus poker',
 "ultimate sevens pharaoh's 777": 'ultimate sevens pharaohs 777',
 'ultimate x gold': 'ultimate x gold',
 'ultimate x keno': 'ultimate x keno',
 'uncle sam': 'uncle sam',
 'unicorn magic': 'unicorn magic',
 'valley of gold mayan': 'valley of gold mayan',
 'valley of the kings': 'valley of the kings',
 'vamps': 'vamps',
 'vanishing act': 'vanishing act',
 'vegas city of riches': 'vegas city of riches',
 'vesuvius': 'vesuvius',
 'vivaldis seasons': 'vivaldis seasons',
 'volatile': 'volatile',
 'water dragons': 'water dragons',
 'way too cute forest': 'way too cute forest',
 'wedding party': 'wedding party',
 'welcome home': 'welcome home',
 'wheel hottie': 'wheel hottie',
 'wheel poker deluxe': 'wheel poker deluxe',
 'wheel shot electric unicorn': 'wheel shot electric unicorn',
 'white hot 7s': 'white hot 7s',
 'white orchid': 'white orchid',
 'white pearl': 'white pearl',
 'wild bear paws': 'wild bear paws',
 'wild fury free games': 'wild fury free games',
 'wild fury jackpot': 'wild fury jackpot',
 'wild garden': 'wild garden',
 'wild hits - clover boost': 'wild hits clover boost',
 'wild hits - pearl boost': 'wild hits pearl boost',
 'wild hits clover': 'wild hits clover',
 'wild hits pearls': 'wild hits pearls',
 'wild mermaid': 'wild mermaid',
 'wild pinata': 'wild pinata',
 'wild pirates': 'wild pirates',
 'wild pride': 'wild pride',
 'wild surf': 'wild surf',
 'wild totem': 'wild totem',
 'wild wild wild poker': 'wild wild wild poker',
 'wild wind ups': 'wild wind ups',
 'wildwood': 'wildwood',
 'winners choice 2': 'winners choice 2',
 'winners only wild koi': 'winners only wild koi',
 'winning colors red white and blue': 'winning colors red white and blue',
 'wish mistress': 'wish mistress',
 'witches riches': 'witches riches',
 'wizard riches': 'wizard riches',
 'wizards gold': 'wizards gold',
 'wolf peak': 'wolf peak',
 'wolf princess': 'wolf princess',
 'wolf ridge': 'wolf ridge',
 'wolf run': 'wolf run',
 'wolf run - wild moon': 'wolf run wild moon',
 'wolf run eclipse': 'wolf run eclipse',
 'wolf run gold': 'wolf run gold',
 'wolf run jackpots free games': 'wolf run jackpots free games',
 'wolf run respin': 'wolf run respin',
 'wonder coins - phoenix': 'wonder coins phoenix',
 'wonders of africa': 'wonders of africa',
 'woolly mammoth': 'woolly mammoth',
 'wrecking ball': 'wrecking ball',
 'wu da jiang': 'wu da jiang',
 'wu dragon': 'wu dragon',
 'wu wang zhe': 'wu wang zhe',
 'wu wu coins big buddha': 'wu wu coins big buddha',
 'wu zhao link - cherished treasures': 'wu zhao link cherished treasures',
 'wu zhao link - zodiac kingdom': 'wu zhao link zodiac kingdom',
 'yin yang': 'yin yang',
 'ying cai shen fortunes fulfilled': 'ying cai shen fortunes fulfilled',
 'ying fu wa greeting wealth': 'ying fu wa greeting wealth',
 'ying fu wa incoming treasure': 'ying fu wa incoming treasure',
 'zhao cai zhu - gettin piggy with it': 'zhao cai zhu gettin piggy with it',
 'zhao cai zhu - yo ho hog': 'zhao cai zhu yo ho hog',
 'zodiac dragon': 'zodiac dragon',
 'zodiac lion': 'zodiac lion'}

SALES_NAME_MAP = {'1421 voyages of zheng he': '1421 voyages of zheng he',
 '2x3x4x5x dragon': '2x3x4x5x dragon',
 '2x3x4x5x gold': '2x3x4x5x gold',
 '2x3x4x5x red hot 7s': '2x3x4x5x red hot 7s',
 '3x wild diamonds': '3x wild diamonds',
 '5 star': '5 star',
 '5x gold': '5x gold',
 '5x pay lucky 7': '5x pay lucky 7s',
 '5x5x5x jackpot gem': '5x5x5x jackpot gem',
 '888 empress': '888 empress',
 '888 panda': '888 panda',
 '99 riches': '99 riches',
 'arctic fox': 'arctic fox',
 'aztec adventures': 'aztec adventures',
 'aztec temple treasures': 'treasures aztec temple',
 'babooshka': 'babooshka',
 'bao zang bao zha': 'bao zang bao zha',
 'beauty of egypt jackpots': 'beauty of egypt jackpots',
 'benny big game': 'benny big game',
 'big bucks bonus': None,
 'big juan dual hhrag': None,
 'big ocean jackpots': 'big ocean jackpots',
 'big tiger': 'big tiger',
 'black widow': 'black widow',
 'black widow grand': 'black widow grand',
 'blue blazes': 'blue blazes',
 'boingy beans bean to be wild': 'boingy beans bean to be wild',
 'bonus wheel triple diamond': 'bonus wheel triple diamond',
 'bring the boom egypt': 'bring the boom egypt',
 'bring the boom pirate bay': 'bring the boom pirate bay',
 'buffalo burst': 'buffalo burst',
 'buffalo storm': 'buffalo storm',
 'buffet mania': 'buffet mania',
 'cable car cash multiway': None,
 'caifu zhi wu': 'caifu zhi wu',
 'cat peak': 'cat peak',
 'cats': 'cats',
 'cats n dogs': 'cats n dogs',
 'cats wild serengeti prog': None,
 'chinese fortune': 'chinese fortune',
 'chinese palace treasures': 'treasures chinese palace',
 'chinese treasure': 'chinese treasure',
 'classic hits money storm': 'money storm classic hits',
 'cleopatra multiplay': 'cleopatra multiplay',
 'cleopatra triple fortune': None,
 'connect collect fire stampede': 'connect and collect fire stampede',
 'coyote moon desert nights': 'coyote moon desert nights',
 'da vinci diamonds masterpiece': 'da vinci diamonds masterpiece',
 'diamond diamond': None,
 'diamond fire': 'diamond fire',
 'diamond jackpots triple double stars': None,
 'diamond queen': 'diamond queen',
 'dianmu surging skies': 'dianmu surging skies',
 'divers dream pearl coast': 'divers dream pearl coast',
 'double diamond': 'double diamond',
 'double gold': 'double gold',
 'double red hot 7s': 'double red hot 7s',
 'double red white blue reel': None,
 'double rollin in the cash': 'double rollin in the cash',
 'double silver fire 777': 'double silver fire 777',
 'dragon bowl': 'dragon bowl',
 'dragon lights mystical falls': 'dragon lights mystical falls',
 'dragon lights secret fortress': 'dragon lights secret fortress',
 'dragon of fortune evolution': 'dragon of fortune evolution',
 'dragon of fortune multiway': None,
 'dragon spell': 'dragon spell',
 'dragons vs pandas': 'dragons vs pandas',
 'dreamy genie': 'dreamy genie',
 'dynamite blast passion flowers': None,
 'egyptian dreams': 'egyptian dreams',
 'egyptian link amasis treasures buy a bonus': None,
 'egyptian link nefturis treasures': 'egyptian link nefturis treasure',
 'egyptian stars': 'egyptian stars',
 'elephants vs tigers': 'elephants vs tigers',
 'emperors throne': None,
 'empresses vs emperors': 'empresses vs emperors',
 'enchanted unicorn returns': 'enchanted unicorn returns',
 'eternal link princess empire': 'eternal link princess empire',
 'eternal link warriors empire': 'eternal link warriors empire',
 'everlasting turtle': 'everlasting turtle',
 'fa cai long': 'fa cai long',
 'fame and fortune': 'fame and fortune',
 'fire it up': None,
 'firebird spirit': None,
 'fortune coin': None,
 'fortune coin extra': 'fortune coin extra',
 'fortune link 4 1': None,
 'fortune owl': 'fortune owl',
 'fortune rockets': 'fortune rockets',
 'fu gui hao men glorious fortunes': 'fu gui hao men glorious fortunes',
 'fu gui hao men ii everlasting fortunes': 'fu gui hao men 2 everlasting fortunes',
 'fu gui hao men ii magnificent fortunes': 'fu gui hao men 2 magnificent fortunes',
 'fu gui hao men splendid fortunes': 'fu gui hao men splendid fortunes',
 'fu gui ji li animal treasures': 'fu gui ji li animal treasure',
 'fu tao peach of riches': 'fu tao peach of riches',
 'genie wishes': 'genie wishes',
 'goddess of gold': 'goddess of gold',
 'gold bar 7s': 'gold bar 7s',
 'golden 12': 'golden 12',
 'golden eagle': 'golden eagle',
 'golden egypt': 'golden egypt',
 'golden egypt grand': 'golden egypt grand',
 'golden gate': 'golden gate',
 'golden goddess': 'golden goddess',
 'golden knight': 'golden knight',
 'golden phoenix': 'golden phoenix',
 'golden stag win zone': 'golden stag win zone',
 'goldify': 'goldify',
 'grand buddha': None,
 'greek lightning': 'greek lightning',
 'griffins throne grand': 'griffins throne grand',
 'guardians of egypt': 'guardians of egypt',
 'haywire deluxe': 'haywire deluxe',
 'haywire reel': None,
 'hexbreaker': 'hexbreaker',
 'hexbreaker 3': 'hexbreaker 3',
 'hidden kingdom': 'hidden kingdom',
 'honey bucks': 'honey bucks',
 'hot hit rainbow riches lucky 7s': None,
 'hot link blue moon ii express': 'hot link blue moon 2 express',
 'hot link money beans express': 'hot link money beans express',
 'hot roll fairy ring': 'hot roll fairy ring',
 'hyper hits flower empress': 'hyper hits flower empress',
 'hyper hits longship legend': 'hyper hits longship legend',
 'hyper hits the golden bow': 'hyper hits the golden bow',
 'island eyes': 'island eyes',
 'its da bomb gophersbgone 3r1l3': None,
 'jackpot 7s free games 20l100c': None,
 'jewel of arabia': 'jewel of arabia',
 'jin hu': 'jin hu',
 'jin tu fa cai': 'jin tu fa cai',
 'jin zhu dragon': 'jin zhu dragon',
 'jin zhu rooster': 'jin zhu rooster',
 'king khufu dawn': 'king khufu dawn',
 'king of huaguo': 'king of huaguo',
 'king of olympus': 'king of olympus',
 'kitty glitter grand': 'kitty glitter grand',
 'lion dance': 'lion dance',
 'lotus flower': 'lotus flower',
 'luck rising brysons bounty': 'luck rising brysons bounty',
 'lucky buddha': 'lucky buddha',
 'lucky coin asian dreaming': 'lucky coin asian dreaming',
 'lucky coin atlantica': 'lucky coin atlantica',
 'lucky larrys lobstermania 4': 'lucky larrys lobstermania 4',
 'lucky larrys lobstermania 5': 'lucky larrys lobstermania 5',
 'lucky larrys lobstermania ultimate treasure': 'lucky larrys lobstermania ultimate treasures',
 'lucky lion fish': 'lucky lion fish',
 'lucky lock': 'lucky lock',
 'lucky ox': 'lucky ox',
 'lucky triple goldsmokeys den': 'lucky triple gold smokeys den',
 'lucky wealth cat': 'lucky wealth cat',
 'lunar queen': 'lunar queen',
 'magic link celestial queen': 'magic link celestial queen',
 'magic link stellar riches': 'magic link stellar riches',
 'magic rockets dragon': 'magic rockets dragon',
 'magic rockets tiger': 'magic rockets tiger',
 'magic treasures dragon': 'magic treasures dragon',
 'magic treasures gold emperor': 'magic treasures gold emperor',
 'magic treasures gold empress': 'magic treasures gold empress',
 'magic treasures lock respin lion': 'magic treasures lock and respin lion',
 'magic wishes': 'magic wishes',
 'magnificent phoenix': 'magnificent phoenix',
 'maharajas money': 'maharajas money',
 'majestic stallion 80 ctc': None,
 'mayan riches ixchel': 'mayan riches ixchel',
 'mayan riches kukulkan': None,
 'midnight treasures': 'midnight treasure',
 'mighty miner': 'mighty miner',
 'miners dream blue mountain': 'miners dream blue mountain',
 'mistress of egypt grand': 'mistress of egypt grand',
 'money mover panther pride': 'money mover panther pride',
 'money mover tiger pride': 'money mover tiger pride',
 'moon spirit': 'moon spirit',
 'mummy link dawn': 'mummy link dawn',
 'mummy link night': 'mummy link night',
 'munster match': None,
 'mystery of the lamp enchanted palace': 'mystery of the lamp enchanted palace',
 'mystery of the lamp mystical gardens': 'mystery of the lamp mystical gardens',
 'mystical treasures lion': 'mystical treasures lion',
 'ocean magic': 'ocean magic',
 'ocean magic bubble boost': 'ocean magic bubble boost',
 'ocean spirit': 'ocean spirit',
 'outgrow spooky moon': 'outgrow spooky moon',
 'outgrow tropical moon': 'outgrow tropical moon',
 'overloot': 'overloot',
 'pharaoh 7s': None,
 'phoenix bowl': 'phoenix bowl',
 'phone tag': 'phone tag',
 'pig and pug': 'pig and pug',
 'pink diamond': 'pink diamond',
 'pirate link drakes treasure': 'pirate link drakes treasure',
 'pirate link marys treasure': 'pirate link marys treasure',
 'progressive bonanza 3x2x legend of the peacock': None,
 'prosperity link blessings skyrise': None,
 'prosperity link cai yun heng tong': 'prosperity link cai yun heng tong',
 'prosperity link wan shi ru yi': 'prosperity link wan shi ru yi',
 'prosperity pearl': 'prosperity pearl',
 'queen of olympus': 'queen of olympus',
 'queenie': 'queenie',
 'raise the sails': 'raise the sails',
 'red hot jackpots double diamond 2': None,
 'red hottie': 'red hottie',
 'reeledge tullys treasure hunt': None,
 'regal link lion': 'regal link lion',
 'regal link raven': 'regal link raven',
 'regal riches': 'regal riches',
 'rembrandt riches masterpiece': 'rembrandt riches masterpiece',
 'rising phoenix': 'rising phoenix',
 'rising reels link mayan magic': 'rising reels link mayan magic',
 'rising rockets empress': 'rising rockets empress',
 'rising spins': 'rising spins',
 'royals egypt': 'royals egypt',
 'samurai 888 katsumi': 'samurai 888 katsumi',
 'samurai 888 rise kimiko': 'samurai rise 888 kimiko',
 'samurai 888 spin kenji': 'samurai 888 spin kenji',
 'samurai 888 spin sakura': 'samurai 888 spin sakura',
 'san xing riches': 'san xing riches',
 'scarab': 'scarab',
 'scarab grand': 'scarab grand',
 'scarab link cleora': 'scarab link cleora',
 'scarab link nefturi': 'scarab link nefturi',
 'scarab multiplay': 'scarab multiplay',
 'secrets of the forest': 'secrets of the forest',
 'sevens': 'sevens',
 'shambala': 'shambala',
 'shamrock stars': 'shamrock stars',
 'siberian storm 2': 'siberian storm 2',
 'sizzling 7s piggy parade': 'sizzling 7s piggy parade',
 'solar disc grand': 'solar disc grand',
 'sphinx': 'sphinx',
 'sphinx link cleora': 'sphinx link cleora',
 'sphinx link nefturi': 'sphinx link nefturi',
 'split n win linkfortune': 'split n win link fortunes',
 'star goddess': 'star goddess',
 'stinkin rich': 'stinkin rich',
 'stinkin richskunks gone wild': 'stinkin rich skunks gone wild',
 'super sallys shrimpmania 5': 'super sallys shrimpmania 5',
 'super times pay 3510 play poker action': None,
 'super wheelmania link legendary beauty': None,
 'super wheelmania link treasure fish': 'super wheelmania link treasure fish',
 'super wheelmania link vip riches': 'super wheelmania link vip riches',
 'super x lucky x 2x3x4x5x x pay': None,
 'surefire jackpots': 'sure fire jackpots',
 'temple of fire': 'temple of fire',
 'temujin treasures': 'temujin treasure',
 'the amazing money machine gold': 'the amazing money machine gold',
 'the amazing money machine sapphire': 'the amazing money machine sapphire',
 'tiger and dragon cash on reels': 'tiger and dragon cash on reels',
 'tiger and dragon link cash on reels': None,
 'treasure blast balloon': 'treasure blast balloon',
 'treasure box dynasty': 'treasure box dynasty',
 'treasure box kingdom': 'treasure box kingdom',
 'treasure fleet': 'treasure fleet',
 'treasures of troy multiway': None,
 'triple big wheel triple double diamond free games': None,
 'triple double red white blue 2x3x red white blue': None,
 'triple fortune dragon gold': 'triple fortune dragon gold',
 'triple fortune link hong se cai fu red riches': 'triple fortune link hong se cai fu red riches',
 'triple fortune link huang se cai fu yellow riches': 'triple fortune link huang se cai fu yellow riches',
 'triple jackpot jewels': 'triple jackpot jewels',
 'twin strike': 'twin strike',
 'twin win': 'twin win',
 'twin win pearl': 'twin win pearl',
 'wheel shot electric unicorn': 'wheel shot electric unicorn',
 'white hot 7': 'white hot 7s',
 'white orchid': 'white orchid',
 'wild hits clovers': 'wild hits clover',
 'wild hits pearls': 'wild hits pearls',
 'wild pinata': 'wild pinata',
 'wild pride': 'wild pride',
 'wild surf': 'wild surf',
 'wild tiles gems': 'gems wild tiles',
 'wish mistress': 'wish mistress',
 'wolf peak': 'wolf peak',
 'wolf run': 'wolf run',
 'wolf run eclipse': 'wolf run eclipse',
 'wolf run gold': 'wolf run gold',
 'wolf run wild moon': 'wolf run wild moon',
 'wonders of africa': 'wonders of africa',
 'wu da jiang': 'wu da jiang',
 'wu dragon': 'wu dragon',
 'wu wu coins big buddha': 'wu wu coins big buddha',
 'zhao cai zhu gettin piggy with it': 'zhao cai zhu gettin piggy with it',
 'zhao cai zhu yo ho hog': 'zhao cai zhu yo ho hog',
 'zodiac dragon': 'zodiac dragon',
 'zodiac lion': 'zodiac lion'}

SALES_CROSSWALK = {'1421 voyages of zheng he': {'canonical_game_name': '1421 voyages of zheng he',
                              'eilers_names': ['1421 voyages of zheng he'],
                              'feature_names': [],
                              'status': 'matched_eilers_only'},
 '2x3x4x5x dragon': {'canonical_game_name': '2x3x4x5x dragon',
                     'eilers_names': ['2x3x4x5x dragon'],
                     'feature_names': [],
                     'status': 'matched_eilers_only'},
 '2x3x4x5x gold': {'canonical_game_name': '2x3x4x5x gold',
                   'eilers_names': ['2x3x4x5x gold'],
                   'feature_names': [],
                   'status': 'matched_eilers_only'},
 '2x3x4x5x red hot 7s': {'canonical_game_name': '2x3x4x5x red hot 7s',
                         'eilers_names': ['2x3x4x5x red hot 7s'],
                         'feature_names': [],
                         'status': 'matched_eilers_only'},
 '3x wild diamonds': {'canonical_game_name': '3x wild diamonds',
                      'eilers_names': ['3x wild diamonds'],
                      'feature_names': ['3x wild diamonds'],
                      'status': 'matched_all_three'},
 '5 star': {'canonical_game_name': '5 star',
            'eilers_names': ['5 star'],
            'feature_names': [],
            'status': 'matched_eilers_only'},
 '5x gold': {'canonical_game_name': '5x gold',
             'eilers_names': ['5x gold'],
             'feature_names': [],
             'status': 'matched_eilers_only'},
 '5x pay lucky 7': {'canonical_game_name': '5x pay lucky 7s',
                    'eilers_names': ['5x pay lucky 7s'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 '5x5x5x jackpot gem': {'canonical_game_name': '5x5x5x jackpot gem',
                        'eilers_names': ['5x5x5x jackpot gem'],
                        'feature_names': ['5x5x5x jackpot gem'],
                        'status': 'matched_all_three'},
 '888 empress': {'canonical_game_name': '888 empress',
                 'eilers_names': ['888 empress'],
                 'feature_names': ['888 empress'],
                 'status': 'matched_all_three'},
 '888 panda': {'canonical_game_name': '888 panda',
               'eilers_names': ['888 panda'],
               'feature_names': ['888 panda'],
               'status': 'matched_all_three'},
 '99 riches': {'canonical_game_name': '99 riches',
               'eilers_names': ['99 riches'],
               'feature_names': ['99 riches'],
               'status': 'matched_all_three'},
 'arctic fox': {'canonical_game_name': 'arctic fox',
                'eilers_names': ['arctic fox'],
                'feature_names': [],
                'status': 'matched_eilers_only'},
 'aztec adventures': {'canonical_game_name': 'aztec adventures',
                      'eilers_names': ['aztec adventures'],
                      'feature_names': ['aztec adventures'],
                      'status': 'matched_all_three'},
 'aztec temple treasures': {'canonical_game_name': 'treasures aztec temple',
                            'eilers_names': ['treasures - aztec temple'],
                            'feature_names': ['treasures - aztec temple'],
                            'status': 'matched_all_three'},
 'babooshka': {'canonical_game_name': 'babooshka',
               'eilers_names': ['babooshka'],
               'feature_names': ['babooshka'],
               'status': 'matched_all_three'},
 'bao zang bao zha': {'canonical_game_name': 'bao zang bao zha',
                      'eilers_names': ['bao zang bao zha'],
                      'feature_names': ['bao zang bao zha'],
                      'status': 'matched_all_three'},
 'beauty of egypt jackpots': {'canonical_game_name': 'beauty of egypt jackpots',
                              'eilers_names': ['beauty of egypt jackpots'],
                              'feature_names': ['beauty of egypt jackpots'],
                              'status': 'matched_all_three'},
 'benny big game': {'canonical_game_name': 'benny big game',
                    'eilers_names': ['benny big game'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 'big bucks bonus': {'canonical_game_name': None,
                     'eilers_names': [],
                     'feature_names': [],
                     'status': 'review_required'},
 'big juan dual hhrag': {'canonical_game_name': None,
                         'eilers_names': [],
                         'feature_names': [],
                         'status': 'review_required'},
 'big ocean jackpots': {'canonical_game_name': 'big ocean jackpots',
                        'eilers_names': ['big ocean jackpots'],
                        'feature_names': ['big ocean jackpots'],
                        'status': 'matched_all_three'},
 'big tiger': {'canonical_game_name': 'big tiger',
               'eilers_names': ['big tiger'],
               'feature_names': ['big tiger'],
               'status': 'matched_all_three'},
 'black widow': {'canonical_game_name': 'black widow',
                 'eilers_names': ['black widow'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'black widow grand': {'canonical_game_name': 'black widow grand',
                       'eilers_names': ['black widow grand'],
                       'feature_names': ['black widow grand'],
                       'status': 'matched_all_three'},
 'blue blazes': {'canonical_game_name': 'blue blazes',
                 'eilers_names': ['blue blazes'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'boingy beans bean to be wild': {'canonical_game_name': 'boingy beans bean to be wild',
                                  'eilers_names': ['boingy beans bean to be wild'],
                                  'feature_names': ['boingy beans bean to be wild'],
                                  'status': 'matched_all_three'},
 'bonus wheel triple diamond': {'canonical_game_name': 'bonus wheel triple diamond',
                                'eilers_names': ['bonus wheel triple diamond'],
                                'feature_names': ['bonus wheel triple diamond'],
                                'status': 'matched_all_three'},
 'bring the boom egypt': {'canonical_game_name': 'bring the boom egypt',
                          'eilers_names': ['bring the boom - egypt'],
                          'feature_names': ['bring the boom - egypt'],
                          'status': 'matched_all_three'},
 'bring the boom pirate bay': {'canonical_game_name': 'bring the boom pirate bay',
                               'eilers_names': ['bring the boom - pirate bay'],
                               'feature_names': ['bring the boom - pirate bay'],
                               'status': 'matched_all_three'},
 'buffalo burst': {'canonical_game_name': 'buffalo burst',
                   'eilers_names': ['buffalo burst'],
                   'feature_names': ['buffalo burst'],
                   'status': 'matched_all_three'},
 'buffalo storm': {'canonical_game_name': 'buffalo storm',
                   'eilers_names': ['buffalo storm'],
                   'feature_names': ['buffalo storm'],
                   'status': 'matched_all_three'},
 'buffet mania': {'canonical_game_name': 'buffet mania',
                  'eilers_names': ['buffet mania'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'cable car cash multiway': {'canonical_game_name': None,
                             'eilers_names': [],
                             'feature_names': [],
                             'status': 'review_required'},
 'caifu zhi wu': {'canonical_game_name': 'caifu zhi wu',
                  'eilers_names': ['caifu zhi wu'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'cat peak': {'canonical_game_name': 'cat peak',
              'eilers_names': ['cat peak'],
              'feature_names': ['cat peak'],
              'status': 'matched_all_three'},
 'cats': {'canonical_game_name': 'cats',
          'eilers_names': ['cats'],
          'feature_names': [],
          'status': 'matched_eilers_only'},
 'cats n dogs': {'canonical_game_name': 'cats n dogs',
                 'eilers_names': ['cats n dogs'],
                 'feature_names': ['cats n dogs'],
                 'status': 'matched_all_three'},
 'cats wild serengeti prog': {'canonical_game_name': None,
                              'eilers_names': [],
                              'feature_names': [],
                              'status': 'review_required'},
 'chinese fortune': {'canonical_game_name': 'chinese fortune',
                     'eilers_names': ['chinese fortune'],
                     'feature_names': ['chinese fortune'],
                     'status': 'matched_all_three'},
 'chinese palace treasures': {'canonical_game_name': 'treasures chinese palace',
                              'eilers_names': ['treasures - chinese palace'],
                              'feature_names': ['treasures - chinese palace'],
                              'status': 'matched_all_three'},
 'chinese treasure': {'canonical_game_name': 'chinese treasure',
                      'eilers_names': ['chinese treasure'],
                      'feature_names': [],
                      'status': 'matched_eilers_only'},
 'classic hits money storm': {'canonical_game_name': 'money storm classic hits',
                              'eilers_names': ['money storm classic hits'],
                              'feature_names': ['money storm classic hits'],
                              'status': 'matched_all_three'},
 'cleopatra multiplay': {'canonical_game_name': 'cleopatra multiplay',
                         'eilers_names': ['cleopatra multiplay'],
                         'feature_names': [],
                         'status': 'matched_eilers_only'},
 'cleopatra triple fortune': {'canonical_game_name': None,
                              'eilers_names': [],
                              'feature_names': [],
                              'status': 'review_required'},
 'connect collect fire stampede': {'canonical_game_name': 'connect and collect fire stampede',
                                   'eilers_names': ['connect & collect - fire stampede'],
                                   'feature_names': ['connect & collect - fire stampede'],
                                   'status': 'matched_all_three'},
 'coyote moon desert nights': {'canonical_game_name': 'coyote moon desert nights',
                               'eilers_names': ['coyote moon - desert nights'],
                               'feature_names': ['coyote moon - desert nights'],
                               'status': 'matched_all_three'},
 'da vinci diamonds masterpiece': {'canonical_game_name': 'da vinci diamonds masterpiece',
                                   'eilers_names': ['da vinci diamonds masterpiece'],
                                   'feature_names': ['da vinci diamonds masterpiece'],
                                   'status': 'matched_all_three'},
 'diamond diamond': {'canonical_game_name': None,
                     'eilers_names': [],
                     'feature_names': [],
                     'status': 'review_required'},
 'diamond fire': {'canonical_game_name': 'diamond fire',
                  'eilers_names': ['diamond fire'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'diamond jackpots triple double stars': {'canonical_game_name': None,
                                          'eilers_names': [],
                                          'feature_names': [],
                                          'status': 'review_required'},
 'diamond queen': {'canonical_game_name': 'diamond queen',
                   'eilers_names': ['diamond queen'],
                   'feature_names': [],
                   'status': 'matched_eilers_only'},
 'dianmu surging skies': {'canonical_game_name': 'dianmu surging skies',
                          'eilers_names': ['dianmu surging skies'],
                          'feature_names': ['dianmu surging skies'],
                          'status': 'matched_all_three'},
 'divers dream pearl coast': {'canonical_game_name': 'divers dream pearl coast',
                              'eilers_names': ['divers dream pearl coast'],
                              'feature_names': ['divers dream pearl coast'],
                              'status': 'matched_all_three'},
 'double diamond': {'canonical_game_name': 'double diamond',
                    'eilers_names': ['double diamond'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 'double gold': {'canonical_game_name': 'double gold',
                 'eilers_names': ['double gold'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'double red hot 7s': {'canonical_game_name': 'double red hot 7s',
                       'eilers_names': ['double red hot 7s'],
                       'feature_names': [],
                       'status': 'matched_eilers_only'},
 'double red white blue reel': {'canonical_game_name': None,
                                'eilers_names': [],
                                'feature_names': [],
                                'status': 'review_required'},
 'double rollin in the cash': {'canonical_game_name': 'double rollin in the cash',
                               'eilers_names': ['double rollin in the cash'],
                               'feature_names': ['double rollin in the cash'],
                               'status': 'matched_all_three'},
 'double silver fire 777': {'canonical_game_name': 'double silver fire 777',
                            'eilers_names': ['double silver fire 777'],
                            'feature_names': ['double silver fire 777'],
                            'status': 'matched_all_three'},
 'dragon bowl': {'canonical_game_name': 'dragon bowl',
                 'eilers_names': ['dragon bowl'],
                 'feature_names': ['dragon bowl'],
                 'status': 'matched_all_three'},
 'dragon lights mystical falls': {'canonical_game_name': 'dragon lights mystical falls',
                                  'eilers_names': ['dragon lights mystical falls'],
                                  'feature_names': [],
                                  'status': 'matched_eilers_only'},
 'dragon lights secret fortress': {'canonical_game_name': 'dragon lights secret fortress',
                                   'eilers_names': ['dragon lights - secret fortress',
                                                    'dragon lights secret fortress'],
                                   'feature_names': [],
                                   'status': 'matched_eilers_only'},
 'dragon of fortune evolution': {'canonical_game_name': 'dragon of fortune evolution',
                                 'eilers_names': ['dragon of fortune evolution'],
                                 'feature_names': ['dragon of fortune evolution'],
                                 'status': 'matched_all_three'},
 'dragon of fortune multiway': {'canonical_game_name': None,
                                'eilers_names': [],
                                'feature_names': [],
                                'status': 'review_required'},
 'dragon spell': {'canonical_game_name': 'dragon spell',
                  'eilers_names': ['dragon spell'],
                  'feature_names': ['dragon spell'],
                  'status': 'matched_all_three'},
 'dragons vs pandas': {'canonical_game_name': 'dragons vs pandas',
                       'eilers_names': ['dragons vs pandas', 'dragons vs. pandas'],
                       'feature_names': ['dragons vs pandas'],
                       'status': 'matched_all_three'},
 'dreamy genie': {'canonical_game_name': 'dreamy genie',
                  'eilers_names': ['dreamy genie'],
                  'feature_names': ['dreamy genie'],
                  'status': 'matched_all_three'},
 'dynamite blast passion flowers': {'canonical_game_name': None,
                                    'eilers_names': [],
                                    'feature_names': [],
                                    'status': 'review_required'},
 'egyptian dreams': {'canonical_game_name': 'egyptian dreams',
                     'eilers_names': ['egyptian dreams'],
                     'feature_names': ['egyptian dreams'],
                     'status': 'matched_all_three'},
 'egyptian link amasis treasures buy a bonus': {'canonical_game_name': None,
                                                'eilers_names': [],
                                                'feature_names': [],
                                                'status': 'review_required'},
 'egyptian link nefturis treasures': {'canonical_game_name': 'egyptian link nefturis treasure',
                                      'eilers_names': ["egyptian link - nefturi's treasures",
                                                       'egyptian link nefturis treasure'],
                                      'feature_names': ['egyptian link nefturis treasure'],
                                      'status': 'matched_all_three'},
 'egyptian stars': {'canonical_game_name': 'egyptian stars',
                    'eilers_names': ['egyptian stars'],
                    'feature_names': ['egyptian stars'],
                    'status': 'matched_all_three'},
 'elephants vs tigers': {'canonical_game_name': 'elephants vs tigers',
                         'eilers_names': ['elephants vs tigers', 'elephants vs. tigers'],
                         'feature_names': ['elephants vs tigers'],
                         'status': 'matched_all_three'},
 'emperors throne': {'canonical_game_name': None,
                     'eilers_names': [],
                     'feature_names': [],
                     'status': 'review_required'},
 'empresses vs emperors': {'canonical_game_name': 'empresses vs emperors',
                           'eilers_names': ['empresses vs emperors'],
                           'feature_names': ['empresses vs emperors'],
                           'status': 'matched_all_three'},
 'enchanted unicorn returns': {'canonical_game_name': 'enchanted unicorn returns',
                               'eilers_names': ['enchanted unicorn returns'],
                               'feature_names': ['enchanted unicorn returns'],
                               'status': 'matched_all_three'},
 'eternal link princess empire': {'canonical_game_name': 'eternal link princess empire',
                                  'eilers_names': ['eternal link - princess empire'],
                                  'feature_names': ['eternal link - princess empire'],
                                  'status': 'matched_all_three'},
 'eternal link warriors empire': {'canonical_game_name': 'eternal link warriors empire',
                                  'eilers_names': ['eternal link - warriors empire'],
                                  'feature_names': ['eternal link - warriors empire'],
                                  'status': 'matched_all_three'},
 'everlasting turtle': {'canonical_game_name': 'everlasting turtle',
                        'eilers_names': ['everlasting turtle'],
                        'feature_names': ['everlasting turtle'],
                        'status': 'matched_all_three'},
 'fa cai long': {'canonical_game_name': 'fa cai long',
                 'eilers_names': ['fa cai long'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'fame and fortune': {'canonical_game_name': 'fame and fortune',
                      'eilers_names': ['fame and fortune'],
                      'feature_names': [],
                      'status': 'matched_eilers_only'},
 'fire it up': {'canonical_game_name': None, 'eilers_names': [], 'feature_names': [], 'status': 'review_required'},
 'firebird spirit': {'canonical_game_name': None,
                     'eilers_names': [],
                     'feature_names': [],
                     'status': 'review_required'},
 'fortune coin': {'canonical_game_name': None,
                  'eilers_names': [],
                  'feature_names': [],
                  'status': 'review_required'},
 'fortune coin extra': {'canonical_game_name': 'fortune coin extra',
                        'eilers_names': ['fortune coin extra'],
                        'feature_names': ['fortune coin extra'],
                        'status': 'matched_all_three'},
 'fortune link 4 1': {'canonical_game_name': None,
                      'eilers_names': [],
                      'feature_names': [],
                      'status': 'review_required'},
 'fortune owl': {'canonical_game_name': 'fortune owl',
                 'eilers_names': ['fortune owl'],
                 'feature_names': ['fortune owl'],
                 'status': 'matched_all_three'},
 'fortune rockets': {'canonical_game_name': 'fortune rockets',
                     'eilers_names': ['fortune rockets'],
                     'feature_names': ['fortune rockets'],
                     'status': 'matched_all_three'},
 'fu gui hao men glorious fortunes': {'canonical_game_name': 'fu gui hao men glorious fortunes',
                                      'eilers_names': ['fu gui hao men glorious fortunes'],
                                      'feature_names': ['fu gui hao men glorious fortunes'],
                                      'status': 'matched_all_three'},
 'fu gui hao men ii everlasting fortunes': {'canonical_game_name': 'fu gui hao men 2 everlasting fortunes',
                                            'eilers_names': ['fu gui hao men 2 - everlasting fortunes'],
                                            'feature_names': ['fu gui hao men 2 - everlasting fortunes'],
                                            'status': 'matched_all_three'},
 'fu gui hao men ii magnificent fortunes': {'canonical_game_name': 'fu gui hao men 2 magnificent fortunes',
                                            'eilers_names': ['fu gui hao men 2 - magnificent fortunes'],
                                            'feature_names': [],
                                            'status': 'matched_eilers_only'},
 'fu gui hao men splendid fortunes': {'canonical_game_name': 'fu gui hao men splendid fortunes',
                                      'eilers_names': ['fu gui hao men splendid fortunes'],
                                      'feature_names': ['fu gui hao men splendid fortunes'],
                                      'status': 'matched_all_three'},
 'fu gui ji li animal treasures': {'canonical_game_name': 'fu gui ji li animal treasure',
                                   'eilers_names': ['fu gui ji li - animal treasure'],
                                   'feature_names': [],
                                   'status': 'matched_eilers_only'},
 'fu tao peach of riches': {'canonical_game_name': 'fu tao peach of riches',
                            'eilers_names': ['fu tao peach of riches'],
                            'feature_names': ['fu tao peach of riches'],
                            'status': 'matched_all_three'},
 'genie wishes': {'canonical_game_name': 'genie wishes',
                  'eilers_names': ['genie wishes'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'goddess of gold': {'canonical_game_name': 'goddess of gold',
                     'eilers_names': ['goddess of gold'],
                     'feature_names': [],
                     'status': 'matched_eilers_only'},
 'gold bar 7s': {'canonical_game_name': 'gold bar 7s',
                 'eilers_names': ['gold bar 7s'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'golden 12': {'canonical_game_name': 'golden 12',
               'eilers_names': ['golden 12'],
               'feature_names': ['golden 12'],
               'status': 'matched_all_three'},
 'golden eagle': {'canonical_game_name': 'golden eagle',
                  'eilers_names': ['golden eagle'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'golden egypt': {'canonical_game_name': 'golden egypt',
                  'eilers_names': ['golden egypt'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'golden egypt grand': {'canonical_game_name': 'golden egypt grand',
                        'eilers_names': ['golden egypt grand'],
                        'feature_names': ['golden egypt grand'],
                        'status': 'matched_all_three'},
 'golden gate': {'canonical_game_name': 'golden gate',
                 'eilers_names': ['golden gate'],
                 'feature_names': ['golden gate'],
                 'status': 'matched_all_three'},
 'golden goddess': {'canonical_game_name': 'golden goddess',
                    'eilers_names': ['golden goddess', 'golden goddess .'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 'golden knight': {'canonical_game_name': 'golden knight',
                   'eilers_names': ['golden knight'],
                   'feature_names': ['golden knight'],
                   'status': 'matched_all_three'},
 'golden phoenix': {'canonical_game_name': 'golden phoenix',
                    'eilers_names': ['golden phoenix'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 'golden stag win zone': {'canonical_game_name': 'golden stag win zone',
                          'eilers_names': ['golden stag win zone'],
                          'feature_names': ['golden stag win zone'],
                          'status': 'matched_all_three'},
 'goldify': {'canonical_game_name': 'goldify',
             'eilers_names': ['goldify'],
             'feature_names': [],
             'status': 'matched_eilers_only'},
 'grand buddha': {'canonical_game_name': None,
                  'eilers_names': [],
                  'feature_names': [],
                  'status': 'review_required'},
 'greek lightning': {'canonical_game_name': 'greek lightning',
                     'eilers_names': ['greek lightning'],
                     'feature_names': ['greek lightning'],
                     'status': 'matched_all_three'},
 'griffins throne grand': {'canonical_game_name': 'griffins throne grand',
                           'eilers_names': ['griffins throne grand'],
                           'feature_names': ['griffins throne grand'],
                           'status': 'matched_all_three'},
 'guardians of egypt': {'canonical_game_name': 'guardians of egypt',
                        'eilers_names': ['guardians of egypt'],
                        'feature_names': ['guardians of egypt'],
                        'status': 'matched_all_three'},
 'haywire deluxe': {'canonical_game_name': 'haywire deluxe',
                    'eilers_names': ['haywire deluxe'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 'haywire reel': {'canonical_game_name': None,
                  'eilers_names': [],
                  'feature_names': [],
                  'status': 'review_required'},
 'hexbreaker': {'canonical_game_name': 'hexbreaker',
                'eilers_names': ['hexbreaker'],
                'feature_names': [],
                'status': 'matched_eilers_only'},
 'hexbreaker 3': {'canonical_game_name': 'hexbreaker 3',
                  'eilers_names': ['hexbreaker 3'],
                  'feature_names': ['hexbreaker 3'],
                  'status': 'matched_all_three'},
 'hidden kingdom': {'canonical_game_name': 'hidden kingdom',
                    'eilers_names': ['hidden kingdom'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 'honey bucks': {'canonical_game_name': 'honey bucks',
                 'eilers_names': ['honey bucks'],
                 'feature_names': ['honey bucks'],
                 'status': 'matched_all_three'},
 'hot hit rainbow riches lucky 7s': {'canonical_game_name': None,
                                     'eilers_names': [],
                                     'feature_names': [],
                                     'status': 'review_required'},
 'hot link blue moon ii express': {'canonical_game_name': 'hot link blue moon 2 express',
                                   'eilers_names': ['hot link blue moon ii express'],
                                   'feature_names': [],
                                   'status': 'matched_eilers_only'},
 'hot link money beans express': {'canonical_game_name': 'hot link money beans express',
                                  'eilers_names': ['hot link money beans express'],
                                  'feature_names': [],
                                  'status': 'matched_eilers_only'},
 'hot roll fairy ring': {'canonical_game_name': 'hot roll fairy ring',
                         'eilers_names': ['hot roll fairy ring'],
                         'feature_names': ['hot roll fairy ring'],
                         'status': 'matched_all_three'},
 'hyper hits flower empress': {'canonical_game_name': 'hyper hits flower empress',
                               'eilers_names': ['hyper hits flower empress'],
                               'feature_names': ['hyper hits flower empress'],
                               'status': 'matched_all_three'},
 'hyper hits longship legend': {'canonical_game_name': 'hyper hits longship legend',
                                'eilers_names': ['hyper hits longship legend'],
                                'feature_names': ['hyper hits longship legend'],
                                'status': 'matched_all_three'},
 'hyper hits the golden bow': {'canonical_game_name': 'hyper hits the golden bow',
                               'eilers_names': ['hyper hits the golden bow'],
                               'feature_names': ['hyper hits the golden bow'],
                               'status': 'matched_all_three'},
 'island eyes': {'canonical_game_name': 'island eyes',
                 'eilers_names': ['island eyes'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'its da bomb gophersbgone 3r1l3': {'canonical_game_name': None,
                                    'eilers_names': [],
                                    'feature_names': [],
                                    'status': 'review_required'},
 'jackpot 7s free games 20l100c': {'canonical_game_name': None,
                                   'eilers_names': [],
                                   'feature_names': [],
                                   'status': 'review_required'},
 'jewel of arabia': {'canonical_game_name': 'jewel of arabia',
                     'eilers_names': ['jewel of arabia'],
                     'feature_names': [],
                     'status': 'matched_eilers_only'},
 'jin hu': {'canonical_game_name': 'jin hu',
            'eilers_names': ['jin hu'],
            'feature_names': [],
            'status': 'matched_eilers_only'},
 'jin tu fa cai': {'canonical_game_name': 'jin tu fa cai',
                   'eilers_names': ['jin tu fa cai'],
                   'feature_names': ['jin tu fa cai'],
                   'status': 'matched_all_three'},
 'jin zhu dragon': {'canonical_game_name': 'jin zhu dragon',
                    'eilers_names': ['jin zhu - dragon'],
                    'feature_names': ['jin zhu - dragon'],
                    'status': 'matched_all_three'},
 'jin zhu rooster': {'canonical_game_name': 'jin zhu rooster',
                     'eilers_names': ['jin zhu - rooster'],
                     'feature_names': ['jin zhu - rooster'],
                     'status': 'matched_all_three'},
 'king khufu dawn': {'canonical_game_name': 'king khufu dawn',
                     'eilers_names': ['king khufu - dawn'],
                     'feature_names': ['king khufu - dawn'],
                     'status': 'matched_all_three'},
 'king of huaguo': {'canonical_game_name': 'king of huaguo',
                    'eilers_names': ['king of huaguo'],
                    'feature_names': ['king of huaguo'],
                    'status': 'matched_all_three'},
 'king of olympus': {'canonical_game_name': 'king of olympus',
                     'eilers_names': ['king of olympus'],
                     'feature_names': [],
                     'status': 'matched_eilers_only'},
 'kitty glitter grand': {'canonical_game_name': 'kitty glitter grand',
                         'eilers_names': ['kitty glitter grand'],
                         'feature_names': ['kitty glitter grand'],
                         'status': 'matched_all_three'},
 'lion dance': {'canonical_game_name': 'lion dance',
                'eilers_names': ['lion dance'],
                'feature_names': [],
                'status': 'matched_eilers_only'},
 'lotus flower': {'canonical_game_name': 'lotus flower',
                  'eilers_names': ['lotus flower'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'luck rising brysons bounty': {'canonical_game_name': 'luck rising brysons bounty',
                                'eilers_names': ['luck rising - brysons bounty'],
                                'feature_names': [],
                                'status': 'matched_eilers_only'},
 'lucky buddha': {'canonical_game_name': 'lucky buddha',
                  'eilers_names': ['lucky buddha'],
                  'feature_names': ['lucky buddha'],
                  'status': 'matched_all_three'},
 'lucky coin asian dreaming': {'canonical_game_name': 'lucky coin asian dreaming',
                               'eilers_names': ['lucky coin asian dreaming'],
                               'feature_names': [],
                               'status': 'matched_eilers_only'},
 'lucky coin atlantica': {'canonical_game_name': 'lucky coin atlantica',
                          'eilers_names': ['lucky coin atlantica'],
                          'feature_names': [],
                          'status': 'matched_eilers_only'},
 'lucky larrys lobstermania 4': {'canonical_game_name': 'lucky larrys lobstermania 4',
                                 'eilers_names': ["lucky larry's lobstermania 4", 'lucky larrys lobstermania 4'],
                                 'feature_names': [],
                                 'status': 'matched_eilers_only'},
 'lucky larrys lobstermania 5': {'canonical_game_name': 'lucky larrys lobstermania 5',
                                 'eilers_names': ['lucky larrys lobstermania 5'],
                                 'feature_names': ['lucky larrys lobstermania 5'],
                                 'status': 'matched_all_three'},
 'lucky larrys lobstermania ultimate treasure': {'canonical_game_name': 'lucky larrys lobstermania ultimate '
                                                                        'treasures',
                                                 'eilers_names': ['lucky larrys lobstermania - ultimate '
                                                                  'treasures'],
                                                 'feature_names': [],
                                                 'status': 'matched_eilers_only'},
 'lucky lion fish': {'canonical_game_name': 'lucky lion fish',
                     'eilers_names': ['lucky lion fish'],
                     'feature_names': [],
                     'status': 'matched_eilers_only'},
 'lucky lock': {'canonical_game_name': 'lucky lock',
                'eilers_names': ['lucky lock'],
                'feature_names': [],
                'status': 'matched_eilers_only'},
 'lucky ox': {'canonical_game_name': 'lucky ox',
              'eilers_names': ['lucky ox'],
              'feature_names': ['lucky ox'],
              'status': 'matched_all_three'},
 'lucky triple goldsmokeys den': {'canonical_game_name': 'lucky triple gold smokeys den',
                                  'eilers_names': ['lucky triple gold - smokeys den'],
                                  'feature_names': ['lucky triple gold - smokeys den'],
                                  'status': 'matched_all_three'},
 'lucky wealth cat': {'canonical_game_name': 'lucky wealth cat',
                      'eilers_names': ['lucky wealth cat'],
                      'feature_names': ['lucky wealth cat'],
                      'status': 'matched_all_three'},
 'lunar queen': {'canonical_game_name': 'lunar queen',
                 'eilers_names': ['lunar queen'],
                 'feature_names': ['lunar queen'],
                 'status': 'matched_all_three'},
 'magic link celestial queen': {'canonical_game_name': 'magic link celestial queen',
                                'eilers_names': ['magic link celestial queen'],
                                'feature_names': [],
                                'status': 'matched_eilers_only'},
 'magic link stellar riches': {'canonical_game_name': 'magic link stellar riches',
                               'eilers_names': ['magic link stellar riches'],
                               'feature_names': [],
                               'status': 'matched_eilers_only'},
 'magic rockets dragon': {'canonical_game_name': 'magic rockets dragon',
                          'eilers_names': ['magic rockets - dragon'],
                          'feature_names': ['magic rockets - dragon'],
                          'status': 'matched_all_three'},
 'magic rockets tiger': {'canonical_game_name': 'magic rockets tiger',
                         'eilers_names': ['magic rockets - tiger'],
                         'feature_names': ['magic rockets - tiger'],
                         'status': 'matched_all_three'},
 'magic treasures dragon': {'canonical_game_name': 'magic treasures dragon',
                            'eilers_names': ['magic treasures dragon'],
                            'feature_names': ['magic treasures dragon'],
                            'status': 'matched_all_three'},
 'magic treasures gold emperor': {'canonical_game_name': 'magic treasures gold emperor',
                                  'eilers_names': ['magic treasures gold - emperor'],
                                  'feature_names': [],
                                  'status': 'matched_eilers_only'},
 'magic treasures gold empress': {'canonical_game_name': 'magic treasures gold empress',
                                  'eilers_names': ['magic treasures gold - empress'],
                                  'feature_names': [],
                                  'status': 'matched_eilers_only'},
 'magic treasures lock respin lion': {'canonical_game_name': 'magic treasures lock and respin lion',
                                      'eilers_names': ['magic treasures lock & respin - lion'],
                                      'feature_names': ['magic treasures lock & respin - lion'],
                                      'status': 'matched_all_three'},
 'magic wishes': {'canonical_game_name': 'magic wishes',
                  'eilers_names': ['magic wishes'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'magnificent phoenix': {'canonical_game_name': 'magnificent phoenix',
                         'eilers_names': ['magnificent phoenix'],
                         'feature_names': ['magnificent phoenix'],
                         'status': 'matched_all_three'},
 'maharajas money': {'canonical_game_name': 'maharajas money',
                     'eilers_names': ['maharajas money'],
                     'feature_names': [],
                     'status': 'matched_eilers_only'},
 'majestic stallion 80 ctc': {'canonical_game_name': None,
                              'eilers_names': [],
                              'feature_names': [],
                              'status': 'review_required'},
 'mayan riches ixchel': {'canonical_game_name': 'mayan riches ixchel',
                         'eilers_names': ['mayan riches ixchel'],
                         'feature_names': ['mayan riches ixchel'],
                         'status': 'matched_all_three'},
 'mayan riches kukulkan': {'canonical_game_name': None,
                           'eilers_names': [],
                           'feature_names': [],
                           'status': 'review_required'},
 'midnight treasures': {'canonical_game_name': 'midnight treasure',
                        'eilers_names': ['midnight treasure'],
                        'feature_names': [],
                        'status': 'matched_eilers_only'},
 'mighty miner': {'canonical_game_name': 'mighty miner',
                  'eilers_names': ['mighty miner'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'miners dream blue mountain': {'canonical_game_name': 'miners dream blue mountain',
                                'eilers_names': ['miners dream - blue mountain'],
                                'feature_names': ['miners dream - blue mountain'],
                                'status': 'matched_all_three'},
 'mistress of egypt grand': {'canonical_game_name': 'mistress of egypt grand',
                             'eilers_names': ['mistress of egypt grand'],
                             'feature_names': [],
                             'status': 'matched_eilers_only'},
 'money mover panther pride': {'canonical_game_name': 'money mover panther pride',
                               'eilers_names': ['money mover panther pride'],
                               'feature_names': ['money mover panther pride'],
                               'status': 'matched_all_three'},
 'money mover tiger pride': {'canonical_game_name': 'money mover tiger pride',
                             'eilers_names': ['money mover tiger pride'],
                             'feature_names': ['money mover tiger pride'],
                             'status': 'matched_all_three'},
 'moon spirit': {'canonical_game_name': 'moon spirit',
                 'eilers_names': ['moon spirit'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'mummy link dawn': {'canonical_game_name': 'mummy link dawn',
                     'eilers_names': ['mummy link - dawn'],
                     'feature_names': ['mummy link - dawn'],
                     'status': 'matched_all_three'},
 'mummy link night': {'canonical_game_name': 'mummy link night',
                      'eilers_names': ['mummy link - night'],
                      'feature_names': ['mummy link - night'],
                      'status': 'matched_all_three'},
 'munster match': {'canonical_game_name': None,
                   'eilers_names': [],
                   'feature_names': [],
                   'status': 'review_required'},
 'mystery of the lamp enchanted palace': {'canonical_game_name': 'mystery of the lamp enchanted palace',
                                          'eilers_names': ['mystery of the lamp enchanted palace'],
                                          'feature_names': [],
                                          'status': 'matched_eilers_only'},
 'mystery of the lamp mystical gardens': {'canonical_game_name': 'mystery of the lamp mystical gardens',
                                          'eilers_names': ['mystery of the lamp mystical gardens'],
                                          'feature_names': [],
                                          'status': 'matched_eilers_only'},
 'mystical treasures lion': {'canonical_game_name': 'mystical treasures lion',
                             'eilers_names': ['mystical treasures lion'],
                             'feature_names': ['mystical treasures lion'],
                             'status': 'matched_all_three'},
 'ocean magic': {'canonical_game_name': 'ocean magic',
                 'eilers_names': ['ocean magic'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'ocean magic bubble boost': {'canonical_game_name': 'ocean magic bubble boost',
                              'eilers_names': ['ocean magic bubble boost'],
                              'feature_names': ['ocean magic bubble boost'],
                              'status': 'matched_all_three'},
 'ocean spirit': {'canonical_game_name': 'ocean spirit',
                  'eilers_names': ['ocean spirit'],
                  'feature_names': ['ocean spirit'],
                  'status': 'matched_all_three'},
 'outgrow spooky moon': {'canonical_game_name': 'outgrow spooky moon',
                         'eilers_names': ['outgrow - spooky moon'],
                         'feature_names': ['outgrow - spooky moon'],
                         'status': 'matched_all_three'},
 'outgrow tropical moon': {'canonical_game_name': 'outgrow tropical moon',
                           'eilers_names': ['outgrow - tropical moon'],
                           'feature_names': ['outgrow - tropical moon'],
                           'status': 'matched_all_three'},
 'overloot': {'canonical_game_name': 'overloot',
              'eilers_names': ['overloot'],
              'feature_names': [],
              'status': 'matched_eilers_only'},
 'pharaoh 7s': {'canonical_game_name': None, 'eilers_names': [], 'feature_names': [], 'status': 'review_required'},
 'phoenix bowl': {'canonical_game_name': 'phoenix bowl',
                  'eilers_names': ['phoenix bowl'],
                  'feature_names': ['phoenix bowl'],
                  'status': 'matched_all_three'},
 'phone tag': {'canonical_game_name': 'phone tag',
               'eilers_names': ['phone tag'],
               'feature_names': [],
               'status': 'matched_eilers_only'},
 'pig and pug': {'canonical_game_name': 'pig and pug',
                 'eilers_names': ['pig and pug'],
                 'feature_names': ['pig and pug'],
                 'status': 'matched_all_three'},
 'pink diamond': {'canonical_game_name': 'pink diamond',
                  'eilers_names': ['pink diamond'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'pirate link drakes treasure': {'canonical_game_name': 'pirate link drakes treasure',
                                 'eilers_names': ["pirate link - drake's treasure", 'pirate link drakes treasure'],
                                 'feature_names': ['pirate link drakes treasure'],
                                 'status': 'matched_all_three'},
 'pirate link marys treasure': {'canonical_game_name': 'pirate link marys treasure',
                                'eilers_names': ["pirate link - mary's treasure", 'pirate link marys treasure'],
                                'feature_names': ['pirate link marys treasure'],
                                'status': 'matched_all_three'},
 'progressive bonanza 3x2x legend of the peacock': {'canonical_game_name': None,
                                                    'eilers_names': [],
                                                    'feature_names': [],
                                                    'status': 'review_required'},
 'prosperity link blessings skyrise': {'canonical_game_name': None,
                                       'eilers_names': [],
                                       'feature_names': [],
                                       'status': 'review_required'},
 'prosperity link cai yun heng tong': {'canonical_game_name': 'prosperity link cai yun heng tong',
                                       'eilers_names': ['prosperity link cai yun heng tong'],
                                       'feature_names': [],
                                       'status': 'matched_eilers_only'},
 'prosperity link wan shi ru yi': {'canonical_game_name': 'prosperity link wan shi ru yi',
                                   'eilers_names': ['prosperity link wan shi ru yi'],
                                   'feature_names': [],
                                   'status': 'matched_eilers_only'},
 'prosperity pearl': {'canonical_game_name': 'prosperity pearl',
                      'eilers_names': ['prosperity pearl'],
                      'feature_names': [],
                      'status': 'matched_eilers_only'},
 'queen of olympus': {'canonical_game_name': 'queen of olympus',
                      'eilers_names': ['queen of olympus'],
                      'feature_names': ['queen of olympus'],
                      'status': 'matched_all_three'},
 'queenie': {'canonical_game_name': 'queenie',
             'eilers_names': ['queenie'],
             'feature_names': [],
             'status': 'matched_eilers_only'},
 'raise the sails': {'canonical_game_name': 'raise the sails',
                     'eilers_names': ['raise the sails'],
                     'feature_names': ['raise the sails'],
                     'status': 'matched_all_three'},
 'red hot jackpots double diamond 2': {'canonical_game_name': None,
                                       'eilers_names': [],
                                       'feature_names': [],
                                       'status': 'review_required'},
 'red hottie': {'canonical_game_name': 'red hottie',
                'eilers_names': ['red hottie'],
                'feature_names': [],
                'status': 'matched_eilers_only'},
 'reeledge tullys treasure hunt': {'canonical_game_name': None,
                                   'eilers_names': [],
                                   'feature_names': [],
                                   'status': 'review_required'},
 'regal link lion': {'canonical_game_name': 'regal link lion',
                     'eilers_names': ['regal link lion'],
                     'feature_names': ['regal link lion'],
                     'status': 'matched_all_three'},
 'regal link raven': {'canonical_game_name': 'regal link raven',
                      'eilers_names': ['regal link raven'],
                      'feature_names': ['regal link raven'],
                      'status': 'matched_all_three'},
 'regal riches': {'canonical_game_name': 'regal riches',
                  'eilers_names': ['regal riches'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'rembrandt riches masterpiece': {'canonical_game_name': 'rembrandt riches masterpiece',
                                  'eilers_names': ['rembrandt riches masterpiece'],
                                  'feature_names': ['rembrandt riches masterpiece'],
                                  'status': 'matched_all_three'},
 'rising phoenix': {'canonical_game_name': 'rising phoenix',
                    'eilers_names': ['rising phoenix'],
                    'feature_names': ['rising phoenix'],
                    'status': 'matched_all_three'},
 'rising reels link mayan magic': {'canonical_game_name': 'rising reels link mayan magic',
                                   'eilers_names': ['rising reels link - mayan magic',
                                                    'rising reels link mayan magic'],
                                   'feature_names': [],
                                   'status': 'matched_eilers_only'},
 'rising rockets empress': {'canonical_game_name': 'rising rockets empress',
                            'eilers_names': ['rising rockets empress'],
                            'feature_names': ['rising rockets empress'],
                            'status': 'matched_all_three'},
 'rising spins': {'canonical_game_name': 'rising spins',
                  'eilers_names': ['rising spins'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'royals egypt': {'canonical_game_name': 'royals egypt',
                  'eilers_names': ['royals egypt'],
                  'feature_names': ['royals egypt'],
                  'status': 'matched_all_three'},
 'samurai 888 katsumi': {'canonical_game_name': 'samurai 888 katsumi',
                         'eilers_names': ['samurai 888 katsumi'],
                         'feature_names': ['samurai 888 katsumi'],
                         'status': 'matched_all_three'},
 'samurai 888 rise kimiko': {'canonical_game_name': 'samurai rise 888 kimiko',
                             'eilers_names': ['samurai rise 888 - kimiko'],
                             'feature_names': ['samurai rise 888 - kimiko'],
                             'status': 'matched_all_three'},
 'samurai 888 spin kenji': {'canonical_game_name': 'samurai 888 spin kenji',
                            'eilers_names': ['samurai 888 spin - kenji'],
                            'feature_names': ['samurai 888 spin - kenji'],
                            'status': 'matched_all_three'},
 'samurai 888 spin sakura': {'canonical_game_name': 'samurai 888 spin sakura',
                             'eilers_names': ['samurai 888 spin - sakura'],
                             'feature_names': ['samurai 888 spin - sakura'],
                             'status': 'matched_all_three'},
 'san xing riches': {'canonical_game_name': 'san xing riches',
                     'eilers_names': ['san xing riches'],
                     'feature_names': [],
                     'status': 'matched_eilers_only'},
 'scarab': {'canonical_game_name': 'scarab',
            'eilers_names': ['scarab'],
            'feature_names': ['scarab'],
            'status': 'matched_all_three'},
 'scarab grand': {'canonical_game_name': 'scarab grand',
                  'eilers_names': ['scarab grand'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'scarab link cleora': {'canonical_game_name': 'scarab link cleora',
                        'eilers_names': ['scarab link - cleora'],
                        'feature_names': ['scarab link - cleora'],
                        'status': 'matched_all_three'},
 'scarab link nefturi': {'canonical_game_name': 'scarab link nefturi',
                         'eilers_names': ['scarab link - nefturi'],
                         'feature_names': ['scarab link - nefturi'],
                         'status': 'matched_all_three'},
 'scarab multiplay': {'canonical_game_name': 'scarab multiplay',
                      'eilers_names': ['scarab multiplay'],
                      'feature_names': [],
                      'status': 'matched_eilers_only'},
 'secrets of the forest': {'canonical_game_name': 'secrets of the forest',
                           'eilers_names': ['secrets of the forest'],
                           'feature_names': [],
                           'status': 'matched_eilers_only'},
 'sevens': {'canonical_game_name': 'sevens',
            'eilers_names': ['sevens'],
            'feature_names': [],
            'status': 'matched_eilers_only'},
 'shambala': {'canonical_game_name': 'shambala',
              'eilers_names': ['shambala'],
              'feature_names': ['shambala'],
              'status': 'matched_all_three'},
 'shamrock stars': {'canonical_game_name': 'shamrock stars',
                    'eilers_names': ['shamrock stars'],
                    'feature_names': ['shamrock stars'],
                    'status': 'matched_all_three'},
 'siberian storm 2': {'canonical_game_name': 'siberian storm 2',
                      'eilers_names': ['siberian storm 2'],
                      'feature_names': ['siberian storm 2'],
                      'status': 'matched_all_three'},
 'sizzling 7s piggy parade': {'canonical_game_name': 'sizzling 7s piggy parade',
                              'eilers_names': ['sizzling 7s piggy parade'],
                              'feature_names': ['sizzling 7s piggy parade'],
                              'status': 'matched_all_three'},
 'solar disc grand': {'canonical_game_name': 'solar disc grand',
                      'eilers_names': ['solar disc grand'],
                      'feature_names': [],
                      'status': 'matched_eilers_only'},
 'sphinx': {'canonical_game_name': 'sphinx',
            'eilers_names': ['sphinx'],
            'feature_names': [],
            'status': 'matched_eilers_only'},
 'sphinx link cleora': {'canonical_game_name': 'sphinx link cleora',
                        'eilers_names': ['sphinx link cleora'],
                        'feature_names': ['sphinx link cleora'],
                        'status': 'matched_all_three'},
 'sphinx link nefturi': {'canonical_game_name': 'sphinx link nefturi',
                         'eilers_names': ['sphinx link nefturi'],
                         'feature_names': ['sphinx link nefturi'],
                         'status': 'matched_all_three'},
 'split n win linkfortune': {'canonical_game_name': 'split n win link fortunes',
                             'eilers_names': ['split n win link fortunes'],
                             'feature_names': [],
                             'status': 'matched_eilers_only'},
 'star goddess': {'canonical_game_name': 'star goddess',
                  'eilers_names': ['star goddess'],
                  'feature_names': ['star goddess'],
                  'status': 'matched_all_three'},
 'stinkin rich': {'canonical_game_name': 'stinkin rich',
                  'eilers_names': ['stinkin rich'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'stinkin richskunks gone wild': {'canonical_game_name': 'stinkin rich skunks gone wild',
                                  'eilers_names': ['stinkin rich skunks gone wild',
                                                   "stinkin' rich - skunks gone wild"],
                                  'feature_names': ['stinkin rich skunks gone wild'],
                                  'status': 'matched_all_three'},
 'super sallys shrimpmania 5': {'canonical_game_name': 'super sallys shrimpmania 5',
                                'eilers_names': ['super sallys shrimpmania 5'],
                                'feature_names': [],
                                'status': 'matched_eilers_only'},
 'super times pay 3510 play poker action': {'canonical_game_name': None,
                                            'eilers_names': [],
                                            'feature_names': [],
                                            'status': 'review_required'},
 'super wheelmania link legendary beauty': {'canonical_game_name': None,
                                            'eilers_names': [],
                                            'feature_names': [],
                                            'status': 'review_required'},
 'super wheelmania link treasure fish': {'canonical_game_name': 'super wheelmania link treasure fish',
                                         'eilers_names': ['super wheelmania link treasure fish'],
                                         'feature_names': [],
                                         'status': 'matched_eilers_only'},
 'super wheelmania link vip riches': {'canonical_game_name': 'super wheelmania link vip riches',
                                      'eilers_names': ['super wheelmania link vip riches'],
                                      'feature_names': [],
                                      'status': 'matched_eilers_only'},
 'super x lucky x 2x3x4x5x x pay': {'canonical_game_name': None,
                                    'eilers_names': [],
                                    'feature_names': [],
                                    'status': 'review_required'},
 'surefire jackpots': {'canonical_game_name': 'sure fire jackpots',
                       'eilers_names': ['sure fire jackpots'],
                       'feature_names': ['sure fire jackpots'],
                       'status': 'matched_all_three'},
 'temple of fire': {'canonical_game_name': 'temple of fire',
                    'eilers_names': ['temple of fire'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 'temujin treasures': {'canonical_game_name': 'temujin treasure',
                       'eilers_names': ['temujin treasure'],
                       'feature_names': [],
                       'status': 'matched_eilers_only'},
 'the amazing money machine gold': {'canonical_game_name': 'the amazing money machine gold',
                                    'eilers_names': ['the amazing money machine gold'],
                                    'feature_names': [],
                                    'status': 'matched_eilers_only'},
 'the amazing money machine sapphire': {'canonical_game_name': 'the amazing money machine sapphire',
                                        'eilers_names': ['the amazing money machine sapphire'],
                                        'feature_names': ['the amazing money machine sapphire'],
                                        'status': 'matched_all_three'},
 'tiger and dragon cash on reels': {'canonical_game_name': 'tiger and dragon cash on reels',
                                    'eilers_names': ['tiger & dragon cash on reels'],
                                    'feature_names': [],
                                    'status': 'matched_eilers_only'},
 'tiger and dragon link cash on reels': {'canonical_game_name': None,
                                         'eilers_names': [],
                                         'feature_names': [],
                                         'status': 'review_required'},
 'treasure blast balloon': {'canonical_game_name': 'treasure blast balloon',
                            'eilers_names': ['treasure blast - balloon'],
                            'feature_names': ['treasure blast - balloon'],
                            'status': 'matched_all_three'},
 'treasure box dynasty': {'canonical_game_name': 'treasure box dynasty',
                          'eilers_names': ['treasure box - dynasty'],
                          'feature_names': ['treasure box - dynasty'],
                          'status': 'matched_all_three'},
 'treasure box kingdom': {'canonical_game_name': 'treasure box kingdom',
                          'eilers_names': ['treasure box - kingdom'],
                          'feature_names': ['treasure box - kingdom'],
                          'status': 'matched_all_three'},
 'treasure fleet': {'canonical_game_name': 'treasure fleet',
                    'eilers_names': ['treasure fleet'],
                    'feature_names': [],
                    'status': 'matched_eilers_only'},
 'treasures of troy multiway': {'canonical_game_name': None,
                                'eilers_names': [],
                                'feature_names': [],
                                'status': 'review_required'},
 'triple big wheel triple double diamond free games': {'canonical_game_name': None,
                                                       'eilers_names': [],
                                                       'feature_names': [],
                                                       'status': 'review_required'},
 'triple double red white blue 2x3x red white blue': {'canonical_game_name': None,
                                                      'eilers_names': [],
                                                      'feature_names': [],
                                                      'status': 'review_required'},
 'triple fortune dragon gold': {'canonical_game_name': 'triple fortune dragon gold',
                                'eilers_names': ['triple fortune dragon gold'],
                                'feature_names': [],
                                'status': 'matched_eilers_only'},
 'triple fortune link hong se cai fu red riches': {'canonical_game_name': 'triple fortune link hong se cai fu red '
                                                                          'riches',
                                                   'eilers_names': ['triple fortune link hong se cai fu red '
                                                                    'riches'],
                                                   'feature_names': [],
                                                   'status': 'matched_eilers_only'},
 'triple fortune link huang se cai fu yellow riches': {'canonical_game_name': 'triple fortune link huang se cai '
                                                                              'fu yellow riches',
                                                       'eilers_names': ['triple fortune link huang se cai fu '
                                                                        'yellow riches'],
                                                       'feature_names': [],
                                                       'status': 'matched_eilers_only'},
 'triple jackpot jewels': {'canonical_game_name': 'triple jackpot jewels',
                           'eilers_names': ['triple jackpot jewels'],
                           'feature_names': ['triple jackpot jewels'],
                           'status': 'matched_all_three'},
 'twin strike': {'canonical_game_name': 'twin strike',
                 'eilers_names': ['twin strike'],
                 'feature_names': ['twin strike'],
                 'status': 'matched_all_three'},
 'twin win': {'canonical_game_name': 'twin win',
              'eilers_names': ['twin win'],
              'feature_names': [],
              'status': 'matched_eilers_only'},
 'twin win pearl': {'canonical_game_name': 'twin win pearl',
                    'eilers_names': ['twin win pearl'],
                    'feature_names': ['twin win pearl'],
                    'status': 'matched_all_three'},
 'wheel shot electric unicorn': {'canonical_game_name': 'wheel shot electric unicorn',
                                 'eilers_names': ['wheel shot electric unicorn'],
                                 'feature_names': [],
                                 'status': 'matched_eilers_only'},
 'white hot 7': {'canonical_game_name': 'white hot 7s',
                 'eilers_names': ['white hot 7s'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'white orchid': {'canonical_game_name': 'white orchid',
                  'eilers_names': ['white orchid'],
                  'feature_names': [],
                  'status': 'matched_eilers_only'},
 'wild hits clovers': {'canonical_game_name': 'wild hits clover',
                       'eilers_names': ['wild hits clover'],
                       'feature_names': ['wild hits clover'],
                       'status': 'matched_all_three'},
 'wild hits pearls': {'canonical_game_name': 'wild hits pearls',
                      'eilers_names': ['wild hits pearls'],
                      'feature_names': ['wild hits pearls'],
                      'status': 'matched_all_three'},
 'wild pinata': {'canonical_game_name': 'wild pinata',
                 'eilers_names': ['wild pinata'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'},
 'wild pride': {'canonical_game_name': 'wild pride',
                'eilers_names': ['wild pride'],
                'feature_names': [],
                'status': 'matched_eilers_only'},
 'wild surf': {'canonical_game_name': 'wild surf',
               'eilers_names': ['wild surf'],
               'feature_names': [],
               'status': 'matched_eilers_only'},
 'wild tiles gems': {'canonical_game_name': 'gems wild tiles',
                     'eilers_names': ['gems wild tiles'],
                     'feature_names': [],
                     'status': 'matched_eilers_only'},
 'wish mistress': {'canonical_game_name': 'wish mistress',
                   'eilers_names': ['wish mistress'],
                   'feature_names': [],
                   'status': 'matched_eilers_only'},
 'wolf peak': {'canonical_game_name': 'wolf peak',
               'eilers_names': ['wolf peak'],
               'feature_names': ['wolf peak'],
               'status': 'matched_all_three'},
 'wolf run': {'canonical_game_name': 'wolf run',
              'eilers_names': ['wolf run'],
              'feature_names': [],
              'status': 'matched_eilers_only'},
 'wolf run eclipse': {'canonical_game_name': 'wolf run eclipse',
                      'eilers_names': ['wolf run eclipse'],
                      'feature_names': ['wolf run eclipse'],
                      'status': 'matched_all_three'},
 'wolf run gold': {'canonical_game_name': 'wolf run gold',
                   'eilers_names': ['wolf run gold'],
                   'feature_names': [],
                   'status': 'matched_eilers_only'},
 'wolf run wild moon': {'canonical_game_name': 'wolf run wild moon',
                        'eilers_names': ['wolf run - wild moon'],
                        'feature_names': ['wolf run - wild moon'],
                        'status': 'matched_all_three'},
 'wonders of africa': {'canonical_game_name': 'wonders of africa',
                       'eilers_names': ['wonders of africa'],
                       'feature_names': ['wonders of africa'],
                       'status': 'matched_all_three'},
 'wu da jiang': {'canonical_game_name': 'wu da jiang',
                 'eilers_names': ['wu da jiang'],
                 'feature_names': ['wu da jiang'],
                 'status': 'matched_all_three'},
 'wu dragon': {'canonical_game_name': 'wu dragon',
               'eilers_names': ['wu dragon'],
               'feature_names': ['wu dragon'],
               'status': 'matched_all_three'},
 'wu wu coins big buddha': {'canonical_game_name': 'wu wu coins big buddha',
                            'eilers_names': ['wu wu coins big buddha'],
                            'feature_names': [],
                            'status': 'matched_eilers_only'},
 'zhao cai zhu gettin piggy with it': {'canonical_game_name': 'zhao cai zhu gettin piggy with it',
                                       'eilers_names': ['zhao cai zhu - gettin piggy with it'],
                                       'feature_names': ['zhao cai zhu - gettin piggy with it'],
                                       'status': 'matched_all_three'},
 'zhao cai zhu yo ho hog': {'canonical_game_name': 'zhao cai zhu yo ho hog',
                            'eilers_names': ['zhao cai zhu - yo ho hog'],
                            'feature_names': [],
                            'status': 'matched_eilers_only'},
 'zodiac dragon': {'canonical_game_name': 'zodiac dragon',
                   'eilers_names': ['zodiac dragon'],
                   'feature_names': [],
                   'status': 'matched_eilers_only'},
 'zodiac lion': {'canonical_game_name': 'zodiac lion',
                 'eilers_names': ['zodiac lion'],
                 'feature_names': [],
                 'status': 'matched_eilers_only'}}

REVIEW_MATCHES = {'big bucks bonus': {'candidate_eilers_names': ['big bucks'],
                     'candidate_feature_names': [],
                     'reason': 'Bonus qualifier is absent from the candidate.'},
 'big juan dual hhrag': {'candidate_eilers_names': ['big juan'],
                         'candidate_feature_names': ['big juan'],
                         'reason': 'HHRAG may identify an HHR version; not treated as a cabinet suffix.'},
 'cable car cash multiway': {'candidate_eilers_names': ['cable car cash'],
                             'candidate_feature_names': ['cable car cash'],
                             'reason': 'Multiway configuration is not explicit in the candidate.'},
 'cats wild serengeti prog': {'candidate_eilers_names': ['cats wild serengeti'],
                              'candidate_feature_names': ['cats wild serengeti'],
                              'reason': 'Progressive qualifier is not explicit in the candidate.'},
 'cleopatra triple fortune': {'candidate_eilers_names': ['cleopatra - triple fortune link'],
                              'candidate_feature_names': ['cleopatra - triple fortune link'],
                              'reason': 'Strong candidate, but Link is omitted in sales.'},
 'diamond diamond': {'candidate_eilers_names': [],
                     'candidate_feature_names': [],
                     'reason': 'No credible counterpart; possible extraction or title duplication error.'},
 'diamond jackpots triple double stars': {'candidate_eilers_names': ['diamond jackpots - double spin - triple '
                                                                     'double stars'],
                                          'candidate_feature_names': [],
                                          'reason': 'Double Spin version is not explicit in sales.'},
 'double red white blue reel': {'candidate_eilers_names': ['double red white & blue'],
                                'candidate_feature_names': ['double red white & blue'],
                                'reason': 'Reel may be a configuration descriptor; confirm before removing.'},
 'dragon of fortune multiway': {'candidate_eilers_names': ['dragon of fortune'],
                                'candidate_feature_names': ['dragon of fortune'],
                                'reason': 'Multiway configuration is not explicit in the candidate; Evolution '
                                          'remains separate.'},
 'dynamite blast passion flowers': {'candidate_eilers_names': ['passion flowers'],
                                    'candidate_feature_names': [],
                                    'reason': 'Dynamite Blast family/version is not explicit in the candidate.'},
 'egyptian link amasis treasures buy a bonus': {'candidate_eilers_names': ['egyptian link - amasis treasure',
                                                                           'egyptian link wealthy wilds - amasis '
                                                                           'treasure'],
                                                'candidate_feature_names': ['egyptian link wealthy wilds - amasis '
                                                                            'treasure'],
                                                'reason': 'Buy a Bonus and Wealthy Wilds may indicate different '
                                                          'versions.'},
 'emperors throne': {'candidate_eilers_names': [],
                     'candidate_feature_names': [],
                     'reason': 'No credible counterpart; Emperor and Griffin titles must not be merged.'},
 'fire it up': {'candidate_eilers_names': ['fire it up free games'],
                'candidate_feature_names': ['fire it up free games'],
                'reason': 'Free Games qualifier is omitted in sales.'},
 'firebird spirit': {'candidate_eilers_names': ['connect & collect - firebird spirit'],
                     'candidate_feature_names': ['connect & collect - firebird spirit'],
                     'reason': 'Strong candidate, but Connect and Collect family is omitted in sales.'},
 'fortune coin': {'candidate_eilers_names': ['fortune coin boost',
                                             'fortune coin extra',
                                             'fortune coin free games',
                                             'fortune coin respin'],
                  'candidate_feature_names': ['fortune coin extra'],
                  'reason': 'Several distinct versions; do not default to Extra.'},
 'fortune link 4 1': {'candidate_eilers_names': ['fortune link 4'],
                      'candidate_feature_names': ['fortune link 4'],
                      'reason': 'Extra trailing 1 is unexplained; numbers are not stripped automatically.'},
 'grand buddha': {'candidate_eilers_names': ['grand buddha link'],
                  'candidate_feature_names': ['grand buddha link'],
                  'reason': 'Grand agrees, but Link is omitted in sales.'},
 'haywire reel': {'candidate_eilers_names': ['haywire deluxe', 'haywire! extreme', 'haywire - supercharged'],
                  'candidate_feature_names': ['haywire - supercharged'],
                  'reason': 'No exact base Haywire title; multiple versions remain possible.'},
 'hot hit rainbow riches lucky 7s': {'candidate_eilers_names': ['hot hit rainbow riches 7s free games'],
                                     'candidate_feature_names': [],
                                     'reason': 'Lucky / Free Games variation needs confirmation.'},
 'its da bomb gophersbgone 3r1l3': {'candidate_eilers_names': ['its da bomb gopher be gone'],
                                    'candidate_feature_names': [],
                                    'reason': 'Likely spelling/configuration difference; confirm the 3r1l3 '
                                              'suffix.'},
 'jackpot 7s free games 20l100c': {'candidate_eilers_names': ['jackpot 7s'],
                                   'candidate_feature_names': [],
                                   'reason': 'Free Games and 20l100c configuration are absent from the '
                                             'candidate.'},
 'majestic stallion 80 ctc': {'candidate_eilers_names': ['majestic stallion'],
                              'candidate_feature_names': ['majestic stallion'],
                              'reason': '80 CTC is a wager configuration; confirm title-level aggregation.'},
 'mayan riches kukulkan': {'candidate_eilers_names': ['mayan riches'],
                           'candidate_feature_names': [],
                           'reason': 'Kukulkan variant is not explicit; Ixchel is a different named variant.'},
 'munster match': {'candidate_eilers_names': ['the munsters'],
                   'candidate_feature_names': ['the munsters'],
                   'reason': 'Shared Munster wording does not establish the same title.'},
 'pharaoh 7s': {'candidate_eilers_names': ["ultimate sevens pharaoh's 777"],
                'candidate_feature_names': [],
                'reason': 'Possible family connection only; title and number differ.'},
 'progressive bonanza 3x2x legend of the peacock': {'candidate_eilers_names': ['legend of the 3x2x peacock'],
                                                    'candidate_feature_names': ['legend of the 3x2x peacock'],
                                                    'reason': 'Progressive Bonanza prefix is unexplained; Wheel '
                                                              'variant remains separate.'},
 'prosperity link blessings skyrise': {'candidate_eilers_names': ['prosperity link blessings'],
                                       'candidate_feature_names': [],
                                       'reason': 'Confirm that Skyrise is only a cabinet/configuration label.'},
 'red hot jackpots double diamond 2': {'candidate_eilers_names': ['red hot jackpots double diamond'],
                                       'candidate_feature_names': [],
                                       'reason': 'Version number 2 is absent from the candidate.'},
 'reeledge tullys treasure hunt': {'candidate_eilers_names': ["tully's treasure hunt"],
                                   'candidate_feature_names': [],
                                   'reason': 'Confirm that ReelEdge is only a platform label.'},
 'super times pay 3510 play poker action': {'candidate_eilers_names': ['super times pay poker'],
                                            'candidate_feature_names': ['super times pay poker'],
                                            'reason': '3/5/10-play configuration is not explicit; do not '
                                                      'substitute 50 or 100 play.'},
 'super wheelmania link legendary beauty': {'candidate_eilers_names': ['super wheelmania legendary beauty'],
                                            'candidate_feature_names': ['super wheelmania legendary beauty'],
                                            'reason': 'Link qualifier is absent; linked/non-linked labels coexist '
                                                      'for other Wheelmania titles.'},
 'super x lucky x 2x3x4x5x x pay': {'candidate_eilers_names': ['2x3x4x5x super lucky pay'],
                                    'candidate_feature_names': [],
                                    'reason': 'Strong candidate, but extra X tokens need confirmation.'},
 'tiger and dragon link cash on reels': {'candidate_eilers_names': ['tiger & dragon cash on reels'],
                                         'candidate_feature_names': [],
                                         'reason': 'Both Link and non-Link sales names exist; do not merge by '
                                                   'default.'},
 'treasures of troy multiway': {'candidate_eilers_names': ['treasures of troy'],
                                'candidate_feature_names': [],
                                'reason': 'Multiway configuration is not explicit; Legacy Link and Dynasty remain '
                                          'separate.'},
 'triple big wheel triple double diamond free games': {'candidate_eilers_names': ['triple big wheel - triple '
                                                                                  'double diamond',
                                                                                  'triple big wheel triple double '
                                                                                  'dmnd'],
                                                       'candidate_feature_names': ['triple big wheel - triple '
                                                                                   'double diamond'],
                                                       'reason': 'Free Games qualifier is absent from the '
                                                                 'candidates.'},
 'triple double red white blue 2x3x red white blue': {'candidate_eilers_names': ['triple double red white & blue',
                                                                                 'red white & blue 2x 3x'],
                                                      'candidate_feature_names': [],
                                                      'reason': 'May combine two titles; preserve the 2x3x '
                                                                'version distinction.'}}

FEATURE_MATCH_REVIEW = {'black widow': {'candidate_feature_names': ['black widow - superstar'],
                 'reason': 'Superstar qualifier is not a verified cabinet label; Grand remains separate.'},
 'golden eagle': {'candidate_feature_names': ['golden link golden eagle'],
                  'reason': 'Golden Link family/version is not explicit in the sales title.'},
 'golden phoenix': {'candidate_feature_names': ['golden link golden phoenix'],
                    'reason': 'Golden Link family/version is not explicit in the sales title.'},
 'lotus flower': {'candidate_feature_names': ['lotus flower blossoms'],
                  'reason': 'Blossoms qualifier may identify a distinct title.'},
 'super wheelmania link treasure fish': {'candidate_feature_names': ['super wheelmania treasure fish'],
                                         'reason': 'Link/non-Link versions both exist in Eilers.'},
 'super wheelmania link vip riches': {'candidate_feature_names': ['super wheelmania vip riches'],
                                      'reason': 'Link/non-Link versions both exist in Eilers.'},
 'treasure fleet': {'candidate_feature_names': ['treasure blast - fleet'],
                    'reason': 'Treasure Blast family/version is not explicit in sales.'}}

SALES_WITHOUT_FEATURE_MATCH = ['1421 voyages of zheng he',
 '2x3x4x5x dragon',
 '2x3x4x5x gold',
 '2x3x4x5x red hot 7s',
 '5 star',
 '5x gold',
 '5x pay lucky 7',
 'arctic fox',
 'benny big game',
 'big bucks bonus',
 'big juan dual hhrag',
 'black widow',
 'blue blazes',
 'buffet mania',
 'cable car cash multiway',
 'caifu zhi wu',
 'cats',
 'cats wild serengeti prog',
 'chinese treasure',
 'cleopatra multiplay',
 'cleopatra triple fortune',
 'diamond diamond',
 'diamond fire',
 'diamond jackpots triple double stars',
 'diamond queen',
 'double diamond',
 'double gold',
 'double red hot 7s',
 'double red white blue reel',
 'dragon lights mystical falls',
 'dragon lights secret fortress',
 'dragon of fortune multiway',
 'dynamite blast passion flowers',
 'egyptian link amasis treasures buy a bonus',
 'emperors throne',
 'fa cai long',
 'fame and fortune',
 'fire it up',
 'firebird spirit',
 'fortune coin',
 'fortune link 4 1',
 'fu gui hao men ii magnificent fortunes',
 'fu gui ji li animal treasures',
 'genie wishes',
 'goddess of gold',
 'gold bar 7s',
 'golden eagle',
 'golden egypt',
 'golden goddess',
 'golden phoenix',
 'goldify',
 'grand buddha',
 'haywire deluxe',
 'haywire reel',
 'hexbreaker',
 'hidden kingdom',
 'hot hit rainbow riches lucky 7s',
 'hot link blue moon ii express',
 'hot link money beans express',
 'island eyes',
 'its da bomb gophersbgone 3r1l3',
 'jackpot 7s free games 20l100c',
 'jewel of arabia',
 'jin hu',
 'king of olympus',
 'lion dance',
 'lotus flower',
 'luck rising brysons bounty',
 'lucky coin asian dreaming',
 'lucky coin atlantica',
 'lucky larrys lobstermania 4',
 'lucky larrys lobstermania ultimate treasure',
 'lucky lion fish',
 'lucky lock',
 'magic link celestial queen',
 'magic link stellar riches',
 'magic treasures gold emperor',
 'magic treasures gold empress',
 'magic wishes',
 'maharajas money',
 'majestic stallion 80 ctc',
 'mayan riches kukulkan',
 'midnight treasures',
 'mighty miner',
 'mistress of egypt grand',
 'moon spirit',
 'munster match',
 'mystery of the lamp enchanted palace',
 'mystery of the lamp mystical gardens',
 'ocean magic',
 'overloot',
 'pharaoh 7s',
 'phone tag',
 'pink diamond',
 'progressive bonanza 3x2x legend of the peacock',
 'prosperity link blessings skyrise',
 'prosperity link cai yun heng tong',
 'prosperity link wan shi ru yi',
 'prosperity pearl',
 'queenie',
 'red hot jackpots double diamond 2',
 'red hottie',
 'reeledge tullys treasure hunt',
 'regal riches',
 'rising reels link mayan magic',
 'rising spins',
 'san xing riches',
 'scarab grand',
 'scarab multiplay',
 'secrets of the forest',
 'sevens',
 'solar disc grand',
 'sphinx',
 'split n win linkfortune',
 'stinkin rich',
 'super sallys shrimpmania 5',
 'super times pay 3510 play poker action',
 'super wheelmania link legendary beauty',
 'super wheelmania link treasure fish',
 'super wheelmania link vip riches',
 'super x lucky x 2x3x4x5x x pay',
 'temple of fire',
 'temujin treasures',
 'the amazing money machine gold',
 'tiger and dragon cash on reels',
 'tiger and dragon link cash on reels',
 'treasure fleet',
 'treasures of troy multiway',
 'triple big wheel triple double diamond free games',
 'triple double red white blue 2x3x red white blue',
 'triple fortune dragon gold',
 'triple fortune link hong se cai fu red riches',
 'triple fortune link huang se cai fu yellow riches',
 'twin win',
 'wheel shot electric unicorn',
 'white hot 7',
 'white orchid',
 'wild pinata',
 'wild pride',
 'wild surf',
 'wild tiles gems',
 'wish mistress',
 'wolf run',
 'wolf run gold',
 'wu wu coins big buddha',
 'zhao cai zhu yo ho hog',
 'zodiac dragon',
 'zodiac lion']

NAME_MAPS = {
    "features": FEATURE_NAME_MAP,
    "eilers": EILERS_NAME_MAP,
    "sales": SALES_NAME_MAP,
}


def add_canonical_name(frame, source, name_column, output_column="canonical_game_name"):
    """Add a name lookup to a Spark DataFrame; unknown/review names remain null.

    Matches the exact raw names in the supplied exports. It does not perform
    fuzzy matching or silently normalize new names that were not reviewed.
    """
    from pyspark.sql import functions as F

    if source not in NAME_MAPS:
        raise ValueError("source must be 'features', 'eilers', or 'sales'")
    if output_column == name_column:
        raise ValueError("Keep the source name and canonical name in separate columns")
    if name_column not in frame.columns:
        raise ValueError(f"Missing input name column: {name_column}")

    pairs = [(raw, canonical) for raw, canonical in NAME_MAPS[source].items()
             if canonical is not None]
    lookup = frame.sparkSession.createDataFrame(
        pairs, "_lookup_source_name string, _lookup_canonical_name string"
    ).withColumnRenamed("_lookup_canonical_name", output_column)

    data = frame.drop(output_column)
    return (
        data.join(
            F.broadcast(lookup),
            data[name_column] == lookup["_lookup_source_name"],
            "left",
        )
        .drop(lookup["_lookup_source_name"])
    )


In [0]:

df = add_canonical_name(df, "features", "game_name")
eilers_df = add_canonical_name(df_eilers_ts, "eilers", "game_name")
df_sales = add_canonical_name(
    df_sales, "sales", "sales_normalized_title"
)


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Similar-game sales forecasts
# MAGIC Run these cells after your three `add_canonical_name(...)` calls.
# MAGIC Inputs: `df` (AI attributes), `eilers_df` (mapped Eilers time series),
# MAGIC and `df_sales` (mapped monthly sales).
# MAGIC
# MAGIC Defaults: oldest approximately 80% train, next 10% validation, newest 10% test.
# MAGIC Games sharing a release date stay together, so percentages can differ.
# MAGIC Only training games can be neighbors. Release date is NOT a similarity feature.
# MAGIC Sales curves start at each game's first recorded sales month, independent of release.
# MAGIC The first 12 months are indices **0-11**; the default six-month forecast is **0-5**.
# MAGIC This is a static comparison using available donor histories, not an as-of-date backtest.
# MAGIC No source DataFrames or saved tables are overwritten.

# COMMAND ----------
# 1. Configuration. Column names verified against your provided Spark schemas.
import math
from pyspark.sql import functions as F, Window

KEY = "canonical_game_name"
TS_MONTH_COL = "yearmonth"                 # Eilers: DATE
SALES_MONTH_COL = "beginning_month_date"    # Sales: DATE
SALES_VALUE_COL = "expr1_sum"               # Sales: LONG monthly units, not cumulative.

# Your Eilers schema has no explicit release-date column. Use the earliest
# observed yearmonth as a proxy. months_since_release is not a date column.
TS_RELEASE_COL = TS_MONTH_COL

#DISTANCE_METRIC = "euclidean"     # "cosine" or "euclidean"
N_NEIGHBORS = 5
SALES_HISTORY_MONTHS = 12
FORECAST_MONTHS = 6            # Set to 12 for a full first-year forecast.
SPLIT_METHOD = "chronological" # Or "random" for a repeatable game-level split.
TRAIN_FRACTION = 0.80
VALID_FRACTION = 0.10
SPLIT_SEED = 42

# All features default to weight 1. Edit only the weights you want to change.
# Weights apply AFTER scaling. A weight of zero disables a feature.
FEATURE_WEIGHT_OVERRIDES = {
    # "reels": 1.0,
    # "lines_or_ways": 1.0,
    # "perceived_persistence_score": 2.0,
    # "mechanic_hold_spin_lock_collect_present": 2.0,
}

# False: an absent month is unknown, so that game cannot supply a complete curve.
# True: absent INTERNAL months count as zero sales. Recorded nulls stay unknown.
# Neither setting extends a game beyond its last recorded month.
MISSING_MONTHS_ARE_ZERO = False

#assert DISTANCE_METRIC in {"cosine", "euclidean"}
assert SPLIT_METHOD in {"chronological", "random"}
assert 0 < TRAIN_FRACTION < 1 and 0 < VALID_FRACTION < 1 - TRAIN_FRACTION
assert 1 <= FORECAST_MONTHS <= SALES_HISTORY_MONTHS
assert isinstance(N_NEIGHBORS, int) and N_NEIGHBORS > 0

def quoted(column_name):
    return "`" + column_name.replace("`", "``") + "`"

def numeric(column_name):
    value = F.expr(f"try_cast({quoted(column_name)} AS DOUBLE)")
    return F.when(
        value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")), value
    )

def parsed_date(column_name):
    # Accept dates/timestamps, YYYY-MM, YYYYMM and YYYYMMDD. Invalid dates -> null.
    text = f"trim(cast({quoted(column_name)} AS STRING))"
    return F.expr(f"""try_cast(CASE
        WHEN {text} RLIKE '^[0-9]{{6}}$'
            THEN concat(substr({text}, 1, 4), '-', substr({text}, 5, 2), '-01')
        WHEN {text} RLIKE '^[0-9]{{8}}$'
            THEN concat(substr({text}, 1, 4), '-', substr({text}, 5, 2), '-', substr({text}, 7, 2))
        WHEN {text} RLIKE '^[0-9]{{4}}-[0-9]{{2}}$'
            THEN concat({text}, '-01')
        ELSE substr({text}, 1, 10)
        END AS DATE)""")

for _frame, _required, _label in [
    (df, [KEY, "game_name"], "df"),
    (eilers_df, [KEY, TS_RELEASE_COL], "eilers_df"),
    (df_sales, [KEY, SALES_MONTH_COL, SALES_VALUE_COL], "df_sales"),
]:
    _missing = set(_required) - set(_frame.columns)
    if _missing:
        raise ValueError(f"{_label} missing {_missing}. Available columns: {_frame.columns}")

print(f"Release date source: minimum {TS_RELEASE_COL} per canonical game.")
if TS_RELEASE_COL == TS_MONTH_COL:
    print("This is the first observed Eilers month, used as a release-date proxy.")

# COMMAND ----------
# 2. One attribute row per game, release dates, and game-level splits.
_theme_cols = [c for c in df.columns if c.startswith("theme_")
               and c.endswith("_score") and c != "theme_presence_score"]
_mechanic_cols = [c for c in df.columns if c.startswith(("mechanic_", "mech_"))
                  and (c.endswith("_present") or
                       (c.endswith("_score") and c[:-6] + "_present" not in df.columns))]
FEATURE_COLS = [c for c in ["reels", "lines_or_ways", "persistence_score",
                           "perceived_persistence_score"] if c in df.columns]
FEATURE_COLS += sorted(_theme_cols) + sorted(_mechanic_cols)
if not FEATURE_COLS:
    raise ValueError("No supported attribute feature columns were found in df.")
_unknown_weights = set(FEATURE_WEIGHT_OVERRIDES) - set(FEATURE_COLS)
if _unknown_weights:
    raise ValueError(f"Feature weight names not found: {_unknown_weights}")
FEATURE_WEIGHTS = {c: float(FEATURE_WEIGHT_OVERRIDES.get(c, 1.0)) for c in FEATURE_COLS}
if any(not math.isfinite(w) or w < 0 for w in FEATURE_WEIGHTS.values()):
    raise ValueError("Feature weights must be finite and nonnegative.")
if not any(FEATURE_WEIGHTS.values()):
    raise ValueError("At least one feature must have a positive weight.")

_attr = df.select(KEY, "game_name", *[numeric(c).alias(c) for c in FEATURE_COLS])
# Keep reels numeric and min-max scaled by the existing training-only scaler.
# Log-transform lines/ways first so very large counts have less influence.
if "lines_or_ways" in FEATURE_COLS:
    _attr = _attr.withColumn(
        "lines_or_ways",
        F.when(
            F.col("lines_or_ways").isNotNull()
            & (F.col("lines_or_ways") >= 0),
            F.log1p(F.col("lines_or_ways"))
        ).otherwise(F.lit(None).cast("double"))
    )
attribute_duplicate_games = (_attr.where(F.col(KEY).isNotNull())
                            .groupBy(KEY).count().where("count > 1"))
# Keep the most complete original feature row. Break ties deterministically.
# This prevents join multiplication without averaging different extraction rows.
# Retain attribute_duplicate_games for later review; the source df stays intact.
_attr = _attr.withColumn("_missing", sum(
    (F.col(c).isNull().cast("int") for c in FEATURE_COLS), F.lit(0)))
_attr_order = Window.partitionBy(KEY).orderBy(
    "_missing", F.to_json(F.struct(*[F.col(c) for c in FEATURE_COLS])), "game_name")
_attr = (_attr.where(F.col(KEY).isNotNull())
         .withColumn("_row", F.row_number().over(_attr_order))
         .where("_row = 1").drop("_missing", "_row")
         .fillna(0, subset=_theme_cols + _mechanic_cols))

_release_rows = eilers_df.select(KEY, parsed_date(TS_RELEASE_COL).alias("_release_date"))
release_date_review = _release_rows.where(F.col(KEY).isNull() | F.col("_release_date").isNull())
_release_dates = (_release_rows.where(F.col(KEY).isNotNull())
                  .groupBy(KEY).agg(F.min("_release_date").alias("game_release_date")))
ai_attr = _attr.join(_release_dates, KEY, "left")

# Split across the attribute games first; sales eligibility is applied afterward.
# A game without a release date is retained as 'unassigned' in chronological mode.
_split_base = ai_attr
if SPLIT_METHOD == "chronological":
    _split_base = _split_base.where(F.col("game_release_date").isNotNull())
    _split_order = Window.orderBy("game_release_date")
else:
    _split_order = Window.orderBy(F.sha2(
        F.concat(F.lit(str(SPLIT_SEED) + ":"), F.col(KEY)), 256))
_splits = (_split_base.withColumn("_position", F.percent_rank().over(_split_order))
           .select(KEY, F.when(F.col("_position") < TRAIN_FRACTION, "train")
                   .when(F.col("_position") < TRAIN_FRACTION + VALID_FRACTION, "valid")
                   .otherwise("test").alias("analysis_split")))
ai_attr = ai_attr.join(_splits, KEY, "left").fillna({"analysis_split": "unassigned"})
train_df = ai_attr.where("analysis_split = 'train'")
valid_df = ai_attr.where("analysis_split = 'valid'")
test_df = ai_attr.where("analysis_split = 'test'")
display(ai_attr.groupBy("analysis_split").agg(
    F.count("*").alias("games"), F.min("game_release_date").alias("earliest_release"),
    F.max("game_release_date").alias("latest_release")))

# COMMAND ----------
# 3. Monthly sales and cumulative curves, independent of release dates.
_sales_rows = df_sales.select(
    KEY, F.col(SALES_MONTH_COL).cast("string").alias("raw_sales_month"),
    F.col(SALES_VALUE_COL).cast("string").alias("raw_sales_value"),
    F.trunc(parsed_date(SALES_MONTH_COL), "month").alias("sales_month"),
    numeric(SALES_VALUE_COL).alias("_units"))
sales_input_review = _sales_rows.where(
    F.col(KEY).isNull() | F.col("sales_month").isNull() | F.col("_units").isNull())
_bad_date_games = (_sales_rows.where(F.col(KEY).isNotNull() & F.col("sales_month").isNull())
                   .select(KEY).distinct())
# Sum additive sales rows within game/month. Source rows must not contain repeated totals.
# If any component of a monthly total is missing/invalid, that total stays unknown.
_monthly_sales = (_sales_rows.where(F.col(KEY).isNotNull() & F.col("sales_month").isNotNull())
                  .groupBy(KEY, "sales_month").agg(
                      F.count("*").alias("_source_rows"),
                      F.when(F.count("_units") == F.count("*"), F.sum("_units"))
                       .alias("monthly_sales")))
_sales_games = (_monthly_sales.where(F.col("monthly_sales").isNotNull())
                .select(KEY).distinct())

_sales_range = (_monthly_sales.groupBy(KEY).agg(
    F.min("sales_month").alias("first_sales_month"),
    F.max("sales_month").alias("last_sales_month")))
_calendar = (_sales_range.withColumn("_last_index", F.least(
    F.months_between("last_sales_month", "first_sales_month").cast("int"),
    F.lit(SALES_HISTORY_MONTHS - 1)))
    .withColumn("month_index", F.explode(F.sequence(F.lit(0), F.col("_last_index"))))
    .withColumn("sales_month", F.add_months("first_sales_month", F.col("month_index")))
    .select(KEY, "sales_month", "month_index"))
_indexed_sales = _calendar.join(_monthly_sales, [KEY, "sales_month"], "left")
if MISSING_MONTHS_ARE_ZERO:
    _indexed_sales = _indexed_sales.withColumn("monthly_sales", F.when(
        F.col("_source_rows").isNull(), F.lit(0.0)).otherwise(F.col("monthly_sales")))
_cum_window = (Window.partitionBy(KEY).orderBy("month_index")
               .rowsBetween(Window.unboundedPreceding, Window.currentRow))
sales_curves = (_indexed_sales.withColumn("sales_cumsum", F.when(
    F.count("monthly_sales").over(_cum_window) == F.col("month_index") + 1,
    F.sum("monthly_sales").over(_cum_window)))
    .select(KEY, "sales_month", "month_index", "monthly_sales", "sales_cumsum"))

# Every selected neighbor must supply the entire requested horizon. This keeps
# exactly the same five games in every month's mean, instead of a changing cohort.
_complete_sales_games = (sales_curves.where(F.col("month_index") < FORECAST_MONTHS)
    .groupBy(KEY).agg(F.count("sales_cumsum").alias("available_months"))
    .where(F.col("available_months") == FORECAST_MONTHS)
    .join(_bad_date_games, KEY, "left_anti").select(KEY))
sales_train_df = train_df.join(_complete_sales_games, KEY, "inner")
sales_valid_df = valid_df.join(_sales_games, KEY, "inner")
sales_test_df = test_df.join(_sales_games, KEY, "inner")

# COMMAND ----------
# 4. Training-only imputation and min-max scaling, followed by feature weights.
# Fit on train_df only; validation/test values never affect scaling or imputation.
# Unknown reels/lines/persistence use the training median. Themes/mechanics use zero.
# Constant/all-missing training features contribute zero distance.
_stats_exprs = [F.count("*").alias("_train_count")]
for i, c in enumerate(FEATURE_COLS):
    _stats_exprs += [F.min(c).alias(f"min_{i}"), F.max(c).alias(f"max_{i}"),
                    F.percentile_approx(c, 0.5, 10000).alias(f"median_{i}")]
_stats = train_df.agg(*_stats_exprs).first().asDict()
if not _stats["_train_count"]:
    raise ValueError("No training games. Check the release-date column and split settings.")

_vector_parts = []
_feature_settings = []
for i, c in enumerate(FEATURE_COLS):
    lo, hi, median = _stats[f"min_{i}"], _stats[f"max_{i}"], _stats[f"median_{i}"]
    weight = FEATURE_WEIGHTS[c]
    active = lo is not None and hi is not None and hi > lo and weight > 0
    _feature_settings.append((c, weight, lo, hi, median, active))
    if active:
        value = F.coalesce(F.col(c), F.lit(median))
        scaled = (value - F.lit(lo)) / F.lit(hi - lo)
        scaled = F.greatest(F.lit(0.0), F.least(F.lit(1.0), scaled))
        # sqrt(weight) gives sum(weight * squared_difference) for Euclidean distance
        # and the corresponding weighted dot product/norms for cosine distance.
        _vector_parts.append(scaled * F.lit(math.sqrt(weight)))
if not _vector_parts:
    raise ValueError("No varying, positively weighted features in the training set.")
feature_settings = spark.createDataFrame(_feature_settings,
    "feature string, weight double, train_min double, train_max double, "
    "train_median double, active boolean")
_vectors = (ai_attr.withColumn("_features", F.array(*_vector_parts))
    .withColumn("_norm2", F.aggregate("_features", F.lit(0.0), lambda a, x: a + x * x)))

# COMMAND ----------
# 5. Rank the nearest five TRAINING games with complete sales histories.
# 5. Find the top N training neighbors using BOTH distance metrics.
DISTANCE_METRICS = ["cosine", "euclidean"]

_targets = (
    _vectors
    .where(F.col("analysis_split").isin("valid", "test"))
    .join(_sales_games, KEY, "inner")
    .select(
        F.col(KEY).alias("target_game_name"),
        F.col("game_name").alias("target_display_name"),
        "game_release_date",
        "analysis_split",
        F.col("_features").alias("target_features"),
        F.col("_norm2").alias("target_norm2"),
    )
)

_neighbors = (
    _vectors
    .where("analysis_split = 'train'")
    .join(_complete_sales_games, KEY, "inner")
    .select(
        F.col(KEY).alias("game_name"),
        F.col("game_name").alias("neighbor_display_name"),
        F.col("_features").alias("neighbor_features"),
        F.col("_norm2").alias("neighbor_norm2"),
    )
)

_base_pairs = (
    _targets.crossJoin(F.broadcast(_neighbors))
    .where(F.col("target_game_name") != F.col("game_name"))
)

similar_games_by_metric = []

for metric in DISTANCE_METRICS:
    pairs = _base_pairs

    if metric == "euclidean":
        distance = F.sqrt(
            F.aggregate(
                F.zip_with(
                    "target_features",
                    "neighbor_features",
                    lambda x, y: (x - y) * (x - y),
                ),
                F.lit(0.0),
                lambda total, value: total + value,
            )
        )
    else:
        # Cosine distance is undefined when either vector has zero length.
        pairs = pairs.where(
            (F.col("target_norm2") > 0)
            & (F.col("neighbor_norm2") > 0)
        )
        dot_product = F.aggregate(
            F.zip_with(
                "target_features",
                "neighbor_features",
                lambda x, y: x * y,
            ),
            F.lit(0.0),
            lambda total, value: total + value,
        )
        distance = F.greatest(
            F.lit(0.0),
            F.least(
                F.lit(2.0),
                F.lit(1.0)
                - dot_product
                / F.sqrt(F.col("target_norm2") * F.col("neighbor_norm2")),
            ),
        )

    rank_window = Window.partitionBy("target_game_name").orderBy(
        "distance", "game_name"
    )

    ranked = (
        pairs
        .withColumn("distance", distance)
        .withColumn("neighbor_rank", F.row_number().over(rank_window))
        .where(F.col("neighbor_rank") <= N_NEIGHBORS)
        .select(
            "target_game_name",
            "target_display_name",
            "analysis_split",
            "game_name",
            "neighbor_display_name",
            "neighbor_rank",
            "distance",
        )
        .withColumn("distance_metric", F.lit(metric))
    )
    similar_games_by_metric.append(ranked)

similar_games = similar_games_by_metric[0]
for frame in similar_games_by_metric[1:]:
    similar_games = similar_games.unionByName(frame)


# 6. Average each metric's selected neighbors' cumulative sales curves.
_neighbor_curves = (
    sales_curves
    .where(F.col("month_index") < FORECAST_MONTHS)
    .select(
        F.col(KEY).alias("game_name"),
        "month_index",
        "monthly_sales",
        "sales_cumsum",
    )
)

_means = (
    similar_games
    .join(_neighbor_curves, "game_name", "inner")
    .groupBy("target_game_name", "analysis_split",
             "distance_metric", "month_index")
    .agg(
        F.avg("sales_cumsum").alias("sales_mean"),
        F.avg("monthly_sales").alias("monthly_sales_mean"),
        F.countDistinct("game_name").alias("n_neighbors"),
    )
)

# Include both metrics in the forecast output, including targets whose cosine
# vector is zero and therefore has no defined cosine neighbors.
_metric_df = spark.createDataFrame(
    [(m,) for m in DISTANCE_METRICS],
    "distance_metric string",
)

_forecast_grid = (
    _targets
    .drop("target_features")
    .crossJoin(_metric_df)
    .crossJoin(
        spark.range(FORECAST_MONTHS)
        .select(F.col("id").cast("int").alias("month_index"))
    )
)

pred_sales_all = (
    _forecast_grid
    .join(
        _means,
        ["target_game_name", "analysis_split", "distance_metric", "month_index"],
        "left",
    )
    .fillna({"n_neighbors": 0})
    .withColumn(
        "forecast_status",
        F.when(
            (F.col("distance_metric") == "cosine")
            & (F.col("target_norm2") == 0),
            "zero_feature_vector",
        )
        .when(F.col("n_neighbors") < N_NEIGHBORS, "not_enough_neighbors")
        .otherwise("ok"),
    )
    .withColumn(
        "sales_mean",
        F.when(F.col("forecast_status") == "ok", F.col("sales_mean")),
    )
    .withColumn(
        "monthly_sales_mean",
        F.when(
            F.col("forecast_status") == "ok",
            F.col("monthly_sales_mean"),
        ),
    )
    .drop("target_norm2")
)

# Attach actual held-out sales after generating predictions.
_actuals = (
    sales_curves
    .join(_bad_date_games, KEY, "left_anti")
    .select(
        F.col(KEY).alias("target_game_name"),
        "month_index",
        F.col("monthly_sales").alias("actual_monthly_sales"),
        F.col("sales_cumsum").alias("actual_sales_cumsum"),
    )
)

pred_sales_all = pred_sales_all.join(
    _actuals, ["target_game_name", "month_index"], "left"
)

# Keep these names for compatibility with the rest of your notebook.
pred_sales = pred_sales_all.where("analysis_split = 'test'")
pred_sales_valid = pred_sales_all.where("analysis_split = 'valid'")
similar_games_test = similar_games.where("analysis_split = 'test'")
similar_games_valid = similar_games.where("analysis_split = 'valid'")

# COMMAND ----------
# 7. Inspect the outputs. No table writes.
print("sales_mean = mean cumulative expr1_sum across the selected training games.")
print(f"History: indices 0-{SALES_HISTORY_MONTHS - 1}; forecast: 0-{FORECAST_MONTHS - 1}.")
display(pred_sales.orderBy("target_game_name", "month_index"))
display(similar_games.orderBy("analysis_split", "target_game_name", "neighbor_rank"))
display(pred_sales_all.groupBy("analysis_split", "forecast_status")
        .agg(F.countDistinct("target_game_name").alias("games")))

# COMMAND ----------
# MAGIC %md
# MAGIC ## Outputs and assumptions
# MAGIC - `ai_attr`: one selected attribute row per canonical game, plus release date and split.
# MAGIC - `train_df`, `valid_df`, `test_df`: the attribute splits, including games without sales.
# MAGIC - `sales_train_df`: training games with complete forecast-horizon sales.
# MAGIC - `sales_valid_df`, `sales_test_df`: held-out attribute games with sales to compare against.
# MAGIC - `sales_curves`: up to 12 calendar months from the first recorded sales month, with cumsum.
# MAGIC - `pred_sales`: TEST forecasts; `pred_sales_valid`: VALIDATION forecasts.
# MAGIC - `pred_sales_all`: both splits. `sales_mean` is predicted CUMULATIVE sales.
# MAGIC - `similar_games`: one row per target/neighbor, `neighbor_rank` 1-5 and `distance`.
# MAGIC   `target_game_name` and neighbor `game_name` contain canonical names. Lower distance is closer.
# MAGIC - `feature_settings`: training ranges, imputation medians, weights and inactive features.
# MAGIC - `attribute_duplicate_games`, `release_date_review`, `sales_input_review`: inspection frames.
# MAGIC
# MAGIC Use validation results to choose weights/metric; keep test results for final evaluation.
# MAGIC The same five training games contribute at every forecast month. Fewer than five available
# MAGIC neighbors, or an undefined cosine comparison, produces a null forecast with a status.
# MAGIC Missing tail months are never invented. True recorded zero sales are retained.
# MAGIC Negative sales adjustments are retained, so cumulative curves can decrease.
# MAGIC Attribute duplicates are reduced deterministically for this model; source rows are preserved.
# MAGIC Games without mapped names cannot join. Without a release date they cannot enter a chronological split.
# MAGIC Release date is the earliest Eilers `yearmonth`, a proxy rather than a confirmed launch date.
# MAGIC Sales dates use `beginning_month_date`; monthly unit sales use `expr1_sum`.
# MAGIC Eilers `months_since_release` is not included in similarity or converted into a date.
# MAGIC First recorded sales month is only the first true sales month if the export covers the full history.
# MAGIC This sums additive `expr1_sum` rows by canonical game/month; repeated totals in the source must be resolved upstream.
# MAGIC All feature weights start at 1; a group with more columns has more total influence.
# MAGIC Min-max scaling uses training ranges, clips values to [0,1], and ignores constant training columns.
# MAGIC
# MAGIC Implementation uses Spark SQL expressions and array functions, without Python UDFs or RDDs.
# MAGIC References: [zip_with](https://spark.apache.org/docs/3.5.7/api/python/reference/pyspark.sql/api/pyspark.sql.functions.zip_with.html),
# MAGIC [aggregate](https://spark.apache.org/docs/3.5.7/api/python/reference/pyspark.sql/api/pyspark.sql.functions.aggregate.html).


In [0]:
import matplotlib.pyplot as plt

viewer_predictions = pred_sales_all.toPandas()
viewer_neighbors = similar_games.toPandas()

viewer_curves = sales_curves.select(
    KEY, "month_index", "sales_cumsum"
).toPandas()

viewer_games = sorted(
    viewer_predictions["target_game_name"].dropna().unique().tolist()
)

if not viewer_games:
    raise ValueError("No forecast games found. Run Cell 7 first.")

# Recreate the widget so this cell can be rerun after the game list changes.
try:
    dbutils.widgets.remove("forecast_game")
except Exception:
    pass

dbutils.widgets.dropdown(
    "forecast_game",
    viewer_games[0],
    viewer_games,
    "Forecast game",
)

selected_game = dbutils.widgets.get("forecast_game")

game_predictions = viewer_predictions[
    viewer_predictions["target_game_name"] == selected_game
]

if game_predictions.empty:
    raise ValueError(f"No forecast found for {selected_game}.")

display_name = game_predictions["target_display_name"].iloc[0]
split_name = game_predictions["analysis_split"].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharey=True)

for ax, metric in zip(axes, ["cosine", "euclidean"]):
    prediction = (
        game_predictions[game_predictions["distance_metric"] == metric]
        .sort_values("month_index")
    )
    neighbors = (
        viewer_neighbors[
            (viewer_neighbors["target_game_name"] == selected_game)
            & (viewer_neighbors["distance_metric"] == metric)
        ]
        .sort_values("neighbor_rank")
        .head(N_NEIGHBORS)
    )

    last_month = (
        int(prediction["month_index"].max())
        if not prediction.empty else FORECAST_MONTHS - 1
    )

    # Plot the historical cumulative curves for the five selected neighbors.
    for neighbor in neighbors.itertuples(index=False):
        curve = viewer_curves[
            (viewer_curves[KEY] == neighbor.game_name)
            & (viewer_curves["month_index"] <= last_month)
        ].sort_values("month_index")

        if not curve.empty:
            ax.plot(
                curve["month_index"],
                curve["sales_cumsum"],
                linewidth=1.3,
                alpha=0.45,
                label=f"Neighbor {neighbor.neighbor_rank}: "
                      f"{neighbor.neighbor_display_name}",
            )

    # Plot the prediction and actual curve for the selected game.
    if not prediction.empty and prediction["sales_mean"].notna().any():
        ax.plot(
            prediction["month_index"],
            prediction["sales_mean"],
            color="blue",
            linewidth=3,
            marker="o",
            label="Prediction: mean of neighbors",
        )

    if not prediction.empty and prediction["actual_sales_cumsum"].notna().any():
        ax.plot(
            prediction["month_index"],
            prediction["actual_sales_cumsum"],
            color="black",
            linewidth=2.5,
            linestyle="--",
            marker="s",
            label="Actual sales",
        )

    statuses = (
        prediction["forecast_status"].dropna().unique().tolist()
        if not prediction.empty else []
    )
    status_text = f" | {', '.join(statuses)}" if statuses and statuses != ["ok"] else ""

    ax.set_title(f"{metric.title()} distance{status_text}")
    ax.set_xlabel("Month index (0 = first recorded sales month)")
    ax.grid(alpha=0.2)

    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1))

axes[0].set_ylabel("Cumulative sales units")
fig.suptitle(f"{display_name} | {split_name} set", fontsize=14)
fig.tight_layout()
plt.show()
plt.close(fig)

# Show each metric's top five neighbors in one table.
top_five = (
    viewer_neighbors[
        viewer_neighbors["target_game_name"] == selected_game
    ]
    .sort_values(["distance_metric", "neighbor_rank"])
    [[
        "distance_metric",
        "neighbor_rank",
        "neighbor_display_name",
        "distance",
    ]]
    .rename(columns={"neighbor_display_name": "game_name"})
)

display(top_five)

In [0]:
import numpy as np

MIN_SALES_MONTHS = 4  # More than 3 observed months
MIN_MAPE_ACTUAL = 4   # MAPE excludes rows where actual sales are zero

# Count months with observed actual cumulative sales for each target.
eligible_games = (
    pred_sales_all
    .groupBy("target_game_name", "analysis_split", "distance_metric")
    .agg(
        F.countDistinct(
            F.when(
                F.col("actual_sales_cumsum").isNotNull(),
                F.col("month_index"),
            )
        ).alias("observed_months")
    )
    .where(F.col("observed_months") >= MIN_SALES_MONTHS)
    .select("target_game_name", "analysis_split", "distance_metric")
)

evaluation_df = (
    pred_sales_all
    .join(
        eligible_games,
        ["target_game_name", "analysis_split", "distance_metric"],
        "inner",
    )
    .where(
        F.col("sales_mean").isNotNull()
        & F.col("actual_sales_cumsum").isNotNull()
    )
    .select(
        "target_game_name",
        "analysis_split",
        "distance_metric",
        "month_index",
        F.col("actual_sales_cumsum").cast("double").alias("actual"),
        F.col("sales_mean").cast("double").alias("predicted"),
    )
)

evaluation_pdf = evaluation_df.toPandas()

if evaluation_pdf.empty:
    raise ValueError(
        "No scored forecasts remain. Check that held-out games have at least "
        "four observed sales months and that forecasts have five neighbors."
    )

evaluation_pdf["error"] = evaluation_pdf["predicted"] - evaluation_pdf["actual"]

def calculate_metrics(frame):
    actual = frame["actual"].to_numpy(dtype=float)
    predicted = frame["predicted"].to_numpy(dtype=float)
    errors = predicted - actual

    mae = float(np.mean(np.abs(errors)))
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    bias = float(np.mean(errors))

    nonzero_actual = actual != MIN_MAPE_ACTUAL
    mape = (
        float(np.mean(np.abs(errors[nonzero_actual] / actual[nonzero_actual])) * 100)
        if nonzero_actual.any()
        else np.nan
    )

    denominator = float(np.sum(np.abs(actual)))
    wape = (
        float(np.sum(np.abs(errors)) / denominator * 100)
        if denominator > 0
        else np.nan
    )

    total_variation = float(np.sum((actual - np.mean(actual)) ** 2))
    r2 = (
        float(1 - np.sum(errors ** 2) / total_variation)
        if len(actual) > 1 and total_variation > 0
        else np.nan
    )

    return {
        "games": int(frame["target_game_name"].nunique()),
        "game_month_points": int(len(frame)),
        "RMSE": rmse,
        "MAE": mae,
        "MAPE_percent": mape,
        "WAPE_percent": wape,
        "Mean_error_bias": bias,
        "R2": r2,
    }

metric_rows = []

# Combined validation + test performance, plus split-level diagnostics.
for metric in ["cosine", "euclidean"]:
    metric_frame = evaluation_pdf[
        evaluation_pdf["distance_metric"] == metric
    ]

    for split_label, split_frame in [
        ("combined_valid_test", metric_frame),
        *[
            (split, metric_frame[metric_frame["analysis_split"] == split])
            for split in ["valid", "test"]
        ],
    ]:
        if not split_frame.empty:
            metric_rows.append({
                "distance_metric": metric,
                "evaluation_split": split_label,
                **calculate_metrics(split_frame),
            })

performance_metrics = spark.createDataFrame(metric_rows)
display(
    performance_metrics.orderBy("evaluation_split", "distance_metric")
)

In [0]:
pred_sales_all.write.mode("overwrite").saveAsTable(dest_ai_path+"pred_games_ai")
similar_games.write.mode("overwrite").saveAsTable(dest_ai_path+"similar_games_ai")

display(pred_sales_all)
display(similar_games)